# 11 Final Results Visualization Research Deliverables

This GitHub copy has been cleaned for readability: duplicate/empty cells were removed and saved cell outputs were cleared. Code and analysis workflow are retained.

**Execution note:** Original project paths may reference Google Drive/Colab. Update paths and provide required source files/API credentials in your own environment before rerunning.

In [ ]:

# ============================================================
# NOTEBOOK 11 — FINAL RESULTS, VISUALIZATION & DELIVERABLES
# Project: ICMB-130
# ============================================================

import os
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Locate project
PROJECT_CANDIDATES = [
    Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics"),
    Path("/content/drive/My Drive/IndianCinemaTrendAnalytics")
]

PROJECT_ROOT = next(
    (p for p in PROJECT_CANDIDATES if p.exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "IndianCinemaTrendAnalytics folder was not found in Google Drive."
    )

# Main directories
DATA_DIR = PROJECT_ROOT / "01_data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
NOTEBOOK_DIR = PROJECT_ROOT / "02_notebooks"

# Final output directories
OUTPUT_DIR = PROJECT_ROOT / "05_outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
REPORT_DIR = OUTPUT_DIR / "reports"
SUMMARY_DIR = OUTPUT_DIR / "summary"

for directory in [
    OUTPUT_DIR, TABLE_DIR, FIGURE_DIR,
    REPORT_DIR, SUMMARY_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Visualization defaults
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

sns.set_theme(style="whitegrid")

print("=" * 70)
print("ICMB-130 | NOTEBOOK 11 ENVIRONMENT CHECK")
print("=" * 70)

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

print("\nProject root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)
print("Final outputs:", OUTPUT_DIR)

print("\nDirectory setup:")
for directory in [
    TABLE_DIR, FIGURE_DIR, REPORT_DIR, SUMMARY_DIR
]:
    print(f"{directory.name}: {directory.exists()}")

print("\nEnvironment setup completed successfully.")


In [ ]:

# ============================================================
# CELL 2 — COMPLETE PROJECT ARTIFACT INVENTORY
# ============================================================

inventory = []

SCAN_DIRS = [PROCESSED_DIR, RAW_DIR, OUTPUT_DIR]

for base_dir in SCAN_DIRS:
    if not base_dir.exists():
        continue

    for file_path in base_dir.rglob("*"):
        if not file_path.is_file():
            continue

        suffix = file_path.suffix.lower()

        if suffix not in [
            ".csv", ".parquet", ".json",
            ".xlsx", ".pkl", ".txt"
        ]:
            continue

        record = {
            "file": file_path.name,
            "folder": str(file_path.parent.relative_to(PROJECT_ROOT)),
            "extension": suffix,
            "size_kb": round(file_path.stat().st_size / 1024, 2),
            "rows": None,
            "columns": None
        }

        try:
            if suffix == ".csv":
                df_temp = pd.read_csv(file_path, nrows=5)
                record["columns"] = len(df_temp.columns)
                record["column_names"] = list(df_temp.columns)

                with open(file_path, "r", encoding="utf-8-sig") as f:
                    record["rows"] = max(
                        sum(1 for _ in f) - 1, 0
                    )

            elif suffix == ".parquet":
                df_temp = pd.read_parquet(file_path)
                record["rows"] = len(df_temp)
                record["columns"] = len(df_temp.columns)

            elif suffix == ".xlsx":
                df_temp = pd.read_excel(file_path, nrows=5)
                record["columns"] = len(df_temp.columns)

        except Exception as e:
            record["inspection_error"] = str(e)[:150]

        inventory.append(record)

inventory_df = pd.DataFrame(inventory)

print("=" * 90)
print("ICMB-130 COMPLETE ARTIFACT INVENTORY")
print("=" * 90)

if not inventory_df.empty:
    display(
        inventory_df.sort_values(
            ["folder", "file"]
        ).reset_index(drop=True)
    )
else:
    print("No analytical artifacts found.")

inventory_path = TABLE_DIR / "ICMB_130_NOTEBOOK11_ARTIFACT_INVENTORY.csv"
inventory_df.to_csv(inventory_path, index=False)

print("\nInventory saved:", inventory_path)
print("Total analytical files:", len(inventory_df))


In [ ]:

# ============================================================
# CELL 3 — FINAL ANALYTICAL OUTPUT DISCOVERY
# ============================================================

from IPython.display import display

# Keywords used to identify research components
categories = {
    "MASTER_DATA": [
        "MASTER_DATA", "FEATURES", "REVENUE_DATA"
    ],
    "BOX_OFFICE": [
        "BOX_OFFICE", "REVENUE", "GROSS", "INDUSTRY"
    ],
    "SUBTITLE_CORPUS": [
        "SUBTITLE", "CORPUS", "MANIFEST"
    ],
    "TOPIC_MODELLING": [
        "TOPIC", "NMF", "BERTopic", "STM"
    ],
    "SENTIMENT_ANALYSIS": [
        "SENTIMENT", "INDICBERT", "POLARITY",
        "EMOTION", "CLASSIFICATION"
    ],
    "YOUTUBE_ANALYSIS": [
        "YOUTUBE", "COMMENT", "AUDIENCE"
    ],
    "FINAL_AUDITS": [
        "AUDIT", "QUALITY", "VALIDATION",
        "READINESS", "VERIFICATION"
    ]
}

# Inventory already created in Cell 2
discovery_records = []

for _, row in inventory_df.iterrows():

    filename = str(row["file"])
    filename_upper = filename.upper()

    matched_categories = []

    for category, keywords in categories.items():

        if any(keyword.upper() in filename_upper
               for keyword in keywords):
            matched_categories.append(category)

    if matched_categories:

        discovery_records.append({
            "file": filename,
            "folder": row["folder"],
            "category": ", ".join(matched_categories),
            "extension": row["extension"],
            "size_kb": row["size_kb"],
            "rows": row["rows"],
            "columns": row["columns"]
        })

discovery_df = pd.DataFrame(discovery_records)

print("=" * 100)
print("ICMB-130 FINAL ANALYTICAL OUTPUT DISCOVERY")
print("=" * 100)

for category in categories:

    subset = discovery_df[
        discovery_df["category"].str.contains(category)
    ]

    print(f"\n{'=' * 80}")
    print(f"{category}: {len(subset)} FILES")
    print("=" * 80)

    if len(subset) > 0:
        display(subset.reset_index(drop=True))
    else:
        print("No matching files found.")

# Save discovery report
discovery_path = TABLE_DIR / "ICMB_130_NOTEBOOK11_ANALYTICAL_OUTPUT_DISCOVERY.csv"

discovery_df.to_csv(discovery_path, index=False)

print("\nDiscovery report saved:", discovery_path)
print("Total matching files:", len(discovery_df))


In [ ]:

# ============================================================
# CELL 4 — FINAL DATASET INSPECTION & TOPIC OUTPUT SEARCH
# ============================================================

def load_project_csv(relative_path):
    path = PROJECT_ROOT / relative_path

    if not path.exists():
        print("MISSING:", relative_path)
        return None

    return pd.read_csv(path, low_memory=False)


# ------------------------------------------------------------
# 1. LOAD MASTER DATASET VERSIONS
# ------------------------------------------------------------

master = load_project_csv(
    "01_data/processed/ICMB_130_MASTER_DATA.csv"
)

master_repaired = load_project_csv(
    "01_data/processed/ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

features = load_project_csv(
    "01_data/processed/ICMB_130_FEATURES.csv"
)

revenue = load_project_csv(
    "01_data/processed/ICMB_130_REVENUE_DATA.csv"
)


print("\n" + "=" * 90)
print("MASTER DATASET COMPARISON")
print("=" * 90)

master_versions = {
    "MASTER_DATA": master,
    "TMDB_REPAIRED": master_repaired,
    "FEATURES": features,
    "REVENUE": revenue
}

for name, df in master_versions.items():

    if df is not None:
        print(f"\n{name}")
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())

        if "movie_record_id" in df.columns:
            print(
                "Unique movie IDs:",
                df["movie_record_id"].nunique()
            )


# Compare original and repaired master datasets
if master is not None and master_repaired is not None:

    common_cols = sorted(
        set(master.columns) & set(master_repaired.columns)
    )

    print("\nOriginal vs Repaired comparison")
    print("Common columns:", len(common_cols))

    if "movie_record_id" in common_cols:

        original_indexed = master.set_index("movie_record_id")
        repaired_indexed = master_repaired.set_index(
            "movie_record_id"
        )

        print(
            "Same movie IDs:",
            set(original_indexed.index) ==
            set(repaired_indexed.index)
        )

        comparable_cols = [
            c for c in common_cols if c != "movie_record_id"
        ]

        comparison = []

        for col in comparable_cols:

            original_values = (
                original_indexed[col].fillna("<MISSING>").astype(str)
            )

            repaired_values = (
                repaired_indexed[col].fillna("<MISSING>").astype(str)
            )

            if original_values.index.equals(repaired_values.index):
                differences = (
                    original_values != repaired_values
                ).sum()
            else:
                differences = "Different movie IDs/order"

            comparison.append({
                "column": col,
                "different_records": differences
            })

        display(pd.DataFrame(comparison))


# ------------------------------------------------------------
# 2. INSPECT FINAL SUBTITLE MANIFEST
# ------------------------------------------------------------

subtitle_manifest = load_project_csv(
    "01_data/processed/subtitle_corpus_v2/"
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

print("\n" + "=" * 90)
print("FINAL SUBTITLE MANIFEST")
print("=" * 90)

if subtitle_manifest is not None:

    print("Shape:", subtitle_manifest.shape)
    print("Columns:", subtitle_manifest.columns.tolist())

    display(subtitle_manifest.head(5))

    for col in [
        "acquisition_status",
        "validation_status",
        "quality_status",
        "subtitle_language"
    ]:
        if col in subtitle_manifest.columns:
            print(f"\nDistribution: {col}")
            display(
                subtitle_manifest[col]
                .value_counts(dropna=False)
                .head(15)
            )


# ------------------------------------------------------------
# 3. INSPECT YOUTUBE SOURCE MANIFEST
# ------------------------------------------------------------

youtube_manifest = load_project_csv(
    "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_FINAL_SOURCE_MANIFEST_BATCH_01.csv"
)

if youtube_manifest is None:

    youtube_manifest = load_project_csv(
        "01_data/processed/audience_sentiment/"
        "ICMB_130_YOUTUBE_ACQUISITION_MANIFEST.csv"
    )

print("\n" + "=" * 90)
print("YOUTUBE SOURCE MANIFEST")
print("=" * 90)

if youtube_manifest is not None:

    print("Shape:", youtube_manifest.shape)
    print("Columns:", youtube_manifest.columns.tolist())

    display(youtube_manifest.head(5))


# ------------------------------------------------------------
# 4. INSPECT SENTIMENT BASELINE
# ------------------------------------------------------------

sentiment = load_project_csv(
    "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_SENTIMENT_BASELINE.csv"
)

print("\n" + "=" * 90)
print("YOUTUBE COMMENTS SENTIMENT BASELINE")
print("=" * 90)

if sentiment is not None:

    print("Shape:", sentiment.shape)
    print("Columns:")

    for col in sentiment.columns:
        print("-", col)

    print("\nSample records:")
    display(sentiment.head(5))

    print("\nMissing values — top 20 columns:")
    display(
        sentiment.isna().sum()
        .sort_values(ascending=False)
        .head(20)
    )

    print("\nPotential sentiment-related columns:")

    sentiment_columns = [
        c for c in sentiment.columns
        if any(k in c.lower() for k in [
            "sentiment", "polarity", "score",
            "label", "emotion", "confidence"
        ])
    ]

    print(sentiment_columns)

    for col in sentiment_columns:

        if sentiment[col].nunique(dropna=True) <= 20:
            print(f"\nDistribution: {col}")
            display(
                sentiment[col]
                .value_counts(dropna=False)
                .head(20)
            )


# ------------------------------------------------------------
# 5. SEARCH ALL PROJECT DIRECTORIES FOR TOPIC OUTPUTS
# ------------------------------------------------------------

topic_keywords = [
    "topic", "nmf", "bertopic", "stm",
    "lda", "coherence", "tfidf", "document_topic",
    "topic_distribution", "topic_terms"
]

topic_files = []

for file_path in PROJECT_ROOT.rglob("*"):

    if not file_path.is_file():
        continue

    if file_path.suffix.lower() not in [
        ".csv", ".json", ".txt", ".xlsx", ".pkl",
        ".joblib", ".parquet", ".npy"
    ]:
        continue

    filename = file_path.name.lower()

    if any(keyword in filename for keyword in topic_keywords):

        topic_files.append({
            "file": file_path.name,
            "folder": str(
                file_path.parent.relative_to(PROJECT_ROOT)
            ),
            "extension": file_path.suffix,
            "size_kb": round(
                file_path.stat().st_size / 1024, 2
            )
        })

topic_files_df = pd.DataFrame(topic_files)

print("\n" + "=" * 90)
print("COMPREHENSIVE TOPIC MODELLING OUTPUT SEARCH")
print("=" * 90)

print("Topic-related files found:", len(topic_files_df))

if not topic_files_df.empty:
    display(
        topic_files_df.sort_values(
            ["folder", "file"]
        ).reset_index(drop=True)
    )
else:
    print("No topic-related files found anywhere in the project.")

# Save inspection report
inspection_summary = {
    "master_shape": (
        list(master.shape) if master is not None else None
    ),
    "repaired_master_shape": (
        list(master_repaired.shape)
        if master_repaired is not None else None
    ),
    "features_shape": (
        list(features.shape) if features is not None else None
    ),
    "revenue_shape": (
        list(revenue.shape) if revenue is not None else None
    ),
    "subtitle_manifest_shape": (
        list(subtitle_manifest.shape)
        if subtitle_manifest is not None else None
    ),
    "sentiment_shape": (
        list(sentiment.shape) if sentiment is not None else None
    ),
    "topic_files_found": len(topic_files_df)
}

summary_path = SUMMARY_DIR / (
    "ICMB_130_NOTEBOOK11_DATASET_INSPECTION.json"
)

with open(summary_path, "w") as f:
    json.dump(inspection_summary, f, indent=4)

print("\nInspection summary saved:", summary_path)
print("\nCELL 4 COMPLETED")


In [ ]:

# ============================================================
# CELL 5 — SOURCE INTEGRITY & TOPIC MODELLING AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 90)
print("ICMB-130 | SOURCE INTEGRITY AUDIT")
print("=" * 90)

# ------------------------------------------------------------
# 1. MASTER DATASET COMPARISON BY MOVIE RECORD ID
# ------------------------------------------------------------

if master is not None and master_repaired is not None:

    original = master.set_index("movie_record_id").sort_index()
    repaired = master_repaired.set_index(
        "movie_record_id"
    ).sort_index()

    shared_ids = original.index.intersection(repaired.index)

    print("\n1. MASTER DATASET COMPARISON")
    print("Original records:", len(original))
    print("Repaired records:", len(repaired))
    print("Shared movie IDs:", len(shared_ids))

    comparison_results = []

    for col in original.columns.intersection(repaired.columns):

        a = original.loc[shared_ids, col]
        b = repaired.loc[shared_ids, col]

        # Compare values while treating missing values consistently
        a = a.astype("string").fillna("<NA>").str.strip()
        b = b.astype("string").fillna("<NA>").str.strip()

        different = (a != b).sum()

        comparison_results.append({
            "column": col,
            "different_records": int(different),
            "original_missing": int(original[col].isna().sum()),
            "repaired_missing": int(repaired[col].isna().sum())
        })

    master_comparison_audit = pd.DataFrame(
        comparison_results
    )

    display(master_comparison_audit)

    print("\nTMDB metadata completeness")

    tmdb_columns = [
        "tmdb_id", "original_title", "release_date",
        "runtime", "genres", "original_language",
        "production_countries", "budget",
        "worldwide_revenue"
    ]

    tmdb_completeness = []

    for name, df in [
        ("Original", master),
        ("Repaired", master_repaired)
    ]:

        for col in tmdb_columns:

            if col in df.columns:

                tmdb_completeness.append({
                    "dataset": name,
                    "column": col,
                    "available": int(df[col].notna().sum()),
                    "missing": int(df[col].isna().sum()),
                    "coverage_pct": round(
                        df[col].notna().mean() * 100, 2
                    )
                })

    tmdb_completeness_df = pd.DataFrame(
        tmdb_completeness
    )

    display(tmdb_completeness_df)

    # Save comparison
    master_comparison_audit.to_csv(
        TABLE_DIR / "ICMB_130_MASTER_VERSION_COMPARISON.csv",
        index=False
    )

    tmdb_completeness_df.to_csv(
        TABLE_DIR / "ICMB_130_TMDB_METADATA_COMPLETENESS.csv",
        index=False
    )


# ------------------------------------------------------------
# 2. SUBTITLE CORPUS QUALITY AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("2. SUBTITLE CORPUS QUALITY AUDIT")
print("=" * 90)

if subtitle_manifest is not None:

    subtitle_audit = subtitle_manifest.copy()

    print("Total manifest records:", len(subtitle_audit))
    print(
        "Unique movie records:",
        subtitle_audit["movie_record_id"].nunique()
    )

    for col in [
        "acquisition_status",
        "validation_status",
        "quality_status",
        "subtitle_language"
    ]:

        if col in subtitle_audit.columns:

            print(f"\n{col} — normalized distribution")

            normalized = (
                subtitle_audit[col]
                .fillna("Missing")
                .astype(str)
                .str.strip()
                .str.lower()
            )

            display(
                normalized.value_counts()
                .rename_axis(col)
                .reset_index(name="records")
            )

    subtitle_numeric = [
        "dialogue_blocks", "word_count", "character_count"
    ]

    for col in subtitle_numeric:

        if col in subtitle_audit.columns:

            subtitle_audit[col] = pd.to_numeric(
                subtitle_audit[col], errors="coerce"
            )

            print(f"\n{col} statistics:")
            display(subtitle_audit[col].describe())


# ------------------------------------------------------------
# 3. YOUTUBE ACQUISITION COVERAGE
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("3. YOUTUBE ACQUISITION COVERAGE")
print("=" * 90)

if youtube_manifest is not None:

    print("Movie records:", len(youtube_manifest))
    print(
        "Unique movie IDs:",
        youtube_manifest["movie_record_id"].nunique()
    )

    for col in [
        "discovery_status",
        "acquisition_status"
    ]:

        if col in youtube_manifest.columns:

            print(f"\n{col} distribution:")

            display(
                youtube_manifest[col]
                .fillna("Missing")
                .value_counts()
                .rename_axis(col)
                .reset_index(name="records")
            )

    for col in [
        "candidate_video_count",
        "verified_video_count",
        "comments_collected"
    ]:

        if col in youtube_manifest.columns:

            youtube_manifest[col] = pd.to_numeric(
                youtube_manifest[col], errors="coerce"
            )

            print(f"\n{col} summary:")
            display(youtube_manifest[col].describe())


# ------------------------------------------------------------
# 4. SENTIMENT RELIABILITY AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("4. SENTIMENT RELIABILITY AUDIT")
print("=" * 90)

if sentiment is not None:

    for col in [
        "final_sentiment",
        "model_sentiment",
        "sentiment_reliability_group",
        "final_language",
        "sentiment_input_type"
    ]:

        if col in sentiment.columns:

            print(f"\n{col} distribution:")

            display(
                sentiment[col]
                .fillna("Missing")
                .value_counts()
                .rename_axis(col)
                .reset_index(name="records")
            )

    # Sentiment confidence statistics
    if "sentiment_confidence" in sentiment.columns:

        sentiment["sentiment_confidence"] = pd.to_numeric(
            sentiment["sentiment_confidence"],
            errors="coerce"
        )

        print("\nSentiment confidence statistics:")

        display(
            sentiment["sentiment_confidence"].describe()
        )

    # Reliability group against final sentiment
    if (
        "sentiment_reliability_group" in sentiment.columns
        and "final_sentiment" in sentiment.columns
    ):

        print("\nReliability group vs final sentiment:")

        reliability_cross_tab = pd.crosstab(
            sentiment["sentiment_reliability_group"],
            sentiment["final_sentiment"],
            margins=True
        )

        display(reliability_cross_tab)

        reliability_cross_tab.to_csv(
            TABLE_DIR / "ICMB_130_SENTIMENT_RELIABILITY_CROSSTAB.csv"
        )


# ------------------------------------------------------------
# 5. TOPIC MODELLING OUTPUT CATALOGUE
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("5. TOPIC MODELLING OUTPUT CATALOGUE")
print("=" * 90)

TOPIC_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"

if not TOPIC_DIR.exists():
    print("Topic modelling directory not found:", TOPIC_DIR)

else:

    topic_catalogue = []

    for file_path in TOPIC_DIR.rglob("*"):

        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in [
            ".csv", ".json", ".txt", ".xlsx",
            ".parquet", ".pkl"
        ]:
            continue

        record = {
            "file": file_path.name,
            "extension": file_path.suffix.lower(),
            "size_kb": round(
                file_path.stat().st_size / 1024, 2
            ),
            "rows": None,
            "columns": None,
            "column_names": None
        }

        try:

            if file_path.suffix.lower() == ".csv":

                sample = pd.read_csv(
                    file_path, nrows=3, low_memory=False
                )

                record["columns"] = len(sample.columns)
                record["column_names"] = sample.columns.tolist()

            elif file_path.suffix.lower() == ".parquet":

                sample = pd.read_parquet(file_path)
                record["rows"] = len(sample)
                record["columns"] = len(sample.columns)
                record["column_names"] = sample.columns.tolist()

        except Exception as e:

            record["inspection_error"] = str(e)[:100]

        topic_catalogue.append(record)

    topic_catalogue_df = pd.DataFrame(topic_catalogue)

    print("Total topic output files:", len(topic_catalogue_df))

    print("\nComplete topic output filenames:")

    for filename in sorted(topic_catalogue_df["file"].tolist()):
        print("-", filename)

    # Identify potentially final result files
    final_candidates = topic_catalogue_df[
        topic_catalogue_df["file"].str.contains(
            "FINAL|REFINED|REBUILT|REPRODUCED|INTERPRETATION|INVENTORY|TEMPORAL|COVERAGE|DISTRIBUTION",
            case=False,
            regex=True
        )
    ].copy()

    print("\nPotential final topic result files:")
    display(final_candidates)

    # Save catalogue
    topic_catalogue_df.to_csv(
        TABLE_DIR / "ICMB_130_TOPIC_MODELLING_FILE_CATALOGUE.csv",
        index=False
    )

print("\n" + "=" * 90)
print("CELL 5 COMPLETED")
print("=" * 90)


In [ ]:

# =============================================================================
# CELL 6 — FINAL TOPIC RESULTS & YOUTUBE SOURCE RECONCILIATION
# =============================================================================

from pathlib import Path
import pandas as pd
import json

print("=" * 90)
print("ICMB-130 | FINAL TOPIC RESULTS & YOUTUBE SOURCE RECONCILIATION")
print("=" * 90)

# ---------------------------------------------------------------------------
# PART A — Inspect final curated topic-modelling outputs
# ---------------------------------------------------------------------------

print("\n" + "=" * 90)
print("PART A — FINAL TOPIC MODELLING OUTPUTS")
print("=" * 90)

final_topic_files = [
    "ICMB_130_FINAL_CURATED_THEME_REGISTER.csv",
    "ICMB_130_FINAL_12_THEME_REGISTER.csv",
    "ICMB_130_FINAL_LONGITUDINAL_THEME_SUMMARY.csv",
    "ICMB_130_FINAL_ANNUAL_THEME_PREVALENCE.csv",
    "ICMB_130_FINAL_THEME_CONCENTRATION.csv",
    "ICMB_130_FINAL_FILM_COVERAGE_MATRIX.csv",
    "ICMB_130_FINAL_DIALOGUE_PREVALENCE_MATRIX.csv",
    "ICMB_130_THEME_YEARLY_FILM_COVERAGE_MATRIX.csv",
    "ICMB_130_THEME_YEARLY_DIALOGUE_PREVALENCE_MATRIX.csv",
    "ICMB_130_THREE_PERIOD_THEME_COMPARISON.csv",
    "ICMB_130_BASELINE_CROSS_MODEL_THEME_COMPARISON.csv",
    "ICMB_130_BERTOPIC_CONSOLIDATED_MODEL_COMPARISON.csv",
    "ICMB_130_SIX_MODEL_SONG_ROBUSTNESS_COMPARISON.csv",
    "ICMB_130_THEME_CONCENTRATION_ROBUST_DIAGNOSTIC.csv",
]

topic_inspection = []

for filename in final_topic_files:
    path = TOPIC_DIR / filename

    if path.exists():
        try:
            df = pd.read_csv(path)
            topic_inspection.append({
                "file": filename,
                "exists": True,
                "rows": len(df),
                "columns": len(df.columns),
                "column_names": ", ".join(df.columns.astype(str))
            })

            print(f"\nFILE: {filename}")
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            print("Preview:")
            display(df.head(5))

        except Exception as e:
            print(f"\nERROR reading {filename}: {e}")

    else:
        print(f"\nNOT FOUND: {filename}")

topic_inspection_df = pd.DataFrame(topic_inspection)

topic_inspection_path = (
    TABLE_DIR / "ICMB_130_FINAL_TOPIC_OUTPUT_INSPECTION.csv"
)
topic_inspection_df.to_csv(topic_inspection_path, index=False)

print("\nSaved topic inspection:", topic_inspection_path)

# ---------------------------------------------------------------------------
# PART B — Inspect final topic-modelling Excel workbook
# ---------------------------------------------------------------------------

print("\n" + "=" * 90)
print("PART B — FINAL TOPIC MODELLING WORKBOOK")
print("=" * 90)

topic_workbook = TOPIC_DIR / "ICMB_130_FINAL_TOPIC_MODELLING_RESULTS.xlsx"

if topic_workbook.exists():
    try:
        excel_file = pd.ExcelFile(topic_workbook)
        print("Workbook:", topic_workbook.name)
        print("Sheet names:", excel_file.sheet_names)

        for sheet in excel_file.sheet_names:
            sheet_df = pd.read_excel(topic_workbook, sheet_name=sheet)
            print(f"\nSHEET: {sheet}")
            print("Shape:", sheet_df.shape)
            print("Columns:", sheet_df.columns.tolist())
            display(sheet_df.head(3))

    except Exception as e:
        print("Workbook inspection error:", e)
else:
    print("Final topic workbook not found.")

# ---------------------------------------------------------------------------
# PART C — Locate all YouTube-related files in the project
# ---------------------------------------------------------------------------

print("\n" + "=" * 90)
print("PART C — YOUTUBE FILE DISCOVERY")
print("=" * 90)

youtube_keywords = [
    "youtube", "comment", "video", "discovery",
    "source_manifest", "verification", "sentiment"
]

youtube_file_records = []

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    if any(word in path.name.lower() for word in youtube_keywords):
        try:
            relative_path = str(path.relative_to(PROJECT_ROOT))
            youtube_file_records.append({
                "file": path.name,
                "relative_path": relative_path,
                "extension": path.suffix.lower(),
                "size_kb": round(path.stat().st_size / 1024, 2)
            })
        except Exception:
            pass

youtube_files_df = pd.DataFrame(youtube_file_records)

if not youtube_files_df.empty:
    youtube_files_df = youtube_files_df.sort_values(
        ["extension", "file"]
    ).reset_index(drop=True)

    print("Total matching files:", len(youtube_files_df))
    display(youtube_files_df.to_string(index=False))

    youtube_catalogue_path = (
        TABLE_DIR / "ICMB_130_YOUTUBE_FILE_DISCOVERY.csv"
    )
    youtube_files_df.to_csv(youtube_catalogue_path, index=False)
    print("\nSaved YouTube file catalogue:", youtube_catalogue_path)
else:
    print("No matching YouTube-related files found.")

# ---------------------------------------------------------------------------
# PART D — Inspect likely YouTube manifests and summary files
# ---------------------------------------------------------------------------

print("\n" + "=" * 90)
print("PART D — YOUTUBE MANIFEST AND SUMMARY INSPECTION")
print("=" * 90)

if not youtube_files_df.empty:
    likely_files = youtube_files_df[
        youtube_files_df["file"].str.lower().str.contains(
            "manifest|summary|verification|discovery|source",
            na=False
        )
        & youtube_files_df["extension"].isin([".csv", ".xlsx"])
    ]

    print("Likely manifest/summary files:", len(likely_files))

    for _, row in likely_files.iterrows():
        path = PROJECT_ROOT / row["relative_path"]

        try:
            print(f"\nFILE: {row['relative_path']}")

            if path.suffix.lower() == ".csv":
                df = pd.read_csv(path, low_memory=False)
                print("Shape:", df.shape)
                print("Columns:", df.columns.tolist())
                display(df.head(3))

            elif path.suffix.lower() == ".xlsx":
                excel_file = pd.ExcelFile(path)
                print("Sheets:", excel_file.sheet_names)

                for sheet in excel_file.sheet_names[:3]:
                    df = pd.read_excel(path, sheet_name=sheet)
                    print(f"Sheet: {sheet} | Shape: {df.shape}")
                    print("Columns:", df.columns.tolist())
                    display(df.head(3))

        except Exception as e:
            print("Inspection error:", e)

# ---------------------------------------------------------------------------
# PART E — Save a compact inspection summary
# ---------------------------------------------------------------------------

inspection_summary = {
    "final_topic_files_inspected": len(topic_inspection_df),
    "youtube_related_files_found": len(youtube_files_df),
    "topic_directory": str(TOPIC_DIR),
    "project_root": str(PROJECT_ROOT),
    "inspection_status": "Completed"
}

summary_path = (
    SUMMARY_DIR / "ICMB_130_NOTEBOOK11_CELL6_INSPECTION.json"
)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(inspection_summary, f, indent=4)

print("\n" + "=" * 90)
print("CELL 6 COMPLETED")
print("=" * 90)
print(json.dumps(inspection_summary, indent=4))


In [ ]:

# =============================================================================
# CELL 7 — YOUTUBE SOURCE MANIFEST & COMMENT COVERAGE RECONCILIATION
# =============================================================================

from pathlib import Path
import pandas as pd
import json

print("=" * 95)
print("ICMB-130 | YOUTUBE SOURCE AND COMMENT COVERAGE RECONCILIATION")
print("=" * 95)

# ---------------------------------------------------------------------------
# PART A — Locate all final YouTube source manifests
# ---------------------------------------------------------------------------

print("\n" + "=" * 95)
print("PART A — FINAL YOUTUBE SOURCE MANIFESTS")
print("=" * 95)

manifest_files = list(PROJECT_ROOT.rglob(
    "*YOUTUBE*FINAL*SOURCE*MANIFEST*.csv"
))

manifest_records = []
manifest_dataframes = []

for path in sorted(manifest_files):
    try:
        df = pd.read_csv(path, low_memory=False)

        video_col = "video_id" if "video_id" in df.columns else None
        movie_col = (
            "movie_record_id"
            if "movie_record_id" in df.columns
            else None
        )

        record = {
            "file": path.name,
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "rows": len(df),
            "columns": len(df.columns),
            "unique_videos": (
                df[video_col].nunique()
                if video_col else None
            ),
            "unique_movies": (
                df[movie_col].nunique()
                if movie_col else None
            ),
            "column_names": ", ".join(df.columns.astype(str))
        }

        manifest_records.append(record)
        manifest_dataframes.append((path, df))

        print(f"\nFILE: {record['relative_path']}")
        print("Shape:", df.shape)
        print("Unique video IDs:", record["unique_videos"])
        print("Unique movie IDs:", record["unique_movies"])
        print("Columns:", df.columns.tolist())

        status_columns = [
            "acquisition_role",
            "acquisition_status",
            "verification_status",
            "manual_verification_status",
            "screening_category",
            "proposed_decision",
            "manual_decision"
        ]

        for col in status_columns:
            if col in df.columns:
                print(f"\n{col} distribution:")
                print(df[col].fillna("Missing").value_counts().head(15))

        print("\nSample records:")
        display(df.head(4))

    except Exception as e:
        print(f"ERROR reading {path.name}: {e}")

manifest_catalogue_df = pd.DataFrame(manifest_records)

manifest_catalogue_path = (
    TABLE_DIR / "ICMB_130_NOTEBOOK11_YOUTUBE_MANIFEST_CATALOGUE.csv"
)

manifest_catalogue_df.to_csv(manifest_catalogue_path, index=False)

print("\nTotal final source-manifest files:", len(manifest_catalogue_df))
print("Saved:", manifest_catalogue_path)

# ---------------------------------------------------------------------------
# PART B — Combined manifest-level reconciliation
# ---------------------------------------------------------------------------

print("\n" + "=" * 95)
print("PART B — COMBINED SOURCE MANIFEST RECONCILIATION")
print("=" * 95)

if manifest_dataframes:

    combined_manifest = pd.concat(
        [df.assign(source_manifest_file=path.name)
         for path, df in manifest_dataframes],
        ignore_index=True,
        sort=False
    )

    print("Combined manifest shape:", combined_manifest.shape)

    if "movie_record_id" in combined_manifest.columns:
        print(
            "Unique movie records:",
            combined_manifest["movie_record_id"].nunique()
        )

    if "video_id" in combined_manifest.columns:
        valid_video = combined_manifest[
            combined_manifest["video_id"].notna()
            & combined_manifest["video_id"].astype(str).str.strip().ne("")
        ]

        print("Rows with video IDs:", len(valid_video))
        print("Unique video IDs:", valid_video["video_id"].nunique())
        print(
            "Duplicate video ID rows:",
            valid_video["video_id"].duplicated().sum()
        )

    for col in [
        "acquisition_role",
        "acquisition_status",
        "verification_status",
        "manual_verification_status",
        "screening_category",
        "proposed_decision"
    ]:
        if col in combined_manifest.columns:
            print(f"\nCOMBINED {col} DISTRIBUTION:")
            print(
                combined_manifest[col]
                .fillna("Missing")
                .value_counts()
                .head(20)
            )

    combined_manifest_path = (
        TABLE_DIR / "ICMB_130_NOTEBOOK11_COMBINED_YOUTUBE_SOURCE_MANIFEST.csv"
    )

    combined_manifest.to_csv(combined_manifest_path, index=False)
    print("\nSaved combined manifest:", combined_manifest_path)

else:
    combined_manifest = pd.DataFrame()
    print("No final source-manifest files found.")

# ---------------------------------------------------------------------------
# PART C — Locate final YouTube comment coverage reports
# ---------------------------------------------------------------------------

print("\n" + "=" * 95)
print("PART C — YOUTUBE COMMENT COVERAGE REPORTS")
print("=" * 95)

coverage_files = list(PROJECT_ROOT.rglob(
    "*FINAL*YOUTUBE*COMMENT*COVERAGE*.csv"
))

coverage_records = []

for path in sorted(coverage_files):
    try:
        df = pd.read_csv(path, low_memory=False)

        record = {
            "file": path.name,
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "rows": len(df),
            "columns": len(df.columns),
            "column_names": ", ".join(df.columns.astype(str))
        }

        coverage_records.append(record)

        print(f"\nFILE: {record['relative_path']}")
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(5))

    except Exception as e:
        print(f"ERROR reading {path.name}: {e}")

coverage_catalogue_df = pd.DataFrame(coverage_records)

coverage_catalogue_path = (
    TABLE_DIR / "ICMB_130_NOTEBOOK11_YOUTUBE_COVERAGE_CATALOGUE.csv"
)

coverage_catalogue_df.to_csv(coverage_catalogue_path, index=False)

print("\nTotal comment coverage files:", len(coverage_catalogue_df))
print("Saved:", coverage_catalogue_path)

# ---------------------------------------------------------------------------
# PART D — Inspect approved acquisition source files
# ---------------------------------------------------------------------------

print("\n" + "=" * 95)
print("PART D — APPROVED YOUTUBE ACQUISITION SOURCES")
print("=" * 95)

approved_files = list(PROJECT_ROOT.rglob(
    "*YOUTUBE*APPROVED*ACQUISITION*SOURCE*.csv"
))

approved_records = []

for path in sorted(approved_files):
    try:
        df = pd.read_csv(path, low_memory=False)

        approved_records.append({
            "file": path.name,
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "rows": len(df),
            "columns": len(df.columns)
        })

        print(f"\nFILE: {path.relative_to(PROJECT_ROOT)}")
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())

        for col in [
            "acquisition_status",
            "manual_verification_status",
            "acquisition_result"
        ]:
            if col in df.columns:
                print(f"\n{col}:")
                print(df[col].fillna("Missing").value_counts())

        display(df.head(5))

    except Exception as e:
        print(f"ERROR reading {path.name}: {e}")

approved_catalogue_df = pd.DataFrame(approved_records)

approved_catalogue_path = (
    TABLE_DIR / "ICMB_130_NOTEBOOK11_APPROVED_YOUTUBE_SOURCES.csv"
)

approved_catalogue_df.to_csv(approved_catalogue_path, index=False)

print("\nApproved source files found:", len(approved_catalogue_df))
print("Saved:", approved_catalogue_path)

# ---------------------------------------------------------------------------
# PART E — Final audit summary
# ---------------------------------------------------------------------------

audit_summary = {
    "final_source_manifest_files": len(manifest_catalogue_df),
    "comment_coverage_files": len(coverage_catalogue_df),
    "approved_acquisition_source_files": len(approved_catalogue_df),
    "combined_manifest_rows": (
        len(combined_manifest)
        if not combined_manifest.empty else 0
    ),
    "combined_manifest_unique_video_ids": (
        int(combined_manifest["video_id"].nunique())
        if (
            not combined_manifest.empty
            and "video_id" in combined_manifest.columns
        ) else 0
    ),
    "status": "Inspection completed; interpretation pending"
}

summary_path = (
    SUMMARY_DIR / "ICMB_130_NOTEBOOK11_CELL7_YOUTUBE_AUDIT.json"
)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(audit_summary, f, indent=4)

print("\n" + "=" * 95)
print("CELL 7 COMPLETED")
print("=" * 95)
print(json.dumps(audit_summary, indent=4))


In [ ]:

# ============================================================
# NOTEBOOK 11 — CELL 8
# YOUTUBE COVERAGE, ACQUISITION & SENTIMENT RECONCILIATION
# ============================================================

import os
import json
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
OUT = os.path.join(BASE, "05_outputs")
TABLES = os.path.join(OUT, "tables")
SUMMARY = os.path.join(OUT, "summary")

os.makedirs(TABLES, exist_ok=True)
os.makedirs(SUMMARY, exist_ok=True)

print("=" * 90)
print("CELL 8 — YOUTUBE COVERAGE AND SENTIMENT RECONCILIATION")
print("=" * 90)

# ------------------------------------------------------------
# 1. LOAD THE TWO FILM-LEVEL COVERAGE REPORTS
# ------------------------------------------------------------

coverage_dir = os.path.join(BASE, "06_outputs", "audience_sentiment")

coverage_files = [
    os.path.join(coverage_dir, "ICMB_130_FINAL_YOUTUBE_COMMENT_COVERAGE_BATCH_01_15.csv"),
    os.path.join(coverage_dir, "ICMB_130_FINAL_YOUTUBE_COMMENT_COVERAGE_BATCH_01_16.csv")
]

coverage_frames = []

for path in coverage_files:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"\nCOVERAGE FILE: {os.path.basename(path)}")
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        coverage_frames.append((os.path.basename(path), df))
    else:
        print("\nMISSING COVERAGE FILE:", path)

if not coverage_frames:
    raise FileNotFoundError("No YouTube coverage reports were found.")

# Use the latest compact coverage report as the primary film-level table.
# Retain the other report for comparison.
coverage = None
for filename, df in coverage_frames:
    if "unique_comment_ids" in df.columns:
        coverage = df.copy()
        primary_coverage_file = filename
        break

if coverage is None:
    coverage = coverage_frames[-1][1].copy()
    primary_coverage_file = coverage_frames[-1][0]

print("\nPRIMARY COVERAGE REPORT:", primary_coverage_file)

required_coverage_cols = [
    "movie_record_id", "movie_title", "total_comments",
    "unique_videos", "coverage_status", "coverage_category"
]

missing = [c for c in required_coverage_cols if c not in coverage.columns]
if missing:
    raise ValueError(f"Missing required coverage columns: {missing}")

coverage["total_comments"] = pd.to_numeric(
    coverage["total_comments"], errors="coerce"
).fillna(0)

coverage["unique_videos"] = pd.to_numeric(
    coverage["unique_videos"], errors="coerce"
).fillna(0)

print("\nFILM-LEVEL COVERAGE SUMMARY")
print("Film records:", coverage["movie_record_id"].nunique())
print("Rows:", len(coverage))
print("Duplicate movie IDs:", coverage["movie_record_id"].duplicated().sum())
print("Total comments reported:", int(coverage["total_comments"].sum()))
print("Total unique-video counts summed across films:",
      int(coverage["unique_videos"].sum()))
print("\nCoverage status:")
print(coverage["coverage_status"].value_counts(dropna=False))
print("\nCoverage category:")
print(coverage["coverage_category"].value_counts(dropna=False))

print("\nCOMMENT COUNT DISTRIBUTION")
print(coverage["total_comments"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90]
).to_string())

print("\nCOMMENT COUNT BANDS")
comment_bins = [-1, 0, 24, 49, 99, 100, 249, 499, 999, np.inf]
comment_labels = [
    "0", "1-24", "25-49", "50-99", "100",
    "101-249", "250-499", "500-999", "1000+"
]
coverage["comment_count_band"] = pd.cut(
    coverage["total_comments"],
    bins=comment_bins,
    labels=comment_labels
)
print(coverage["comment_count_band"].value_counts(
    sort=False, dropna=False
))

# ------------------------------------------------------------
# 2. COMPARE THE TWO COVERAGE REPORTS
# ------------------------------------------------------------

if len(coverage_frames) >= 2:
    left_name, left = coverage_frames[0]
    right_name, right = coverage_frames[1]

    comparison_cols = ["movie_record_id", "total_comments", "unique_videos"]

    left_cols = [c for c in comparison_cols if c in left.columns]
    right_cols = [c for c in comparison_cols if c in right.columns]

    if "movie_record_id" in left_cols and "movie_record_id" in right_cols:
        left_small = left[left_cols].copy()
        right_small = right[right_cols].copy()

        compare = left_small.merge(
            right_small,
            on="movie_record_id",
            how="outer",
            suffixes=("_report1", "_report2"),
            indicator=True
        )

        print("\nCOVERAGE REPORT COMPARISON")
        print("Report 1:", left_name, left.shape)
        print("Report 2:", right_name, right.shape)
        print("Join status:")
        print(compare["_merge"].value_counts(dropna=False))

        for col in ["total_comments", "unique_videos"]:
            c1, c2 = f"{col}_report1", f"{col}_report2"
            if c1 in compare.columns and c2 in compare.columns:
                compare[f"{col}_difference"] = (
                    pd.to_numeric(compare[c2], errors="coerce") -
                    pd.to_numeric(compare[c1], errors="coerce")
                )
                print(
                    f"{col} — rows with differences:",
                    int((compare[f"{col}_difference"].fillna(0) != 0).sum())
                )

        compare.to_csv(
            os.path.join(TABLES, "ICMB_130_NOTEBOOK11_COVERAGE_REPORT_COMPARISON.csv"),
            index=False
        )

# ------------------------------------------------------------
# 3. INSPECT COMBINED APPROVED ACQUISITION SOURCES
# ------------------------------------------------------------

approved_path = os.path.join(
    TABLES, "ICMB_130_NOTEBOOK11_APPROVED_YOUTUBE_SOURCES.csv"
)

if os.path.exists(approved_path):
    approved = pd.read_csv(approved_path)
    print("\nCOMBINED ACQUISITION SOURCE FILE")
    print("Shape:", approved.shape)
    print("Columns:", approved.columns.tolist())

    for col in [
        "acquisition_status", "acquisition_result",
        "manual_verification_status", "content_review_status",
        "acquisition_approval_status", "identity_verification_status"
    ]:
        if col in approved.columns:
            print(f"\n{col} distribution:")
            print(approved[col].fillna("Missing").value_counts().head(20))

    if "movie_record_id" in approved.columns:
        print("\nAcquisition source coverage:")
        print("Unique movie records:", approved["movie_record_id"].nunique())

    video_col = "video_id" if "video_id" in approved.columns else None
    if video_col:
        print("Rows with video IDs:", approved[video_col].notna().sum())
        print("Unique video IDs:", approved[video_col].nunique())
        print("Duplicate video ID rows:", approved[video_col].duplicated().sum())

    approved.to_csv(
        os.path.join(TABLES, "ICMB_130_NOTEBOOK11_ACQUISITION_SOURCE_AUDIT.csv"),
        index=False
    )
else:
    approved = pd.DataFrame()
    print("\nWARNING: Combined acquisition source file not found.")

# ------------------------------------------------------------
# 4. AGGREGATE SENTIMENT BASELINE BY MOVIE
# ------------------------------------------------------------

sentiment_path = os.path.join(
    BASE, "01_data", "processed", "audience_sentiment",
    "ICMB_130_YOUTUBE_COMMENTS_SENTIMENT_BASELINE.csv"
)

if not os.path.exists(sentiment_path):
    raise FileNotFoundError(f"Sentiment baseline not found: {sentiment_path}")

sentiment = pd.read_csv(sentiment_path, low_memory=False)

print("\nSENTIMENT BASELINE")
print("Shape:", sentiment.shape)
print("Columns:", sentiment.columns.tolist())

if "movie_record_id" not in sentiment.columns:
    raise ValueError("Sentiment baseline does not contain movie_record_id.")

sentiment["movie_record_id"] = sentiment["movie_record_id"].astype(str)

# Count sentiment labels by movie.
sentiment_summary = sentiment.groupby(
    "movie_record_id", dropna=False
).size().rename("sentiment_comment_rows").reset_index()

if "final_sentiment" in sentiment.columns:
    sentiment_labels = pd.crosstab(
        sentiment["movie_record_id"],
        sentiment["final_sentiment"].fillna("Missing")
    ).reset_index()

    sentiment_summary = sentiment_summary.merge(
        sentiment_labels,
        on="movie_record_id",
        how="left"
    )

if "video_id" in sentiment.columns:
    video_summary = sentiment.groupby("movie_record_id")[
        "video_id"
    ].nunique().rename("sentiment_unique_videos").reset_index()

    sentiment_summary = sentiment_summary.merge(
        video_summary,
        on="movie_record_id",
        how="left"
    )

print("\nSENTIMENT MOVIE-LEVEL SUMMARY")
print("Unique movie records:", sentiment_summary["movie_record_id"].nunique())
print("Total sentiment rows:", int(sentiment_summary["sentiment_comment_rows"].sum()))
print("\nSentiment comment count distribution:")
print(sentiment_summary["sentiment_comment_rows"].describe().to_string())

# ------------------------------------------------------------
# 5. JOIN COVERAGE WITH SENTIMENT COUNTS
# ------------------------------------------------------------

coverage["movie_record_id"] = coverage["movie_record_id"].astype(str)

reconciliation = coverage.merge(
    sentiment_summary,
    on="movie_record_id",
    how="outer",
    indicator=True
)

reconciliation["coverage_total_comments"] = pd.to_numeric(
    reconciliation["total_comments"], errors="coerce"
)

reconciliation["sentiment_comment_rows"] = pd.to_numeric(
    reconciliation["sentiment_comment_rows"], errors="coerce"
)

reconciliation["comment_count_difference"] = (
    reconciliation["sentiment_comment_rows"] -
    reconciliation["coverage_total_comments"]
)

print("\nCOVERAGE ↔ SENTIMENT RECONCILIATION")
print("Rows:", len(reconciliation))
print("Join status:")
print(reconciliation["_merge"].value_counts(dropna=False))
print("Coverage films:", coverage["movie_record_id"].nunique())
print("Sentiment films:", sentiment_summary["movie_record_id"].nunique())
print(
    "Films with coverage but no sentiment rows:",
    int(
        (
            reconciliation["_merge"].eq("left_only")
        ).sum()
    )
)
print(
    "Films with sentiment rows but no coverage record:",
    int(
        (
            reconciliation["_merge"].eq("right_only")
        ).sum()
    )
)
print(
    "Films where comment counts differ:",
    int(
        reconciliation["comment_count_difference"]
        .fillna(0).ne(0).sum()
    )
)

print("\nLARGEST ABSOLUTE COUNT DIFFERENCES")
display(
    reconciliation.assign(
        abs_difference=reconciliation["comment_count_difference"].abs()
    ).sort_values("abs_difference", ascending=False).head(15)
)

# ------------------------------------------------------------
# 6. SAVE AUDIT TABLES AND SUMMARY
# ------------------------------------------------------------

coverage.to_csv(
    os.path.join(TABLES, "ICMB_130_NOTEBOOK11_FILM_LEVEL_YOUTUBE_COVERAGE_AUDIT.csv"),
    index=False
)

sentiment_summary.to_csv(
    os.path.join(TABLES, "ICMB_130_NOTEBOOK11_FILM_LEVEL_SENTIMENT_SUMMARY.csv"),
    index=False
)

reconciliation.to_csv(
    os.path.join(TABLES, "ICMB_130_NOTEBOOK11_YOUTUBE_SENTIMENT_RECONCILIATION.csv"),
    index=False
)

summary = {
    "primary_coverage_file": primary_coverage_file,
    "coverage_rows": int(len(coverage)),
    "coverage_unique_movies": int(coverage["movie_record_id"].nunique()),
    "coverage_total_comments": int(coverage["total_comments"].sum()),
    "coverage_zero_comment_movies": int((coverage["total_comments"] == 0).sum()),
    "coverage_median_comments_per_movie": float(coverage["total_comments"].median()),
    "sentiment_total_rows": int(len(sentiment)),
    "sentiment_unique_movies": int(sentiment["movie_record_id"].nunique()),
    "reconciliation_rows": int(len(reconciliation)),
    "reconciliation_count_mismatches": int(
        reconciliation["comment_count_difference"].fillna(0).ne(0).sum()
    ),
    "acquisition_source_rows": int(len(approved)),
    "acquisition_source_unique_movies": (
        int(approved["movie_record_id"].nunique())
        if not approved.empty and "movie_record_id" in approved.columns else 0
    ),
    "status": "Cell 8 reconciliation completed"
}

summary_path = os.path.join(
    SUMMARY, "ICMB_130_NOTEBOOK11_CELL8_YOUTUBE_RECONCILIATION.json"
)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print("\n" + "=" * 90)
print("CELL 8 COMPLETED")
print("=" * 90)
print(json.dumps(summary, indent=4))
print("\nSaved audit tables in:", TABLES)
print("Saved summary:", summary_path)


In [ ]:

# =============================================================================
# NOTEBOOK 11 — CELL 9
# COVERAGE DIFFERENCE INVESTIGATION & YOUTUBE ACQUISITION SOURCE AUDIT
# =============================================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

print("=" * 90)
print("CELL 9 — COVERAGE DIFFERENCE AND ACQUISITION SOURCE AUDIT")
print("=" * 90)

# ---------------------------------------------------------------------------
# 1. PATHS
# ---------------------------------------------------------------------------

TABLE_DIR = PROJECT_ROOT / "05_outputs" / "tables"
SUMMARY_DIR = PROJECT_ROOT / "05_outputs" / "summary"

coverage_1_path = (
    PROJECT_ROOT / "06_outputs" / "audience_sentiment" /
    "ICMB_130_FINAL_YOUTUBE_COMMENT_COVERAGE_BATCH_01_15.csv"
)

coverage_2_path = (
    PROJECT_ROOT / "06_outputs" / "audience_sentiment" /
    "ICMB_130_FINAL_YOUTUBE_COMMENT_COVERAGE_BATCH_01_16.csv"
)

approved_sources_path = (
    TABLE_DIR / "ICMB_130_NOTEBOOK11_APPROVED_YOUTUBE_SOURCES.csv"
)

# ---------------------------------------------------------------------------
# 2. LOAD COVERAGE REPORTS
# ---------------------------------------------------------------------------

cov1 = pd.read_csv(coverage_1_path)
cov2 = pd.read_csv(coverage_2_path)

cov1["movie_record_id"] = cov1["movie_record_id"].astype(str)
cov2["movie_record_id"] = cov2["movie_record_id"].astype(str)

comparison = cov1.merge(
    cov2,
    on="movie_record_id",
    how="outer",
    suffixes=("_report15", "_report16"),
    indicator=True
)

comparison["comments_difference"] = (
    comparison["total_comments_report15"] -
    comparison["total_comments_report16"]
)

comparison["videos_difference"] = (
    comparison["unique_videos_report15"] -
    comparison["unique_videos_report16"]
)

differences = comparison[
    (comparison["comments_difference"] != 0) |
    (comparison["videos_difference"] != 0) |
    (comparison["_merge"] != "both")
].copy()

print("\n1. COVERAGE REPORT DIFFERENCES")
print("-" * 70)
print("Number of differing records:", len(differences))

display_columns = [
    "movie_record_id",
    "movie_title_report15",
    "year_report15",
    "total_comments_report15",
    "total_comments_report16",
    "comments_difference",
    "unique_videos_report15",
    "unique_videos_report16",
    "videos_difference",
    "_merge"
]

display(
    differences[
        [c for c in display_columns if c in differences.columns]
    ]
)

# ---------------------------------------------------------------------------
# 3. LOAD THE ACTUAL ACQUISITION SOURCE DATASET
# ---------------------------------------------------------------------------

print("\n2. ACTUAL ACQUISITION SOURCE DATASET")
print("-" * 70)

if not approved_sources_path.exists():
    raise FileNotFoundError(
        f"Actual acquisition source file not found: {approved_sources_path}"
    )

sources = pd.read_csv(approved_sources_path, low_memory=False)

print("Shape:", sources.shape)
print("Columns:", sources.columns.tolist())

# Identify likely movie identifier and status columns
movie_id_candidates = [
    "movie_record_id", "movie_id", "record_id"
]

movie_id_col = next(
    (c for c in movie_id_candidates if c in sources.columns),
    None
)

print("\nMovie identifier column:", movie_id_col)

if movie_id_col:
    sources[movie_id_col] = sources[movie_id_col].astype(str)
    print("Rows:", len(sources))
    print("Unique movie IDs:", sources[movie_id_col].nunique())
    print("Missing movie IDs:", sources[movie_id_col].isna().sum())

    print("\nRows per movie ID:")
    print(sources[movie_id_col].value_counts().head(15))

else:
    print(
        "WARNING: No recognized movie ID column found. "
        "Inspect the displayed schema before drawing conclusions."
    )

# ---------------------------------------------------------------------------
# 4. INSPECT ACQUISITION AND VERIFICATION STATUS COLUMNS
# ---------------------------------------------------------------------------

print("\n3. ACQUISITION STATUS DISTRIBUTIONS")
print("-" * 70)

status_columns = [
    c for c in sources.columns
    if any(
        keyword in c.lower()
        for keyword in [
            "status", "approval", "decision",
            "verification", "review", "result"
        ]
    )
]

for col in status_columns:
    print(f"\n{col}")
    print(sources[col].fillna("Missing").value_counts().head(20))

# ---------------------------------------------------------------------------
# 5. CHECK DUPLICATE SOURCE RECORDS
# ---------------------------------------------------------------------------

print("\n4. DUPLICATE AND VIDEO ID AUDIT")
print("-" * 70)

video_id_col = next(
    (c for c in ["video_id", "youtube_video_id"] if c in sources.columns),
    None
)

if video_id_col:
    print("Video ID column:", video_id_col)
    print("Rows with video IDs:", sources[video_id_col].notna().sum())
    print("Unique video IDs:", sources[video_id_col].nunique())
    print("Duplicate video ID rows:", sources[video_id_col].duplicated().sum())
else:
    print("No recognized video ID column found.")

# ---------------------------------------------------------------------------
# 6. SAVE AUDIT OUTPUTS
# ---------------------------------------------------------------------------

differences.to_csv(
    TABLE_DIR / "ICMB_130_NOTEBOOK11_COVERAGE_REPORT_DIFFERENCES.csv",
    index=False
)

sources.to_csv(
    TABLE_DIR / "ICMB_130_NOTEBOOK11_ACQUISITION_SOURCE_DATA_AUDIT.csv",
    index=False
)

# ---------------------------------------------------------------------------
# 7. SUMMARY
# ---------------------------------------------------------------------------

summary = {
    "coverage_report_15_rows": int(len(cov1)),
    "coverage_report_16_rows": int(len(cov2)),
    "coverage_difference_records": int(len(differences)),
    "acquisition_source_rows": int(len(sources)),
    "acquisition_source_columns": int(len(sources.columns)),
    "movie_id_column": movie_id_col,
    "unique_movie_ids": (
        int(sources[movie_id_col].nunique())
        if movie_id_col else None
    ),
    "video_id_column": video_id_col,
    "unique_video_ids": (
        int(sources[video_id_col].nunique())
        if video_id_col else None
    ),
    "status_columns_inspected": status_columns
}

with open(
    SUMMARY_DIR / "ICMB_130_NOTEBOOK11_CELL9_AUDIT.json",
    "w"
) as f:
    json.dump(summary, f, indent=4, default=str)

print("\n" + "=" * 90)
print("CELL 9 COMPLETED")
print("=" * 90)
print(json.dumps(summary, indent=4, default=str))

print("\nSaved audit outputs in:", TABLE_DIR)


In [ ]:

# =============================================================================
# NOTEBOOK 11 — CELL 9B
# LOAD AND AUDIT THE ACTUAL YOUTUBE ACQUISITION SOURCE FILES
# =============================================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

print("=" * 90)
print("CELL 9B — ACTUAL YOUTUBE ACQUISITION SOURCE CONSOLIDATION")
print("=" * 90)

TABLE_DIR = PROJECT_ROOT / "05_outputs" / "tables"
SUMMARY_DIR = PROJECT_ROOT / "05_outputs" / "summary"

catalogue_path = (
    TABLE_DIR / "ICMB_130_NOTEBOOK11_YOUTUBE_MANIFEST_CATALOGUE.csv"
)

# ---------------------------------------------------------------------------
# 1. LOAD THE FILE CATALOGUE
# ---------------------------------------------------------------------------

if not catalogue_path.exists():
    raise FileNotFoundError(f"Catalogue not found: {catalogue_path}")

catalogue = pd.read_csv(catalogue_path)

print("\nCATALOGUE")
print("Rows:", len(catalogue))
print("Columns:", catalogue.columns.tolist())

required_catalogue_columns = ["file", "relative_path"]

missing = [
    c for c in required_catalogue_columns
    if c not in catalogue.columns
]

if missing:
    raise ValueError(f"Missing catalogue columns: {missing}")

# ---------------------------------------------------------------------------
# 2. LOCATE AND LOAD THE ACTUAL CSV FILES
# ---------------------------------------------------------------------------

loaded_frames = []
file_audit = []

for _, row in catalogue.iterrows():

    file_name = str(row["file"])
    relative_path = str(row["relative_path"])

    source_path = Path(relative_path)

    # Support both absolute paths and paths relative to PROJECT_ROOT
    if not source_path.is_absolute():
        source_path = PROJECT_ROOT / source_path

    audit_record = {
        "file": file_name,
        "relative_path": relative_path,
        "resolved_path": str(source_path),
        "exists": source_path.exists(),
        "loaded": False,
        "rows_loaded": 0,
        "columns_loaded": 0,
        "error": None
    }

    if not source_path.exists():
        audit_record["error"] = "File not found"
        file_audit.append(audit_record)
        continue

    try:
        df = pd.read_csv(source_path, low_memory=False)

        # Preserve source lineage
        df["source_file"] = file_name
        df["source_relative_path"] = relative_path

        loaded_frames.append(df)

        audit_record["loaded"] = True
        audit_record["rows_loaded"] = len(df)
        audit_record["columns_loaded"] = len(df.columns)

    except Exception as e:
        audit_record["error"] = str(e)

    file_audit.append(audit_record)

file_audit_df = pd.DataFrame(file_audit)

print("\nFILE LOADING AUDIT")
display(file_audit_df)

print("\nFiles listed:", len(catalogue))
print("Files loaded:", len(loaded_frames))
print("Files missing or failed:", len(catalogue) - len(loaded_frames))

if not loaded_frames:
    raise RuntimeError(
        "No acquisition source files were loaded. "
        "Check the resolved paths in the file audit."
    )

# ---------------------------------------------------------------------------
# 3. CONSOLIDATE THE SOURCE RECORDS
# ---------------------------------------------------------------------------

acquisition = pd.concat(
    loaded_frames,
    ignore_index=True,
    sort=False
)

print("\nCONSOLIDATED ACQUISITION SOURCE DATA")
print("Shape:", acquisition.shape)
print("Columns:")
print(acquisition.columns.tolist())

# ---------------------------------------------------------------------------
# 4. IDENTIFY MOVIE AND VIDEO IDENTIFIERS
# ---------------------------------------------------------------------------

movie_candidates = [
    "movie_record_id", "movie_id", "record_id"
]

video_candidates = [
    "video_id", "youtube_video_id"
]

movie_id_col = next(
    (c for c in movie_candidates if c in acquisition.columns),
    None
)

video_id_col = next(
    (c for c in video_candidates if c in acquisition.columns),
    None
)

print("\nIDENTIFIER AUDIT")
print("Movie ID column:", movie_id_col)
print("Video ID column:", video_id_col)

if movie_id_col:
    print(
        "Unique movie IDs:",
        acquisition[movie_id_col].dropna().astype(str).nunique()
    )
    print(
        "Missing movie IDs:",
        acquisition[movie_id_col].isna().sum()
    )

if video_id_col:
    print(
        "Rows with video IDs:",
        acquisition[video_id_col].notna().sum()
    )
    print(
        "Unique video IDs:",
        acquisition[video_id_col].dropna().astype(str).nunique()
    )
    print(
        "Duplicate video ID rows:",
        acquisition[video_id_col].dropna().astype(str).duplicated().sum()
    )

# ---------------------------------------------------------------------------
# 5. INSPECT ALL STATUS AND VERIFICATION FIELDS
# ---------------------------------------------------------------------------

status_columns = [
    c for c in acquisition.columns
    if any(
        word in c.lower()
        for word in [
            "status", "approval", "decision",
            "verification", "review", "result"
        ]
    )
]

print("\nSTATUS FIELD DISTRIBUTIONS")

for col in status_columns:
    print(f"\n{col}")
    print(
        acquisition[col]
        .fillna("Missing")
        .astype(str)
        .value_counts()
        .head(25)
    )

# ---------------------------------------------------------------------------
# 6. MOVIE-LEVEL SOURCE COUNTS
# ---------------------------------------------------------------------------

if movie_id_col:

    movie_source_counts = (
        acquisition
        .dropna(subset=[movie_id_col])
        .groupby(movie_id_col)
        .agg(
            source_rows=(movie_id_col, "size"),
            source_files=("source_file", "nunique")
        )
        .reset_index()
        .rename(columns={movie_id_col: "movie_record_id"})
    )

    if video_id_col:
        unique_video_counts = (
            acquisition
            .dropna(subset=[movie_id_col, video_id_col])
            .groupby(movie_id_col)[video_id_col]
            .nunique()
            .reset_index(name="unique_video_ids")
            .rename(columns={movie_id_col: "movie_record_id"})
        )

        movie_source_counts = movie_source_counts.merge(
            unique_video_counts,
            on="movie_record_id",
            how="left"
        )

    print("\nMOVIE-LEVEL SOURCE COUNTS")
    print("Movies represented:", len(movie_source_counts))
    display(movie_source_counts.head(20))

else:
    movie_source_counts = pd.DataFrame()
    print("\nMovie-level counts cannot be calculated: no movie ID column.")

# ---------------------------------------------------------------------------
# 7. SAVE OUTPUTS
# ---------------------------------------------------------------------------

file_audit_df.to_csv(
    TABLE_DIR / "ICMB_130_NOTEBOOK11_ACQUISITION_FILE_LOADING_AUDIT.csv",
    index=False
)

acquisition.to_csv(
    TABLE_DIR / "ICMB_130_NOTEBOOK11_ACQUISITION_SOURCE_CONSOLIDATED.csv",
    index=False
)

if not movie_source_counts.empty:
    movie_source_counts.to_csv(
        TABLE_DIR / "ICMB_130_NOTEBOOK11_ACQUISITION_MOVIE_LEVEL_COUNTS.csv",
        index=False
    )

summary = {
    "catalogue_files": int(len(catalogue)),
    "files_loaded": int(len(loaded_frames)),
    "files_failed_or_missing": int(len(catalogue) - len(loaded_frames)),
    "consolidated_rows": int(len(acquisition)),
    "consolidated_columns": int(len(acquisition.columns)),
    "movie_id_column": movie_id_col,
    "video_id_column": video_id_col,
    "unique_movie_ids": (
        int(acquisition[movie_id_col].dropna().astype(str).nunique())
        if movie_id_col else None
    ),
    "unique_video_ids": (
        int(acquisition[video_id_col].dropna().astype(str).nunique())
        if video_id_col else None
    ),
    "status_columns": status_columns
}

with open(
    SUMMARY_DIR / "ICMB_130_NOTEBOOK11_CELL9B_ACQUISITION_AUDIT.json",
    "w"
) as f:
    json.dump(summary, f, indent=4, default=str)

print("\n" + "=" * 90)
print("CELL 9B COMPLETED")
print("=" * 90)
print(json.dumps(summary, indent=4, default=str))


In [ ]:

# =============================================================================
# NOTEBOOK 11 — CELL 10
# FINAL RESEARCH RESULTS CONSOLIDATION
# =============================================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

print("=" * 90)
print("CELL 10 — FINAL RESEARCH RESULTS CONSOLIDATION")
print("=" * 90)

# ---------------------------------------------------------------------------
# 1. DEFINE INPUT PATHS
# ---------------------------------------------------------------------------

PROCESSED_DIR = PROJECT_ROOT / "01_data" / "processed"
TOPIC_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"
SENTIMENT_DIR = PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"

TABLE_DIR = PROJECT_ROOT / "05_outputs" / "tables"
SUMMARY_DIR = PROJECT_ROOT / "05_outputs" / "summary"

master_path = (
    PROCESSED_DIR / "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

features_path = PROCESSED_DIR / "ICMB_130_FEATURES.csv"

theme_register_path = (
    TOPIC_DIR / "ICMB_130_FINAL_12_THEME_REGISTER.csv"
)

annual_theme_path = (
    TOPIC_DIR / "ICMB_130_FINAL_ANNUAL_THEME_PREVALENCE.csv"
)

longitudinal_path = (
    TOPIC_DIR / "ICMB_130_FINAL_LONGITUDINAL_THEME_SUMMARY.csv"
)

sentiment_path = (
    SENTIMENT_DIR / "ICMB_130_YOUTUBE_COMMENTS_SENTIMENT_BASELINE.csv"
)

# ---------------------------------------------------------------------------
# 2. VERIFY REQUIRED INPUT FILES
# ---------------------------------------------------------------------------

input_paths = {
    "Repaired master": master_path,
    "Feature dataset": features_path,
    "Final 12-theme register": theme_register_path,
    "Annual theme prevalence": annual_theme_path,
    "Longitudinal theme summary": longitudinal_path,
    "Sentiment baseline": sentiment_path
}

print("\n1. INPUT FILE AVAILABILITY")
print("-" * 70)

missing_files = []

for name, path in input_paths.items():
    exists = path.exists()
    print(f"{name}: {'FOUND' if exists else 'MISSING'}")

    if not exists:
        missing_files.append(str(path))

if missing_files:
    raise FileNotFoundError(
        "Required inputs missing:\n" + "\n".join(missing_files)
    )

# ---------------------------------------------------------------------------
# 3. LOAD DATASETS
# ---------------------------------------------------------------------------

master = pd.read_csv(master_path)
features = pd.read_csv(features_path)
theme_register = pd.read_csv(theme_register_path)
annual_theme = pd.read_csv(annual_theme_path)
longitudinal = pd.read_csv(longitudinal_path)
sentiment = pd.read_csv(sentiment_path, low_memory=False)

datasets = {
    "Master": master,
    "Features": features,
    "Theme register": theme_register,
    "Annual theme prevalence": annual_theme,
    "Longitudinal theme summary": longitudinal,
    "Sentiment baseline": sentiment
}

print("\n2. DATASET DIMENSIONS")
print("-" * 70)

for name, df in datasets.items():
    print(f"{name:30s} Shape: {df.shape}")

# ---------------------------------------------------------------------------
# 4. MASTER DATA VALIDATION
# ---------------------------------------------------------------------------

print("\n3. MASTER DATA VALIDATION")
print("-" * 70)

master["movie_record_id"] = master["movie_record_id"].astype(str)

print("Movie records:", len(master))
print("Unique movie IDs:", master["movie_record_id"].nunique())
print("Duplicate movie IDs:", master["movie_record_id"].duplicated().sum())
print("Year range:", master["year"].min(), "-", master["year"].max())
print("Missing movie titles:", master["movie_title"].isna().sum())

if len(master) != 130:
    raise ValueError("Master dataset does not contain 130 rows.")

if master["movie_record_id"].nunique() != 130:
    raise ValueError("Master dataset has duplicate or missing movie IDs.")

# Year-wise film counts
yearly_films = (
    master.groupby("year")
    .agg(
        film_count=("movie_record_id", "nunique"),
        mean_india_gross=("india_gross_cr", "mean"),
        median_india_gross=("india_gross_cr", "median"),
        total_india_gross=("india_gross_cr", "sum")
    )
    .reset_index()
)

print("\nYear-wise film count summary:")
print(yearly_films["film_count"].describe())

# ---------------------------------------------------------------------------
# 5. FEATURE DATA VALIDATION
# ---------------------------------------------------------------------------

print("\n4. FEATURE DATA VALIDATION")
print("-" * 70)

features["movie_record_id"] = features["movie_record_id"].astype(str)

feature_master_check = master[
    ["movie_record_id", "year", "rank", "movie_title", "language"]
].merge(
    features[
        ["movie_record_id", "year", "rank", "movie_title", "language"]
    ],
    on="movie_record_id",
    how="outer",
    suffixes=("_master", "_features"),
    indicator=True
)

print("Feature records:", len(features))
print("Unique feature movie IDs:", features["movie_record_id"].nunique())
print("Master-feature join:")
print(feature_master_check["_merge"].value_counts())

# ---------------------------------------------------------------------------
# 6. TOPIC MODEL RESULTS VALIDATION
# ---------------------------------------------------------------------------

print("\n5. TOPIC MODEL RESULTS VALIDATION")
print("-" * 70)

print("Curated themes:", len(theme_register))
print("Annual prevalence rows:", len(annual_theme))
print("Longitudinal summary rows:", len(longitudinal))

theme_col = next(
    (
        c for c in ["theme", "theme_name", "curated_theme"]
        if c in theme_register.columns
    ),
    None
)

print("Theme name column:", theme_col)

if theme_col:
    print("Unique theme names:", theme_register[theme_col].nunique())
    print("\nTheme register preview:")
    display(theme_register.head(12))

print("\nAnnual prevalence columns:")
print(annual_theme.columns.tolist())

print("\nLongitudinal summary columns:")
print(longitudinal.columns.tolist())

# ---------------------------------------------------------------------------
# 7. SENTIMENT DATA VALIDATION
# ---------------------------------------------------------------------------

print("\n6. SENTIMENT DATA VALIDATION")
print("-" * 70)

sentiment["movie_record_id"] = sentiment["movie_record_id"].astype(str)

print("Sentiment rows:", len(sentiment))
print(
    "Unique sentiment movie IDs:",
    sentiment["movie_record_id"].nunique()
)

print("\nFinal sentiment distribution:")
print(sentiment["final_sentiment"].value_counts(dropna=False))

print("\nReliability group distribution:")
print(
    sentiment["sentiment_reliability_group"]
    .value_counts(dropna=False)
)

# Film-level sentiment aggregation
film_sentiment = (
    sentiment.groupby("movie_record_id")
    .agg(
        sentiment_comment_count=("movie_record_id", "size"),
        positive_comments=(
            "final_sentiment",
            lambda x: (x == "positive").sum()
        ),
        negative_comments=(
            "final_sentiment",
            lambda x: (x == "negative").sum()
        ),
        neutral_comments=(
            "final_sentiment",
            lambda x: (x == "neutral").sum()
        ),
        uncertain_comments=(
            "final_sentiment",
            lambda x: (x == "Uncertain").sum()
        ),
        emoji_only_comments=(
            "final_sentiment",
            lambda x: (x == "Emoji_Only_Not_Modelled").sum()
        )
    )
    .reset_index()
)

# ---------------------------------------------------------------------------
# 8. BUILD CONSOLIDATED FILM-LEVEL RESULTS
# ---------------------------------------------------------------------------

print("\n7. BUILDING CONSOLIDATED FILM-LEVEL RESULTS")
print("-" * 70)

film_results = master.merge(
    features,
    on="movie_record_id",
    how="left",
    suffixes=("_master", "_features"),
    validate="one_to_one"
)

film_results = film_results.merge(
    film_sentiment,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

print("Consolidated shape:", film_results.shape)
print("Unique films:", film_results["movie_record_id"].nunique())
print(
    "Films without sentiment aggregation:",
    film_results["sentiment_comment_count"].isna().sum()
)

# ---------------------------------------------------------------------------
# 9. SAVE RESEARCH RESULTS TABLES
# ---------------------------------------------------------------------------

yearly_films.to_csv(
    TABLE_DIR / "ICMB_130_FINAL_YEARLY_BOX_OFFICE_SUMMARY.csv",
    index=False
)

film_sentiment.to_csv(
    TABLE_DIR / "ICMB_130_FINAL_FILM_LEVEL_SENTIMENT_SUMMARY.csv",
    index=False
)

film_results.to_csv(
    TABLE_DIR / "ICMB_130_FINAL_CONSOLIDATED_FILM_RESULTS.csv",
    index=False
)

# ---------------------------------------------------------------------------
# 10. SAVE CELL SUMMARY
# ---------------------------------------------------------------------------

summary = {
    "master_rows": int(len(master)),
    "master_unique_movies": int(master["movie_record_id"].nunique()),
    "master_year_min": int(master["year"].min()),
    "master_year_max": int(master["year"].max()),
    "feature_rows": int(len(features)),
    "feature_unique_movies": int(features["movie_record_id"].nunique()),
    "theme_register_rows": int(len(theme_register)),
    "annual_theme_rows": int(len(annual_theme)),
    "longitudinal_summary_rows": int(len(longitudinal)),
    "sentiment_rows": int(len(sentiment)),
    "sentiment_unique_movies": int(sentiment["movie_record_id"].nunique()),
    "film_level_sentiment_rows": int(len(film_sentiment)),
    "consolidated_film_results_rows": int(len(film_results)),
    "status": "Cell 10 research-results consolidation completed"
}

with open(
    SUMMARY_DIR / "ICMB_130_NOTEBOOK11_CELL10_SUMMARY.json",
    "w"
) as f:
    json.dump(summary, f, indent=4, default=str)

print("\n" + "=" * 90)
print("CELL 10 COMPLETED")
print("=" * 90)
print(json.dumps(summary, indent=4, default=str))

print("\nSaved final research tables in:", TABLE_DIR)


In [ ]:

# =============================================================================
# NOTEBOOK 11 — CELL 11
# FINAL VISUALIZATIONS AND DESCRIPTIVE RESEARCH ANALYSIS
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

print("=" * 90)
print("CELL 11 — FINAL VISUALIZATIONS AND DESCRIPTIVE ANALYSIS")
print("=" * 90)

# ---------------------------------------------------------------------------
# 1. DIRECTORIES AND INPUT FILES
# ---------------------------------------------------------------------------

FIGURE_DIR = PROJECT_ROOT / "05_outputs" / "figures"
TABLE_DIR = PROJECT_ROOT / "05_outputs" / "tables"
SUMMARY_DIR = PROJECT_ROOT / "05_outputs" / "summary"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

master_path = (
    PROJECT_ROOT / "01_data" / "processed" /
    "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

annual_theme_path = (
    PROJECT_ROOT / "06_outputs" / "topic_modelling" /
    "ICMB_130_FINAL_ANNUAL_THEME_PREVALENCE.csv"
)

longitudinal_path = (
    PROJECT_ROOT / "06_outputs" / "topic_modelling" /
    "ICMB_130_FINAL_LONGITUDINAL_THEME_SUMMARY.csv"
)

sentiment_path = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment" /
    "ICMB_130_YOUTUBE_COMMENTS_SENTIMENT_BASELINE.csv"
)

# ---------------------------------------------------------------------------
# 2. LOAD DATA
# ---------------------------------------------------------------------------

master = pd.read_csv(master_path)
annual_theme = pd.read_csv(annual_theme_path)
longitudinal = pd.read_csv(longitudinal_path)
sentiment = pd.read_csv(sentiment_path, low_memory=False)

master["year"] = pd.to_numeric(master["year"], errors="coerce")
master["india_gross_cr"] = pd.to_numeric(
    master["india_gross_cr"], errors="coerce"
)

annual_theme["year"] = pd.to_numeric(
    annual_theme["year"], errors="coerce"
)

annual_theme["dialogue_prevalence_pct"] = pd.to_numeric(
    annual_theme["dialogue_prevalence_pct"], errors="coerce"
)

print("\nINPUT DIMENSIONS")
print("Master:", master.shape)
print("Annual themes:", annual_theme.shape)
print("Longitudinal:", longitudinal.shape)
print("Sentiment:", sentiment.shape)

# ---------------------------------------------------------------------------
# 3. CHART 1 — ANNUAL INDIA GROSS TREND
# ---------------------------------------------------------------------------

yearly_gross = (
    master.groupby("year")
    .agg(
        film_count=("movie_record_id", "nunique"),
        mean_india_gross_cr=("india_gross_cr", "mean"),
        median_india_gross_cr=("india_gross_cr", "median"),
        total_india_gross_cr=("india_gross_cr", "sum")
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(
    yearly_gross["year"],
    yearly_gross["mean_india_gross_cr"],
    marker="o",
    label="Mean India Gross"
)

ax.plot(
    yearly_gross["year"],
    yearly_gross["median_india_gross_cr"],
    marker="s",
    label="Median India Gross"
)

ax.set_title(
    "Annual India Gross Trends of Top 5 Indian Films (2000–2025)"
)
ax.set_xlabel("Year")
ax.set_ylabel("India Gross (INR crore)")
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

fig.savefig(
    FIGURE_DIR / "ICMB_130_FIG01_ANNUAL_INDIA_GROSS_TREND.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# ---------------------------------------------------------------------------
# 4. CHART 2 — TOP 10 FILMS BY INDIA GROSS
# ---------------------------------------------------------------------------

top_10 = (
    master.nlargest(10, "india_gross_cr")[
        ["movie_title", "year", "language", "india_gross_cr"]
    ]
    .sort_values("india_gross_cr")
)

fig, ax = plt.subplots(figsize=(12, 7))

ax.barh(
    top_10["movie_title"] + " (" + top_10["year"].astype(int).astype(str) + ")",
    top_10["india_gross_cr"]
)

ax.set_title("Top 10 Films by India Gross — ICMB-130")
ax.set_xlabel("India Gross (INR crore)")
ax.set_ylabel("Film")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

fig.savefig(
    FIGURE_DIR / "ICMB_130_FIG02_TOP10_INDIA_GROSS.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# ---------------------------------------------------------------------------
# 5. CHART 3 — ANNUAL CURATED THEME PREVALENCE
# ---------------------------------------------------------------------------

top_themes = (
    annual_theme.groupby("curated_theme")[
        "dialogue_prevalence_pct"
    ]
    .mean()
    .nlargest(6)
    .index.tolist()
)

annual_top_themes = annual_theme[
    annual_theme["curated_theme"].isin(top_themes)
].copy()

fig, ax = plt.subplots(figsize=(14, 7))

for theme in top_themes:
    theme_data = annual_top_themes[
        annual_top_themes["curated_theme"] == theme
    ].sort_values("year")

    ax.plot(
        theme_data["year"],
        theme_data["dialogue_prevalence_pct"],
        marker="o",
        markersize=3,
        label=theme
    )

ax.set_title(
    "Annual Dialogue Prevalence of the Six Most Prevalent Curated Themes"
)
ax.set_xlabel("Year")
ax.set_ylabel("Dialogue Prevalence (%)")
ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
ax.grid(alpha=0.3)
plt.tight_layout()

fig.savefig(
    FIGURE_DIR / "ICMB_130_FIG03_ANNUAL_THEME_PREVALENCE.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# ---------------------------------------------------------------------------
# 6. CHART 4 — LONGITUDINAL PERIOD COMPARISON
# ---------------------------------------------------------------------------

period_columns = [
    "Early (2000–2008) Dialogue Share (%)",
    "Middle (2009–2017) Dialogue Share (%)",
    "Recent (2018–2025) Dialogue Share (%)"
]

missing_period_columns = [
    c for c in period_columns if c not in longitudinal.columns
]

if missing_period_columns:
    raise ValueError(
        f"Missing longitudinal columns: {missing_period_columns}"
    )

period_data = longitudinal[
    ["Theme"] + period_columns
].copy()

period_data = period_data.set_index("Theme")

fig, ax = plt.subplots(figsize=(15, 8))

period_data.plot(
    kind="bar",
    ax=ax,
    width=0.8
)

ax.set_title(
    "Longitudinal Comparison of Curated Dialogue Themes (2000–2025)"
)
ax.set_xlabel("Curated Theme")
ax.set_ylabel("Dialogue Share (%)")
ax.legend(title="Period")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=55, ha="right")
plt.tight_layout()

fig.savefig(
    FIGURE_DIR / "ICMB_130_FIG04_LONGITUDINAL_THEME_COMPARISON.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# ---------------------------------------------------------------------------
# 7. CHART 5 — FULL SENTIMENT CLASSIFICATION DISTRIBUTION
# ---------------------------------------------------------------------------

sentiment_counts = (
    sentiment["final_sentiment"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("sentiment_class")
    .reset_index(name="comment_count")
)

sentiment_counts["percentage"] = (
    sentiment_counts["comment_count"] / len(sentiment) * 100
)

fig, ax = plt.subplots(figsize=(11, 6))

ax.bar(
    sentiment_counts["sentiment_class"],
    sentiment_counts["comment_count"]
)

ax.set_title(
    "YouTube Comment Sentiment Classification — All Records"
)
ax.set_xlabel("Final Sentiment Classification")
ax.set_ylabel("Number of Comments")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

fig.savefig(
    FIGURE_DIR / "ICMB_130_FIG05_FULL_SENTIMENT_DISTRIBUTION.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# ---------------------------------------------------------------------------
# 8. CHART 6 — CLASSIFIABLE SENTIMENT ONLY
# ---------------------------------------------------------------------------

classifiable_labels = ["positive", "negative", "neutral"]

classifiable = sentiment[
    sentiment["final_sentiment"].isin(classifiable_labels)
].copy()

classifiable_counts = (
    classifiable["final_sentiment"]
    .value_counts()
    .reindex(classifiable_labels, fill_value=0)
    .rename_axis("sentiment_class")
    .reset_index(name="comment_count")
)

classifiable_total = len(classifiable)

classifiable_counts["percentage_of_classifiable"] = (
    classifiable_counts["comment_count"] /
    classifiable_total * 100
)

fig, ax = plt.subplots(figsize=(9, 6))

ax.bar(
    classifiable_counts["sentiment_class"],
    classifiable_counts["comment_count"]
)

ax.set_title(
    "Positive, Negative and Neutral Comments Among Classifiable Records"
)
ax.set_xlabel("Sentiment")
ax.set_ylabel("Number of Comments")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

fig.savefig(
    FIGURE_DIR / "ICMB_130_FIG06_CLASSIFIABLE_SENTIMENT.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# ---------------------------------------------------------------------------
# 9. SAVE SUPPORTING RESEARCH TABLES
# ---------------------------------------------------------------------------

yearly_gross.to_csv(
    TABLE_DIR / "ICMB_130_FINAL_YEARLY_GROSS_VISUALIZATION_DATA.csv",
    index=False
)

top_10.to_csv(
    TABLE_DIR / "ICMB_130_FINAL_TOP10_FILMS_BY_INDIA_GROSS.csv",
    index=False
)

annual_top_themes.to_csv(
    TABLE_DIR / "ICMB_130_FINAL_TOP6_ANNUAL_THEME_DATA.csv",
    index=False
)

period_data.reset_index().to_csv(
    TABLE_DIR / "ICMB_130_FINAL_LONGITUDINAL_THEME_COMPARISON.csv",
    index=False
)

sentiment_counts.to_csv(
    TABLE_DIR / "ICMB_130_FINAL_SENTIMENT_DISTRIBUTION.csv",
    index=False
)

classifiable_counts.to_csv(
    TABLE_DIR / "ICMB_130_FINAL_CLASSIFIABLE_SENTIMENT_DISTRIBUTION.csv",
    index=False
)

# ---------------------------------------------------------------------------
# 10. FINAL CELL SUMMARY
# ---------------------------------------------------------------------------

summary = {
    "yearly_gross_records": int(len(yearly_gross)),
    "top_10_films": int(len(top_10)),
    "curated_themes": int(annual_theme["curated_theme"].nunique()),
    "annual_theme_records": int(len(annual_theme)),
    "themes_visualized_annually": int(len(top_themes)),
    "sentiment_total_comments": int(len(sentiment)),
    "classifiable_sentiment_comments": int(classifiable_total),
    "classifiable_sentiment_share_pct": round(
        classifiable_total / len(sentiment) * 100, 2
    ),
    "figures_created": 6,
    "status": "Cell 11 visualization generation completed"
}

with open(
    SUMMARY_DIR / "ICMB_130_NOTEBOOK11_CELL11_SUMMARY.json",
    "w"
) as f:
    json.dump(summary, f, indent=4, default=str)

print("\n" + "=" * 90)
print("CELL 11 COMPLETED")
print("=" * 90)
print(json.dumps(summary, indent=4, default=str))

print("\nFigures saved in:", FIGURE_DIR)
print("Supporting tables saved in:", TABLE_DIR)


In [ ]:

# =============================================================================
# NOTEBOOK 11 — CELL 12
# FINAL MULTIMODAL DATA READINESS AND ELIGIBILITY AUDIT
# =============================================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

print("=" * 90)
print("CELL 12 — FINAL MULTIMODAL DATA READINESS AUDIT")
print("=" * 90)

# ---------------------------------------------------------------------------
# 1. PATHS
# ---------------------------------------------------------------------------

PROCESSED_DIR = PROJECT_ROOT / "01_data" / "processed"
TOPIC_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"
SENTIMENT_DIR = PROCESSED_DIR / "audience_sentiment"
SUBTITLE_DIR = PROCESSED_DIR / "subtitle_corpus_v2"

TABLE_DIR = PROJECT_ROOT / "05_outputs" / "tables"
SUMMARY_DIR = PROJECT_ROOT / "05_outputs" / "summary"

# ---------------------------------------------------------------------------
# 2. LOAD CORE DATASETS
# ---------------------------------------------------------------------------

master = pd.read_csv(
    PROCESSED_DIR / "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

features = pd.read_csv(
    PROCESSED_DIR / "ICMB_130_FEATURES.csv"
)

sentiment = pd.read_csv(
    SENTIMENT_DIR / "ICMB_130_YOUTUBE_COMMENTS_SENTIMENT_BASELINE.csv",
    low_memory=False
)

coverage_path = (
    PROJECT_ROOT / "06_outputs" / "audience_sentiment" /
    "ICMB_130_FINAL_YOUTUBE_COMMENT_COVERAGE_BATCH_01_16.csv"
)

subtitle_manifest_path = (
    SUBTITLE_DIR / "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

annual_theme_path = (
    TOPIC_DIR / "ICMB_130_FINAL_ANNUAL_THEME_PREVALENCE.csv"
)

longitudinal_path = (
    TOPIC_DIR / "ICMB_130_FINAL_LONGITUDINAL_THEME_SUMMARY.csv"
)

theme_register_path = (
    TOPIC_DIR / "ICMB_130_FINAL_12_THEME_REGISTER.csv"
)

coverage = pd.read_csv(coverage_path)
subtitle_manifest = pd.read_csv(subtitle_manifest_path)
annual_theme = pd.read_csv(annual_theme_path)
longitudinal = pd.read_csv(longitudinal_path)
theme_register = pd.read_csv(theme_register_path)

# Standardize movie IDs
for df in [master, features, sentiment, coverage, subtitle_manifest]:
    if "movie_record_id" in df.columns:
        df["movie_record_id"] = df["movie_record_id"].astype(str)

print("\n1. CORE DATASET DIMENSIONS")
print("-" * 70)

for name, df in {
    "Master": master,
    "Features": features,
    "Sentiment": sentiment,
    "YouTube coverage": coverage,
    "Subtitle manifest": subtitle_manifest,
    "Annual themes": annual_theme,
    "Longitudinal themes": longitudinal
}.items():
    print(f"{name:25s} {df.shape}")

# ---------------------------------------------------------------------------
# 3. BOX-OFFICE READINESS
# ---------------------------------------------------------------------------

print("\n2. BOX-OFFICE READINESS")
print("-" * 70)

gross = pd.to_numeric(master["india_gross_cr"], errors="coerce")

box_office_audit = {
    "total_films": int(len(master)),
    "unique_movie_ids": int(master["movie_record_id"].nunique()),
    "gross_values_available": int(gross.notna().sum()),
    "gross_values_missing": int(gross.isna().sum()),
    "gross_values_non_positive": int((gross <= 0).sum()),
    "india_gross_source_available": int(
        master["india_gross_source"].notna().sum()
    ),
    "india_gross_verified_yes": int(
        master["india_gross_verified"]
        .astype(str).str.lower().isin(["yes", "true", "1"]).sum()
    ),
    "tmdb_ids_available": int(master["tmdb_id"].notna().sum()),
    "year_min": int(master["year"].min()),
    "year_max": int(master["year"].max())
}

print(json.dumps(box_office_audit, indent=4))

# ---------------------------------------------------------------------------
# 4. SUBTITLE AND DIALOGUE READINESS
# ---------------------------------------------------------------------------

print("\n3. SUBTITLE AND DIALOGUE READINESS")
print("-" * 70)

subtitle_manifest["dialogue_blocks"] = pd.to_numeric(
    subtitle_manifest["dialogue_blocks"], errors="coerce"
)

subtitle_manifest["word_count"] = pd.to_numeric(
    subtitle_manifest["word_count"], errors="coerce"
)

subtitle_audit = {
    "manifest_rows": int(len(subtitle_manifest)),
    "unique_movies": int(
        subtitle_manifest["movie_record_id"].nunique()
    ),
    "validated_file_paths_available": int(
        subtitle_manifest["validated_subtitle_file"].notna().sum()
    ),
    "positive_dialogue_blocks": int(
        (subtitle_manifest["dialogue_blocks"] > 0).sum()
    ),
    "positive_word_counts": int(
        (subtitle_manifest["word_count"] > 0).sum()
    )
}

print("Subtitle audit:")
print(json.dumps(subtitle_audit, indent=4))

print("\nValidation status distribution:")
print(
    subtitle_manifest["validation_status"]
    .fillna("Missing")
    .value_counts()
)

print("\nQuality status distribution:")
print(
    subtitle_manifest["quality_status"]
    .fillna("Missing")
    .value_counts()
)

# ---------------------------------------------------------------------------
# 5. TOPIC MODELLING FILE DISCOVERY
# ---------------------------------------------------------------------------

print("\n4. TOPIC MODELLING FILM-LEVEL OUTPUT DISCOVERY")
print("-" * 70)

topic_catalogue = []

for path in TOPIC_DIR.rglob("*.csv"):
    try:
        header = pd.read_csv(path, nrows=0)
        cols = header.columns.tolist()

        has_movie_id = "movie_record_id" in cols

        topic_catalogue.append({
            "file": path.name,
            "path": str(path),
            "has_movie_record_id": has_movie_id,
            "has_curated_theme": "curated_theme" in cols,
            "has_theme": "theme" in cols or "theme_name" in cols,
            "has_year": "year" in cols,
            "has_dialogue_prevalence": (
                "dialogue_prevalence_pct" in cols
            ),
            "columns": ", ".join(cols)
        })

    except Exception as e:
        topic_catalogue.append({
            "file": path.name,
            "path": str(path),
            "error": str(e)
        })

topic_catalogue_df = pd.DataFrame(topic_catalogue)

if not topic_catalogue_df.empty:
    display(
        topic_catalogue_df[
            topic_catalogue_df["has_movie_record_id"] |
            topic_catalogue_df["has_curated_theme"] |
            topic_catalogue_df["has_theme"]
        ]
    )

    topic_catalogue_df.to_csv(
        TABLE_DIR / "ICMB_130_NOTEBOOK11_CELL12_TOPIC_FILE_DISCOVERY.csv",
        index=False
    )

# Annual theme coverage is not automatically film-level coverage.
print("\nCurated theme count:", len(theme_register))
print("Annual theme records:", len(annual_theme))
print("Longitudinal theme records:", len(longitudinal))

# ---------------------------------------------------------------------------
# 6. YOUTUBE AND SENTIMENT READINESS
# ---------------------------------------------------------------------------

print("\n5. YOUTUBE AND SENTIMENT READINESS")
print("-" * 70)

sentiment_counts = (
    sentiment.groupby("movie_record_id")
    .agg(
        sentiment_rows=("movie_record_id", "size"),
        positive=("final_sentiment", lambda x: (x == "positive").sum()),
        negative=("final_sentiment", lambda x: (x == "negative").sum()),
        neutral=("final_sentiment", lambda x: (x == "neutral").sum()),
        uncertain=("final_sentiment", lambda x: (x == "Uncertain").sum()),
        emoji_only=(
            "final_sentiment",
            lambda x: (x == "Emoji_Only_Not_Modelled").sum()
        )
    )
    .reset_index()
)

coverage["movie_record_id"] = coverage["movie_record_id"].astype(str)

youtube_audit = {
    "coverage_movies": int(coverage["movie_record_id"].nunique()),
    "coverage_comments": int(coverage["total_comments"].sum()),
    "sentiment_rows": int(len(sentiment)),
    "sentiment_movies": int(sentiment["movie_record_id"].nunique()),
    "sentiment_reliability_groups": int(
        sentiment["sentiment_reliability_group"].nunique()
    ),
    "classifiable_comments": int(
        sentiment["final_sentiment"]
        .isin(["positive", "negative", "neutral"]).sum()
    ),
    "uncertain_comments": int(
        (sentiment["final_sentiment"] == "Uncertain").sum()
    )
}

print(json.dumps(youtube_audit, indent=4))

# ---------------------------------------------------------------------------
# 7. BUILD FILM-LEVEL READINESS MATRIX
# ---------------------------------------------------------------------------

readiness = master[
    ["movie_record_id", "movie_title", "year", "rank",
     "language", "india_gross_cr"]
].copy()

# Box-office eligibility
readiness["box_office_available"] = (
    pd.to_numeric(readiness["india_gross_cr"], errors="coerce").notna()
)

# Subtitle manifest: identify records with positive dialogue blocks
subtitle_movie = (
    subtitle_manifest.groupby("movie_record_id")
    .agg(
        subtitle_manifest_records=("movie_record_id", "size"),
        positive_dialogue_records=(
            "dialogue_blocks", lambda x: (x > 0).sum()
        ),
        total_dialogue_blocks=("dialogue_blocks", "sum"),
        total_subtitle_words=("word_count", "sum")
    )
    .reset_index()
)

readiness = readiness.merge(
    subtitle_movie,
    on="movie_record_id",
    how="left"
)

readiness["subtitle_manifest_available"] = (
    readiness["subtitle_manifest_records"].fillna(0) > 0
)

readiness["dialogue_blocks_available"] = (
    readiness["positive_dialogue_records"].fillna(0) > 0
)

# Sentiment eligibility
readiness = readiness.merge(
    sentiment_counts,
    on="movie_record_id",
    how="left"
)

readiness["sentiment_rows"] = readiness["sentiment_rows"].fillna(0)

readiness["sentiment_available"] = (
    readiness["sentiment_rows"] > 0
)

readiness["classifiable_sentiment_count"] = (
    readiness[["positive", "negative", "neutral"]]
    .fillna(0).sum(axis=1)
)

readiness["classifiable_sentiment_available"] = (
    readiness["classifiable_sentiment_count"] > 0
)

# Overall availability flags
readiness["three_dimension_data_available"] = (
    readiness["box_office_available"] &
    readiness["dialogue_blocks_available"] &
    readiness["sentiment_available"]
)

readiness["three_dimension_classifiable_available"] = (
    readiness["box_office_available"] &
    readiness["dialogue_blocks_available"] &
    readiness["classifiable_sentiment_available"]
)

print("\n6. FILM-LEVEL READINESS SUMMARY")
print("-" * 70)

readiness_columns = [
    "box_office_available",
    "subtitle_manifest_available",
    "dialogue_blocks_available",
    "sentiment_available",
    "classifiable_sentiment_available",
    "three_dimension_data_available",
    "three_dimension_classifiable_available"
]

for col in readiness_columns:
    print(
        f"{col}: {int(readiness[col].sum())} / {len(readiness)}"
    )

# ---------------------------------------------------------------------------
# 8. CREATE READINESS SUMMARY TABLE
# ---------------------------------------------------------------------------

readiness_summary = pd.DataFrame([
    {
        "dimension": "Box-office",
        "eligible_films": int(readiness["box_office_available"].sum()),
        "total_films": len(readiness)
    },
    {
        "dimension": "Subtitle manifest",
        "eligible_films": int(
            readiness["subtitle_manifest_available"].sum()
        ),
        "total_films": len(readiness)
    },
    {
        "dimension": "Positive dialogue blocks",
        "eligible_films": int(
            readiness["dialogue_blocks_available"].sum()
        ),
        "total_films": len(readiness)
    },
    {
        "dimension": "YouTube sentiment records",
        "eligible_films": int(
            readiness["sentiment_available"].sum()
        ),
        "total_films": len(readiness)
    },
    {
        "dimension": "Classifiable sentiment",
        "eligible_films": int(
            readiness["classifiable_sentiment_available"].sum()
        ),
        "total_films": len(readiness)
    },
    {
        "dimension": "All three dimensions available",
        "eligible_films": int(
            readiness["three_dimension_data_available"].sum()
        ),
        "total_films": len(readiness)
    },
    {
        "dimension": "All three with classifiable sentiment",
        "eligible_films": int(
            readiness["three_dimension_classifiable_available"].sum()
        ),
        "total_films": len(readiness)
    }
])

display(readiness_summary)

# ---------------------------------------------------------------------------
# 9. SAVE OUTPUTS
# ---------------------------------------------------------------------------

readiness.to_csv(
    TABLE_DIR / "ICMB_130_FINAL_MULTIMODAL_FILM_READINESS.csv",
    index=False
)

readiness_summary.to_csv(
    TABLE_DIR / "ICMB_130_FINAL_MULTIMODAL_READINESS_SUMMARY.csv",
    index=False
)

summary = {
    "total_films": int(len(readiness)),
    "box_office_eligible": int(
        readiness["box_office_available"].sum()
    ),
    "subtitle_manifest_eligible": int(
        readiness["subtitle_manifest_available"].sum()
    ),
    "dialogue_eligible": int(
        readiness["dialogue_blocks_available"].sum()
    ),
    "sentiment_eligible": int(
        readiness["sentiment_available"].sum()
    ),
    "classifiable_sentiment_eligible": int(
        readiness["classifiable_sentiment_available"].sum()
    ),
    "all_three_dimensions_eligible": int(
        readiness["three_dimension_data_available"].sum()
    ),
    "all_three_classifiable_eligible": int(
        readiness["three_dimension_classifiable_available"].sum()
    ),
    "topic_film_level_files_discovered": int(
        topic_catalogue_df["has_movie_record_id"].sum()
    ) if not topic_catalogue_df.empty else 0,
    "status": "Cell 12 readiness audit completed"
}

with open(
    SUMMARY_DIR / "ICMB_130_NOTEBOOK11_CELL12_READINESS_SUMMARY.json",
    "w"
) as f:
    json.dump(summary, f, indent=4, default=str)

print("\n" + "=" * 90)
print("CELL 12 COMPLETED")
print("=" * 90)
print(json.dumps(summary, indent=4, default=str))

print("\nSaved readiness outputs in:", TABLE_DIR)


In [ ]:

# =============================================================================
# NOTEBOOK 11 — CELL 13
# FILM-LEVEL THEME FEATURE AND SUBTITLE PATH VERIFICATION
# =============================================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

print("=" * 90)
print("CELL 13 — FILM-LEVEL THEME FEATURE INSPECTION")
print("=" * 90)

TOPIC_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"
PROCESSED_DIR = PROJECT_ROOT / "01_data" / "processed"
TABLE_DIR = PROJECT_ROOT / "05_outputs" / "tables"
SUMMARY_DIR = PROJECT_ROOT / "05_outputs" / "summary"

# ---------------------------------------------------------------------------
# 1. LOAD FILM-LEVEL THEME CONTRIBUTIONS
# ---------------------------------------------------------------------------

theme_path = (
    TOPIC_DIR / "ICMB_130_THEME_YEAR_FILM_CONTRIBUTIONS.csv"
)

if not theme_path.exists():
    raise FileNotFoundError(theme_path)

theme_contributions = pd.read_csv(theme_path)

print("\n1. THEME CONTRIBUTION DATASET")
print("-" * 70)
print("Shape:", theme_contributions.shape)
print("Columns:")
print(theme_contributions.columns.tolist())

print("\nFirst five records:")
display(theme_contributions.head())

# ---------------------------------------------------------------------------
# 2. IDENTIFY FILM AND THEME COLUMNS
# ---------------------------------------------------------------------------

movie_col = (
    "movie_record_id"
    if "movie_record_id" in theme_contributions.columns
    else None
)

theme_col = next(
    (
        c for c in [
            "curated_theme",
            "theme",
            "theme_name"
        ]
        if c in theme_contributions.columns
    ),
    None
)

print("\n2. IDENTIFIED COLUMNS")
print("-" * 70)
print("Movie identifier:", movie_col)
print("Theme identifier:", theme_col)

# ---------------------------------------------------------------------------
# 3. INSPECT NUMERIC FEATURE COLUMNS
# ---------------------------------------------------------------------------

numeric_columns = theme_contributions.select_dtypes(
    include=np.number
).columns.tolist()

print("\nNumeric columns:")
print(numeric_columns)

# ---------------------------------------------------------------------------
# 4. FILM-LEVEL COVERAGE
# ---------------------------------------------------------------------------

master = pd.read_csv(
    PROCESSED_DIR / "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

master["movie_record_id"] = master["movie_record_id"].astype(str)

if movie_col:
    theme_contributions[movie_col] = (
        theme_contributions[movie_col].astype(str)
    )

    print("\n3. FILM-LEVEL COVERAGE")
    print("-" * 70)

    print(
        "Unique films in contribution data:",
        theme_contributions[movie_col].nunique()
    )

    print(
        "Duplicate rows:",
        theme_contributions.duplicated().sum()
    )

    print(
        "Missing movie IDs:",
        theme_contributions[movie_col].isna().sum()
    )

    print(
        "Films in master:",
        master["movie_record_id"].nunique()
    )

    print(
        "Contribution films matching master:",
        theme_contributions[movie_col].isin(
            master["movie_record_id"]
        ).sum()
    )

# ---------------------------------------------------------------------------
# 5. THEME DISTRIBUTION
# ---------------------------------------------------------------------------

if theme_col:
    print("\n4. THEME DISTRIBUTION")
    print("-" * 70)

    print(
        theme_contributions[theme_col]
        .value_counts(dropna=False)
    )

else:
    print(
        "\nNo explicit curated-theme column found. "
        "The schema must be interpreted before feature construction."
    )

# ---------------------------------------------------------------------------
# 6. SUBTITLE PATH EXISTENCE CHECK
# ---------------------------------------------------------------------------

print("\n5. SUBTITLE FILE PATH VERIFICATION")
print("-" * 70)

manifest_path = (
    PROCESSED_DIR / "subtitle_corpus_v2" /
    "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
)

manifest = pd.read_csv(manifest_path)

path_col = "validated_subtitle_file"

if path_col in manifest.columns:

    def check_subtitle_path(value):
        if pd.isna(value) or not str(value).strip():
            return False

        p = Path(str(value))

        if p.is_absolute():
            return p.exists()

        return (PROJECT_ROOT / p).exists()

    manifest["validated_path_exists"] = (
        manifest[path_col].apply(check_subtitle_path)
    )

    print(
        "Non-empty path entries:",
        manifest[path_col].notna().sum()
    )

    print(
        "Existing validated paths:",
        manifest["validated_path_exists"].sum()
    )

    print(
        "Missing or unresolved paths:",
        (~manifest["validated_path_exists"]).sum()
    )

    print("\nPath existence by validation status:")
    display(
        manifest.groupby("validation_status")[
            "validated_path_exists"
        ].agg(["count", "sum"])
    )

# ---------------------------------------------------------------------------
# 7. SAVE INSPECTION SUMMARY
# ---------------------------------------------------------------------------

summary = {
    "theme_contribution_rows": int(len(theme_contributions)),
    "theme_contribution_columns": int(
        len(theme_contributions.columns)
    ),
    "movie_id_column": movie_col,
    "theme_column": theme_col,
    "numeric_columns": numeric_columns,
    "unique_theme_films": (
        int(theme_contributions[movie_col].nunique())
        if movie_col else None
    ),
    "unique_themes": (
        int(theme_contributions[theme_col].nunique())
        if theme_col else None
    ),
    "subtitle_paths_existing": (
        int(manifest["validated_path_exists"].sum())
        if "validated_path_exists" in manifest.columns
        else None
    ),
    "status": "Cell 13 inspection completed"
}

with open(
    SUMMARY_DIR / "ICMB_130_NOTEBOOK11_CELL13_SUMMARY.json",
    "w"
) as f:
    json.dump(summary, f, indent=4, default=str)

print("\n" + "=" * 90)
print("CELL 13 COMPLETED")
print("=" * 90)
print(json.dumps(summary, indent=4, default=str))


In [ ]:

# ================================================================
# CELL 14 — BUILD FILM-LEVEL THEME MATRIX & INSPECT INTEGRATION
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
import json

print("=" * 90)
print("CELL 14 — FILM-LEVEL THEME MATRIX AND INTEGRATION INSPECTION")
print("=" * 90)

# ------------------------------------------------
# 1. LOAD INPUTS
# ------------------------------------------------

theme_path = (
    PROJECT_ROOT / "06_outputs" / "topic_modelling" /
    "ICMB_130_THEME_YEAR_FILM_CONTRIBUTIONS.csv"
)

master_path = (
    PROJECT_ROOT / "01_data" / "processed" /
    "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

sentiment_path = (
    PROJECT_ROOT / "05_outputs" / "tables" /
    "ICMB_130_FINAL_FILM_LEVEL_SENTIMENT_SUMMARY.csv"
)

theme_df = pd.read_csv(theme_path)
master_df = pd.read_csv(master_path)
sentiment_df = pd.read_csv(sentiment_path)

print("\nINPUT SHAPES")
print("-" * 70)
print("Theme contributions:", theme_df.shape)
print("Master:", master_df.shape)
print("Film-level sentiment:", sentiment_df.shape)

# ------------------------------------------------
# 2. VALIDATE THE FILM-THEME DATA
# ------------------------------------------------

required_theme_cols = [
    "movie_record_id",
    "curated_theme",
    "theme_chunks"
]

missing = [c for c in required_theme_cols if c not in theme_df.columns]

if missing:
    raise ValueError(f"Missing required theme columns: {missing}")

if theme_df.duplicated(["movie_record_id", "curated_theme"]).any():
    raise ValueError(
        "Duplicate film-theme combinations detected. "
        "Resolve duplicates before pivoting."
    )

theme_df["theme_chunks"] = pd.to_numeric(
    theme_df["theme_chunks"], errors="coerce"
).fillna(0)

# ------------------------------------------------
# 3. CREATE FILM-LEVEL THEME COUNT MATRIX
# ------------------------------------------------

theme_counts = theme_df.pivot(
    index="movie_record_id",
    columns="curated_theme",
    values="theme_chunks"
).fillna(0)

theme_counts.columns = [
    "theme_chunks_" + str(c).lower()
    .replace(" ", "_")
    .replace(",", "")
    .replace("-", "_")
    for c in theme_counts.columns
]

theme_counts = theme_counts.reset_index()

# ------------------------------------------------
# 4. CALCULATE WITHIN-FILM THEME SHARES
# ------------------------------------------------

theme_feature_cols = [
    c for c in theme_counts.columns
    if c.startswith("theme_chunks_")
]

theme_counts["total_curated_theme_chunks"] = (
    theme_counts[theme_feature_cols].sum(axis=1)
)

for col in theme_feature_cols:
    theme_name = col.replace("theme_chunks_", "")
    theme_counts["theme_share_" + theme_name] = np.where(
        theme_counts["total_curated_theme_chunks"] > 0,
        theme_counts[col] /
        theme_counts["total_curated_theme_chunks"] * 100,
        np.nan
    )

theme_share_cols = [
    c for c in theme_counts.columns
    if c.startswith("theme_share_")
]

print("\nTHEME MATRIX")
print("-" * 70)
print("Film rows:", len(theme_counts))
print("Theme count features:", len(theme_feature_cols))
print("Theme share features:", len(theme_share_cols))
print("Total theme chunks — median:",
      theme_counts["total_curated_theme_chunks"].median())
print("\nFirst five rows:")
display(theme_counts.head())

# ------------------------------------------------
# 5. INSPECT SENTIMENT SCHEMA
# ------------------------------------------------

print("\nSENTIMENT SUMMARY COLUMNS")
print("-" * 70)
print(sentiment_df.columns.tolist())

print("\nSENTIMENT SUMMARY SAMPLE")
display(sentiment_df.head(5))

# ------------------------------------------------
# 6. CHECK JOIN KEYS AND MASTER COLUMNS
# ------------------------------------------------

for label, df in [
    ("Master", master_df),
    ("Sentiment", sentiment_df),
]:
    print(f"\n{label} — JOIN CHECK")
    print("-" * 70)

    if "movie_record_id" not in df.columns:
        print("WARNING: movie_record_id missing")
        continue

    print("Rows:", len(df))
    print("Unique film IDs:", df["movie_record_id"].nunique())
    print("Duplicate film IDs:",
          df["movie_record_id"].duplicated().sum())

print("\nMASTER COLUMNS")
print("-" * 70)
print(master_df.columns.tolist())

# ------------------------------------------------
# 7. CHECK FILM COVERAGE ACROSS ALL THREE INPUTS
# ------------------------------------------------

theme_ids = set(theme_counts["movie_record_id"].dropna())
master_ids = set(master_df["movie_record_id"].dropna())
sentiment_ids = set(sentiment_df["movie_record_id"].dropna())

common_ids = theme_ids & master_ids & sentiment_ids

print("\nTHREE-WAY FILM COVERAGE")
print("-" * 70)
print("Theme matrix films:", len(theme_ids))
print("Master films:", len(master_ids))
print("Sentiment films:", len(sentiment_ids))
print("Common films:", len(common_ids))
print("Theme films missing from sentiment:",
      len(theme_ids - sentiment_ids))
print("Theme films missing from master:",
      len(theme_ids - master_ids))

# ------------------------------------------------
# 8. SAVE FILM-LEVEL THEME FEATURES
# ------------------------------------------------

out_path = (
    PROJECT_ROOT / "05_outputs" / "tables" /
    "ICMB_130_FILM_LEVEL_THEME_FEATURE_MATRIX.csv"
)

theme_counts.to_csv(out_path, index=False)

summary = {
    "theme_contribution_rows": int(len(theme_df)),
    "theme_matrix_films": int(len(theme_counts)),
    "theme_count_features": int(len(theme_feature_cols)),
    "theme_share_features": int(len(theme_share_cols)),
    "common_films_across_three_inputs": int(len(common_ids)),
    "theme_matrix_output": str(out_path),
    "status": "Film-level theme matrix created"
}

summary_path = (
    PROJECT_ROOT / "05_outputs" / "summary" /
    "ICMB_130_NOTEBOOK11_CELL14_SUMMARY.json"
)

summary_path.parent.mkdir(parents=True, exist_ok=True)

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("\nSAVED")
print("-" * 70)
print(out_path)
print(summary_path)

print("\n" + "=" * 90)
print("CELL 14 COMPLETED")
print("=" * 90)
print(json.dumps(summary, indent=4))


In [ ]:

# ================================================================
# CELL 15 — INTEGRATED MULTIMODAL STATISTICAL ANALYSIS
# ================================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path
from scipy.stats import spearmanr

print("=" * 95)
print("CELL 15 — INTEGRATED MULTIMODAL STATISTICAL ANALYSIS")
print("=" * 95)

# ------------------------------------------------
# 1. LOAD DATASETS
# ------------------------------------------------

master_path = (
    PROJECT_ROOT / "01_data" / "processed" /
    "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

theme_path = (
    PROJECT_ROOT / "05_outputs" / "tables" /
    "ICMB_130_FILM_LEVEL_THEME_FEATURE_MATRIX.csv"
)

sentiment_path = (
    PROJECT_ROOT / "05_outputs" / "tables" /
    "ICMB_130_FINAL_FILM_LEVEL_SENTIMENT_SUMMARY.csv"
)

master = pd.read_csv(master_path)
themes = pd.read_csv(theme_path)
sentiment = pd.read_csv(sentiment_path)

print("\nINPUT DATA")
print("-" * 75)
print("Master:", master.shape)
print("Theme matrix:", themes.shape)
print("Sentiment:", sentiment.shape)

# ------------------------------------------------
# 2. VALIDATE AND MERGE
# ------------------------------------------------

key = "movie_record_id"

for name, df in [
    ("Master", master),
    ("Themes", themes),
    ("Sentiment", sentiment)
]:
    if key not in df.columns:
        raise ValueError(f"{name} is missing {key}")

    if df[key].duplicated().any():
        raise ValueError(f"{name} contains duplicate film IDs")

required_master = ["year", "india_gross_cr", "movie_title"]
required_sentiment = [
    "sentiment_comment_count",
    "positive_comments",
    "negative_comments",
    "neutral_comments",
    "uncertain_comments",
    "emoji_only_comments"
]

for col in required_master:
    if col not in master.columns:
        raise ValueError(f"Master is missing required column: {col}")

for col in required_sentiment:
    if col not in sentiment.columns:
        raise ValueError(f"Sentiment summary is missing: {col}")

df = (
    master.merge(themes, on=key, how="inner", validate="one_to_one")
          .merge(sentiment, on=key, how="inner", validate="one_to_one")
)

if len(df) != 130:
    raise ValueError(
        f"Expected 130 integrated films, but obtained {len(df)}"
    )

# Convert analytical fields to numeric
numeric_cols = [
    "year", "india_gross_cr",
    *required_sentiment
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nINTEGRATED DATASET")
print("-" * 75)
print("Shape:", df.shape)
print("Unique films:", df[key].nunique())
print("Missing gross:", df["india_gross_cr"].isna().sum())

# ------------------------------------------------
# 3. CREATE SENTIMENT FEATURES
# ------------------------------------------------

df["classifiable_comments"] = (
    df["positive_comments"].fillna(0) +
    df["negative_comments"].fillna(0) +
    df["neutral_comments"].fillna(0)
)

df["sentiment_total_components"] = (
    df["classifiable_comments"] +
    df["uncertain_comments"].fillna(0) +
    df["emoji_only_comments"].fillna(0)
)

# Shares among classifiable comments
for label in ["positive", "negative", "neutral"]:
    count_col = f"{label}_comments"

    df[f"{label}_share_classifiable_pct"] = np.where(
        df["classifiable_comments"] > 0,
        df[count_col] / df["classifiable_comments"] * 100,
        np.nan
    )

# Classifiable coverage and uncertainty
df["classifiable_share_total_pct"] = np.where(
    df["sentiment_comment_count"] > 0,
    df["classifiable_comments"] /
    df["sentiment_comment_count"] * 100,
    np.nan
)

df["uncertain_share_total_pct"] = np.where(
    df["sentiment_comment_count"] > 0,
    df["uncertain_comments"] /
    df["sentiment_comment_count"] * 100,
    np.nan
)

# Log transformation for descriptive modelling readiness
df["log_india_gross"] = np.where(
    df["india_gross_cr"] > 0,
    np.log1p(df["india_gross_cr"]),
    np.nan
)

# ------------------------------------------------
# 4. VALIDATE SENTIMENT COUNTS
# ------------------------------------------------

df["sentiment_count_difference"] = (
    df["sentiment_comment_count"] -
    df["sentiment_total_components"]
)

print("\nSENTIMENT FEATURE VALIDATION")
print("-" * 75)
print("Total comments:", int(df["sentiment_comment_count"].sum()))
print("Classifiable comments:", int(df["classifiable_comments"].sum()))
print("Positive comments:", int(df["positive_comments"].sum()))
print("Negative comments:", int(df["negative_comments"].sum()))
print("Neutral comments:", int(df["neutral_comments"].sum()))
print("Uncertain comments:", int(df["uncertain_comments"].sum()))
print("Emoji-only comments:", int(df["emoji_only_comments"].sum()))
print(
    "Films with sentiment component count mismatch:",
    int((df["sentiment_count_difference"] != 0).sum())
)

print("\nOVERALL CLASSIFIABLE SENTIMENT")
print("-" * 75)

overall_classifiable = df["classifiable_comments"].sum()

for label in ["positive", "negative", "neutral"]:
    total = df[f"{label}_comments"].sum()
    share = (
        total / overall_classifiable * 100
        if overall_classifiable > 0 else np.nan
    )
    print(f"{label.title()}: {int(total):,} ({share:.2f}%)")

print(
    "Classifiable share of all comments:",
    round(
        df["classifiable_comments"].sum() /
        df["sentiment_comment_count"].sum() * 100, 2
    ),
    "%"
)

# ------------------------------------------------
# 5. SPEARMAN CORRELATION HELPER
# ------------------------------------------------

def spearman_result(data, x, y, analysis_group):
    temp = data[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()

    if len(temp) < 5:
        return {
            "analysis_group": analysis_group,
            "feature": x,
            "outcome": y,
            "n": len(temp),
            "spearman_rho": np.nan,
            "p_value": np.nan
        }

    rho, p_value = spearmanr(temp[x], temp[y])

    return {
        "analysis_group": analysis_group,
        "feature": x,
        "outcome": y,
        "n": int(len(temp)),
        "spearman_rho": float(rho),
        "p_value": float(p_value)
    }


# Benjamini-Hochberg FDR adjustment
def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p), np.nan)

    valid_idx = np.where(np.isfinite(p))[0]

    if len(valid_idx) == 0:
        return adjusted

    valid_p = p[valid_idx]
    order = np.argsort(valid_p)
    ranked_p = valid_p[order]

    m = len(ranked_p)
    adjusted_sorted = ranked_p * m / np.arange(1, m + 1)

    # Enforce monotonicity from largest rank backwards
    adjusted_sorted = np.minimum.accumulate(
        adjusted_sorted[::-1]
    )[::-1]

    adjusted_sorted = np.clip(adjusted_sorted, 0, 1)

    restored = np.empty(m)
    restored[order] = adjusted_sorted
    adjusted[valid_idx] = restored

    return adjusted


# ------------------------------------------------
# 6. IDENTIFY THEME SHARE FEATURES
# ------------------------------------------------

theme_share_cols = [
    c for c in df.columns
    if c.startswith("theme_share_")
]

if len(theme_share_cols) != 12:
    raise ValueError(
        f"Expected 12 theme share features, found {len(theme_share_cols)}"
    )

# Check shares sum to 100 for films with theme contributions
df["theme_share_sum_pct"] = df[theme_share_cols].sum(
    axis=1, min_count=1
)

print("\nTHEME SHARE VALIDATION")
print("-" * 75)
print("Theme share feature count:", len(theme_share_cols))
print("Median theme share sum:",
      round(df["theme_share_sum_pct"].median(), 2))
print("Films with zero total theme chunks:",
      int((df["total_curated_theme_chunks"] == 0).sum()))

# ------------------------------------------------
# 7. ANALYSIS A — THEMES VS INDIA GROSS
# ------------------------------------------------

results = []

for feature in theme_share_cols:
    results.append(
        spearman_result(
            df, feature, "india_gross_cr",
            "Theme share vs India Gross"
        )
    )

theme_gross_results = pd.DataFrame(results)

theme_gross_results["p_value_fdr"] = benjamini_hochberg(
    theme_gross_results["p_value"]
)

theme_gross_results["significant_fdr_05"] = (
    theme_gross_results["p_value_fdr"] < 0.05
)

theme_gross_results = theme_gross_results.sort_values(
    "spearman_rho",
    key=lambda s: s.abs(),
    ascending=False
)

# ------------------------------------------------
# 8. ANALYSIS B — SENTIMENT VS INDIA GROSS
# ------------------------------------------------

sentiment_features = [
    "positive_share_classifiable_pct",
    "negative_share_classifiable_pct",
    "neutral_share_classifiable_pct",
    "classifiable_share_total_pct",
    "uncertain_share_total_pct"
]

sentiment_gross_results = pd.DataFrame([
    spearman_result(
        df, feature, "india_gross_cr",
        "Sentiment feature vs India Gross"
    )
    for feature in sentiment_features
])

sentiment_gross_results["p_value_fdr"] = benjamini_hochberg(
    sentiment_gross_results["p_value"]
)

sentiment_gross_results["significant_fdr_05"] = (
    sentiment_gross_results["p_value_fdr"] < 0.05
)

# ------------------------------------------------
# 9. ANALYSIS C — THEMES VS SENTIMENT
# ------------------------------------------------

theme_sentiment_results = []

for theme_feature in theme_share_cols:
    for sentiment_feature in [
        "positive_share_classifiable_pct",
        "negative_share_classifiable_pct",
        "neutral_share_classifiable_pct"
    ]:
        theme_sentiment_results.append(
            spearman_result(
                df,
                theme_feature,
                sentiment_feature,
                "Theme share vs sentiment"
            )
        )

theme_sentiment_results = pd.DataFrame(
    theme_sentiment_results
)

theme_sentiment_results["p_value_fdr"] = benjamini_hochberg(
    theme_sentiment_results["p_value"]
)

theme_sentiment_results["significant_fdr_05"] = (
    theme_sentiment_results["p_value_fdr"] < 0.05
)

theme_sentiment_results = theme_sentiment_results.sort_values(
    "p_value_fdr",
    na_position="last"
)

# ------------------------------------------------
# 10. PRINT RESULTS
# ------------------------------------------------

print("\n" + "=" * 95)
print("ANALYSIS A — THEME SHARE VS INDIA GROSS")
print("=" * 95)
display(theme_gross_results.round(4))

print("\n" + "=" * 95)
print("ANALYSIS B — SENTIMENT VS INDIA GROSS")
print("=" * 95)
display(sentiment_gross_results.round(4))

print("\n" + "=" * 95)
print("ANALYSIS C — THEME SHARE VS SENTIMENT")
print("=" * 95)
display(theme_sentiment_results.round(4))

# ------------------------------------------------
# 11. PERIOD-WISE DESCRIPTIVE SUMMARY
# ------------------------------------------------

def assign_period(year):
    if 2000 <= year <= 2008:
        return "Early (2000–2008)"
    elif 2009 <= year <= 2017:
        return "Middle (2009–2017)"
    elif 2018 <= year <= 2025:
        return "Recent (2018–2025)"
    return np.nan

df["period"] = df["year"].apply(assign_period)

period_summary = (
    df.groupby("period", dropna=False)
      .agg(
          films=("movie_record_id", "nunique"),
          mean_india_gross_cr=("india_gross_cr", "mean"),
          median_india_gross_cr=("india_gross_cr", "median"),
          mean_positive_share=("positive_share_classifiable_pct", "mean"),
          mean_negative_share=("negative_share_classifiable_pct", "mean"),
          mean_neutral_share=("neutral_share_classifiable_pct", "mean"),
          mean_classifiable_coverage=("classifiable_share_total_pct", "mean"),
          mean_theme_chunks=("total_curated_theme_chunks", "mean")
      )
      .reset_index()
)

print("\n" + "=" * 95)
print("PERIOD-WISE DESCRIPTIVE SUMMARY")
print("=" * 95)
display(period_summary.round(3))

# ------------------------------------------------
# 12. SAVE ANALYTICAL OUTPUTS
# ------------------------------------------------

TABLE_DIR = PROJECT_ROOT / "05_outputs" / "tables"
SUMMARY_DIR = PROJECT_ROOT / "05_outputs" / "summary"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(
    TABLE_DIR / "ICMB_130_FINAL_INTEGRATED_FILM_ANALYTICS.csv",
    index=False
)

theme_gross_results.to_csv(
    TABLE_DIR / "ICMB_130_THEME_VS_BOXOFFICE_SPEARMAN.csv",
    index=False
)

sentiment_gross_results.to_csv(
    TABLE_DIR / "ICMB_130_SENTIMENT_VS_BOXOFFICE_SPEARMAN.csv",
    index=False
)

theme_sentiment_results.to_csv(
    TABLE_DIR / "ICMB_130_THEME_VS_SENTIMENT_SPEARMAN.csv",
    index=False
)

period_summary.to_csv(
    TABLE_DIR / "ICMB_130_PERIOD_WISE_INTEGRATED_SUMMARY.csv",
    index=False
)

# ------------------------------------------------
# 13. CREATE SUMMARY
# ------------------------------------------------

summary = {
    "integrated_films": int(df["movie_record_id"].nunique()),
    "integrated_dataset_shape": list(df.shape),
    "theme_share_features": int(len(theme_share_cols)),
    "total_youtube_comments": int(df["sentiment_comment_count"].sum()),
    "classifiable_comments": int(df["classifiable_comments"].sum()),
    "classifiable_comment_percentage": round(
        df["classifiable_comments"].sum() /
        df["sentiment_comment_count"].sum() * 100, 2
    ),
    "sentiment_count_mismatch_films": int(
        (df["sentiment_count_difference"] != 0).sum()
    ),
    "theme_gross_tests": int(len(theme_gross_results)),
    "theme_gross_fdr_significant": int(
        theme_gross_results["significant_fdr_05"].sum()
    ),
    "sentiment_gross_tests": int(len(sentiment_gross_results)),
    "sentiment_gross_fdr_significant": int(
        sentiment_gross_results["significant_fdr_05"].sum()
    ),
    "theme_sentiment_tests": int(len(theme_sentiment_results)),
    "theme_sentiment_fdr_significant": int(
        theme_sentiment_results["significant_fdr_05"].sum()
    ),
    "interpretation": (
        "Exploratory associations only; no causal inference. "
        "India Gross provenance is undocumented in the current master. "
        "YouTube sentiment includes substantial uncertain classifications."
    )
}

summary_path = (
    SUMMARY_DIR / "ICMB_130_NOTEBOOK11_CELL15_SUMMARY.json"
)

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("\n" + "=" * 95)
print("CELL 15 — SAVED OUTPUTS")
print("=" * 95)

print("1. ICMB_130_FINAL_INTEGRATED_FILM_ANALYTICS.csv")
print("2. ICMB_130_THEME_VS_BOXOFFICE_SPEARMAN.csv")
print("3. ICMB_130_SENTIMENT_VS_BOXOFFICE_SPEARMAN.csv")
print("4. ICMB_130_THEME_VS_SENTIMENT_SPEARMAN.csv")
print("5. ICMB_130_PERIOD_WISE_INTEGRATED_SUMMARY.csv")
print("6. ICMB_130_NOTEBOOK11_CELL15_SUMMARY.json")

print("\nSUMMARY")
print(json.dumps(summary, indent=4))

print("\n" + "=" * 95)
print("CELL 15 COMPLETED")
print("=" * 95)


In [ ]:

# ================================================================
# CELL 16 — YEAR-ADJUSTED ASSOCIATIONS & SENTIMENT COUNT CHECK
# ================================================================

import pandas as pd
import numpy as np
import json
from scipy.stats import rankdata, pearsonr
from pathlib import Path

print("=" * 95)
print("CELL 16 — YEAR-ADJUSTED ROBUSTNESS CHECK")
print("=" * 95)

# ------------------------------------------------
# 1. LOAD INTEGRATED DATA
# ------------------------------------------------

input_path = (
    PROJECT_ROOT / "05_outputs" / "tables" /
    "ICMB_130_FINAL_INTEGRATED_FILM_ANALYTICS.csv"
)

df = pd.read_csv(input_path)

print("\nDATA")
print("-" * 70)
print("Shape:", df.shape)
print("Unique films:", df["movie_record_id"].nunique())

# ------------------------------------------------
# 2. IDENTIFY SENTIMENT COUNT MISMATCH
# ------------------------------------------------

component_cols = [
    "positive_comments",
    "negative_comments",
    "neutral_comments",
    "uncertain_comments",
    "emoji_only_comments"
]

df["recalculated_component_total"] = (
    df[component_cols].fillna(0).sum(axis=1)
)

df["component_difference"] = (
    df["sentiment_comment_count"] -
    df["recalculated_component_total"]
)

mismatch = df.loc[
    df["component_difference"] != 0,
    [
        "movie_record_id",
        "movie_title",
        "year",
        "sentiment_comment_count",
        *component_cols,
        "recalculated_component_total",
        "component_difference"
    ]
].copy()

print("\nSENTIMENT COUNT RECONCILIATION")
print("-" * 70)
print("Mismatch films:", len(mismatch))
display(mismatch)

# ------------------------------------------------
# 3. PARTIAL SPEARMAN FUNCTION
#    Controls for year using year fixed effects.
#    Rank-transform both variables, then remove
#    year-specific mean ranks before correlation.
# ------------------------------------------------

def partial_spearman_year(data, x_col, y_col):
    temp = data[
        ["year", x_col, y_col]
    ].replace([np.inf, -np.inf], np.nan).dropna().copy()

    n = len(temp)

    if n < 10 or temp["year"].nunique() < 2:
        return {
            "feature": x_col,
            "outcome": y_col,
            "n": n,
            "partial_spearman_rho": np.nan,
            "p_value": np.nan
        }

    # Rank-transform variables
    x_rank = rankdata(temp[x_col].to_numpy())
    y_rank = rankdata(temp[y_col].to_numpy())

    # Year fixed-effect design matrix
    year_dummies = pd.get_dummies(
        temp["year"].astype(str),
        drop_first=True,
        dtype=float
    ).to_numpy()

    design = np.column_stack([
        np.ones(n),
        year_dummies
    ])

    # Residualize ranked variables by year
    x_resid = x_rank - design @ np.linalg.lstsq(
        design, x_rank, rcond=None
    )[0]

    y_resid = y_rank - design @ np.linalg.lstsq(
        design, y_rank, rcond=None
    )[0]

    if np.std(x_resid) == 0 or np.std(y_resid) == 0:
        rho, p_value = np.nan, np.nan
    else:
        rho, p_value = pearsonr(x_resid, y_resid)

    return {
        "feature": x_col,
        "outcome": y_col,
        "n": n,
        "partial_spearman_rho": float(rho),
        "p_value": float(p_value)
    }


# ------------------------------------------------
# 4. BENJAMINI-HOCHBERG CORRECTION
# ------------------------------------------------

def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p), np.nan)

    valid = np.where(np.isfinite(p))[0]

    if len(valid) == 0:
        return adjusted

    pv = p[valid]
    order = np.argsort(pv)
    sorted_p = pv[order]

    m = len(sorted_p)

    adjusted_sorted = sorted_p * m / np.arange(1, m + 1)
    adjusted_sorted = np.minimum.accumulate(
        adjusted_sorted[::-1]
    )[::-1]
    adjusted_sorted = np.clip(adjusted_sorted, 0, 1)

    restored = np.empty(m)
    restored[order] = adjusted_sorted
    adjusted[valid] = restored

    return adjusted


# ------------------------------------------------
# 5. THEME VS GROSS, CONTROLLING FOR YEAR
# ------------------------------------------------

theme_cols = [
    c for c in df.columns
    if c.startswith("theme_share_")
]

theme_year_results = pd.DataFrame([
    partial_spearman_year(
        df, feature, "india_gross_cr"
    )
    for feature in theme_cols
])

theme_year_results["p_value_fdr"] = benjamini_hochberg(
    theme_year_results["p_value"]
)

theme_year_results["significant_fdr_05"] = (
    theme_year_results["p_value_fdr"] < 0.05
)

theme_year_results = theme_year_results.sort_values(
    "p_value_fdr",
    na_position="last"
)

print("\n" + "=" * 95)
print("THEME SHARE VS INDIA GROSS — CONTROLLING FOR YEAR")
print("=" * 95)
display(theme_year_results.round(4))

# ------------------------------------------------
# 6. SENTIMENT VS GROSS, CONTROLLING FOR YEAR
# ------------------------------------------------

sentiment_cols = [
    "positive_share_classifiable_pct",
    "negative_share_classifiable_pct",
    "neutral_share_classifiable_pct",
    "classifiable_share_total_pct",
    "uncertain_share_total_pct"
]

sentiment_year_results = pd.DataFrame([
    partial_spearman_year(
        df, feature, "india_gross_cr"
    )
    for feature in sentiment_cols
])

sentiment_year_results["p_value_fdr"] = benjamini_hochberg(
    sentiment_year_results["p_value"]
)

sentiment_year_results["significant_fdr_05"] = (
    sentiment_year_results["p_value_fdr"] < 0.05
)

sentiment_year_results = sentiment_year_results.sort_values(
    "p_value_fdr",
    na_position="last"
)

print("\n" + "=" * 95)
print("SENTIMENT VS INDIA GROSS — CONTROLLING FOR YEAR")
print("=" * 95)
display(sentiment_year_results.round(4))

# ------------------------------------------------
# 7. THEMES VS SENTIMENT, CONTROLLING FOR YEAR
# ------------------------------------------------

theme_sentiment_rows = []

for theme_feature in theme_cols:
    for sentiment_feature in [
        "positive_share_classifiable_pct",
        "negative_share_classifiable_pct",
        "neutral_share_classifiable_pct"
    ]:
        result = partial_spearman_year(
            df, theme_feature, sentiment_feature
        )
        theme_sentiment_rows.append(result)

theme_sentiment_year_results = pd.DataFrame(
    theme_sentiment_rows
)

theme_sentiment_year_results["p_value_fdr"] = (
    benjamini_hochberg(
        theme_sentiment_year_results["p_value"]
    )
)

theme_sentiment_year_results["significant_fdr_05"] = (
    theme_sentiment_year_results["p_value_fdr"] < 0.05
)

theme_sentiment_year_results = (
    theme_sentiment_year_results.sort_values(
        "p_value_fdr",
        na_position="last"
    )
)

print("\n" + "=" * 95)
print("THEME SHARE VS SENTIMENT — CONTROLLING FOR YEAR")
print("=" * 95)
display(theme_sentiment_year_results.round(4))

# ------------------------------------------------
# 8. YEAR ASSOCIATION WITH INDIA GROSS
# ------------------------------------------------

year_gross = partial_spearman_year(
    df.assign(year_numeric=df["year"]),
    "year_numeric",
    "india_gross_cr"
)

# Ordinary Spearman is more appropriate here because
# year itself is the variable whose association is tested.
from scipy.stats import spearmanr

year_rho, year_p = spearmanr(
    df["year"], df["india_gross_cr"],
    nan_policy="omit"
)

print("\nYEAR VS INDIA GROSS")
print("-" * 70)
print("Spearman rho:", round(float(year_rho), 4))
print("p-value:", round(float(year_p), 6))

# ------------------------------------------------
# 9. SAVE RESULTS
# ------------------------------------------------

TABLE_DIR = PROJECT_ROOT / "05_outputs" / "tables"
SUMMARY_DIR = PROJECT_ROOT / "05_outputs" / "summary"

mismatch.to_csv(
    TABLE_DIR / "ICMB_130_SENTIMENT_COUNT_MISMATCH_CHECK.csv",
    index=False
)

theme_year_results.to_csv(
    TABLE_DIR / "ICMB_130_THEME_VS_BOXOFFICE_YEAR_ADJUSTED.csv",
    index=False
)

sentiment_year_results.to_csv(
    TABLE_DIR / "ICMB_130_SENTIMENT_VS_BOXOFFICE_YEAR_ADJUSTED.csv",
    index=False
)

theme_sentiment_year_results.to_csv(
    TABLE_DIR / "ICMB_130_THEME_VS_SENTIMENT_YEAR_ADJUSTED.csv",
    index=False
)

summary = {
    "films": int(df["movie_record_id"].nunique()),
    "sentiment_count_mismatch_films": int(len(mismatch)),
    "year_vs_gross_spearman_rho": float(year_rho),
    "year_vs_gross_p_value": float(year_p),
    "year_adjusted_theme_gross_significant": int(
        theme_year_results["significant_fdr_05"].sum()
    ),
    "year_adjusted_sentiment_gross_significant": int(
        sentiment_year_results["significant_fdr_05"].sum()
    ),
    "year_adjusted_theme_sentiment_significant": int(
        theme_sentiment_year_results["significant_fdr_05"].sum()
    ),
    "method": (
        "Partial Spearman using rank-transformed variables and "
        "year fixed-effect residualization; exploratory association."
    )
}

with open(
    SUMMARY_DIR / "ICMB_130_NOTEBOOK11_CELL16_SUMMARY.json",
    "w"
) as f:
    json.dump(summary, f, indent=4)

print("\nSAVED YEAR-ADJUSTED RESULTS AND MISMATCH CHECK")
print(json.dumps(summary, indent=4))

print("\n" + "=" * 95)
print("CELL 16 COMPLETED")
print("=" * 95)


In [ ]:

# ================================================================
# CELL 17 — CORRECTED YEAR-ADJUSTED INFERENCE
# ================================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path
from scipy.stats import rankdata, t as student_t

print("=" * 95)
print("CELL 17 — CORRECTED YEAR-ADJUSTED INFERENCE")
print("=" * 95)

# ------------------------------------------------
# 1. LOAD INTEGRATED DATA
# ------------------------------------------------

input_path = (
    PROJECT_ROOT / "05_outputs" / "tables" /
    "ICMB_130_FINAL_INTEGRATED_FILM_ANALYTICS.csv"
)

df = pd.read_csv(input_path)

# Exactly the 12 theme features.
# Exclude theme_share_sum_pct, which is a derived total.
theme_cols = [
    c for c in df.columns
    if c.startswith("theme_share_")
    and c != "theme_share_sum_pct"
]

if len(theme_cols) != 12:
    raise ValueError(
        f"Expected exactly 12 theme features; found {len(theme_cols)}"
    )

sentiment_cols = [
    "positive_share_classifiable_pct",
    "negative_share_classifiable_pct",
    "neutral_share_classifiable_pct",
    "classifiable_share_total_pct",
    "uncertain_share_total_pct"
]

print("\nFEATURE VALIDATION")
print("-" * 70)
print("Theme features:", len(theme_cols))
print("Sentiment features:", len(sentiment_cols))
print("Unique years:", df["year"].nunique())
print("Films:", df["movie_record_id"].nunique())

# ------------------------------------------------
# 2. BENJAMINI-HOCHBERG CORRECTION
# ------------------------------------------------

def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p), np.nan)

    valid = np.where(np.isfinite(p))[0]

    if len(valid) == 0:
        return adjusted

    pv = p[valid]
    order = np.argsort(pv)
    sorted_p = pv[order]

    m = len(sorted_p)

    sorted_adjusted = sorted_p * m / np.arange(1, m + 1)
    sorted_adjusted = np.minimum.accumulate(
        sorted_adjusted[::-1]
    )[::-1]
    sorted_adjusted = np.clip(sorted_adjusted, 0, 1)

    restored = np.empty(m)
    restored[order] = sorted_adjusted
    adjusted[valid] = restored

    return adjusted


# ------------------------------------------------
# 3. YEAR-ADJUSTED RANK CORRELATION
# ------------------------------------------------

def corrected_partial_spearman(data, x_col, y_col):

    temp = data[
        ["year", x_col, y_col]
    ].replace([np.inf, -np.inf], np.nan).dropna().copy()

    n = len(temp)
    year_levels = temp["year"].nunique()

    if n < 10 or year_levels < 2:
        return {
            "feature": x_col,
            "outcome": y_col,
            "n": n,
            "year_categories": year_levels,
            "year_controls": max(year_levels - 1, 0),
            "degrees_of_freedom": np.nan,
            "partial_spearman_rho": np.nan,
            "p_value_corrected_df": np.nan
        }

    # Rank-transform the two variables
    x_rank = rankdata(temp[x_col].to_numpy())
    y_rank = rankdata(temp[y_col].to_numpy())

    # Year fixed effects
    year_dummies = pd.get_dummies(
        temp["year"].astype(str),
        drop_first=True,
        dtype=float
    ).to_numpy()

    design = np.column_stack([
        np.ones(n),
        year_dummies
    ])

    # Residualize ranks after accounting for year
    x_resid = x_rank - design @ np.linalg.lstsq(
        design, x_rank, rcond=None
    )[0]

    y_resid = y_rank - design @ np.linalg.lstsq(
        design, y_rank, rcond=None
    )[0]

    if np.std(x_resid) == 0 or np.std(y_resid) == 0:
        rho = np.nan
        p_value = np.nan
    else:
        rho = float(np.corrcoef(x_resid, y_resid)[0, 1])

        # Number of independent year controls
        k = year_levels - 1

        # Partial correlation degrees of freedom:
        # n - k - 2
        df_residual = n - k - 2

        if df_residual <= 0 or abs(rho) >= 1:
            p_value = np.nan
        else:
            t_stat = rho * np.sqrt(
                df_residual / (1 - rho ** 2)
            )

            p_value = float(
                2 * student_t.sf(
                    abs(t_stat), df_residual
                )
            )

    k = year_levels - 1
    df_residual = n - k - 2

    return {
        "feature": x_col,
        "outcome": y_col,
        "n": n,
        "year_categories": year_levels,
        "year_controls": k,
        "degrees_of_freedom": df_residual,
        "partial_spearman_rho": rho,
        "p_value_corrected_df": p_value
    }


# ------------------------------------------------
# 4. THEME SHARE VS INDIA GROSS
# ------------------------------------------------

theme_results = pd.DataFrame([
    corrected_partial_spearman(
        df, feature, "india_gross_cr"
    )
    for feature in theme_cols
])

theme_results["p_value_fdr"] = benjamini_hochberg(
    theme_results["p_value_corrected_df"]
)

theme_results["significant_fdr_05"] = (
    theme_results["p_value_fdr"] < 0.05
)

theme_results = theme_results.sort_values(
    "p_value_fdr",
    na_position="last"
)

print("\n" + "=" * 95)
print("12 THEMES VS INDIA GROSS — CORRECTED YEAR-ADJUSTED TESTS")
print("=" * 95)
display(theme_results.round(5))

# ------------------------------------------------
# 5. SENTIMENT VS INDIA GROSS
# ------------------------------------------------

sentiment_results = pd.DataFrame([
    corrected_partial_spearman(
        df, feature, "india_gross_cr"
    )
    for feature in sentiment_cols
])

sentiment_results["p_value_fdr"] = benjamini_hochberg(
    sentiment_results["p_value_corrected_df"]
)

sentiment_results["significant_fdr_05"] = (
    sentiment_results["p_value_fdr"] < 0.05
)

sentiment_results = sentiment_results.sort_values(
    "p_value_fdr",
    na_position="last"
)

print("\n" + "=" * 95)
print("SENTIMENT VS INDIA GROSS — CORRECTED YEAR-ADJUSTED TESTS")
print("=" * 95)
display(sentiment_results.round(5))

# ------------------------------------------------
# 6. THEME SHARE VS SENTIMENT
# ------------------------------------------------

theme_sentiment_rows = []

for theme_feature in theme_cols:
    for sentiment_feature in [
        "positive_share_classifiable_pct",
        "negative_share_classifiable_pct",
        "neutral_share_classifiable_pct"
    ]:
        theme_sentiment_rows.append(
            corrected_partial_spearman(
                df, theme_feature, sentiment_feature
            )
        )

theme_sentiment_results = pd.DataFrame(
    theme_sentiment_rows
)

theme_sentiment_results["p_value_fdr"] = benjamini_hochberg(
    theme_sentiment_results["p_value_corrected_df"]
)

theme_sentiment_results["significant_fdr_05"] = (
    theme_sentiment_results["p_value_fdr"] < 0.05
)

theme_sentiment_results = theme_sentiment_results.sort_values(
    "p_value_fdr",
    na_position="last"
)

print("\n" + "=" * 95)
print("THEMES VS SENTIMENT — CORRECTED YEAR-ADJUSTED TESTS")
print("=" * 95)
display(theme_sentiment_results.round(5))

# ------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------

TABLE_DIR = PROJECT_ROOT / "05_outputs" / "tables"
SUMMARY_DIR = PROJECT_ROOT / "05_outputs" / "summary"

theme_results.to_csv(
    TABLE_DIR / "ICMB_130_THEME_VS_BOXOFFICE_YEAR_ADJUSTED_FINAL.csv",
    index=False
)

sentiment_results.to_csv(
    TABLE_DIR / "ICMB_130_SENTIMENT_VS_BOXOFFICE_YEAR_ADJUSTED_FINAL.csv",
    index=False
)

theme_sentiment_results.to_csv(
    TABLE_DIR / "ICMB_130_THEME_VS_SENTIMENT_YEAR_ADJUSTED_FINAL.csv",
    index=False
)

# ------------------------------------------------
# 8. FINAL SUMMARY
# ------------------------------------------------

summary = {
    "films": int(df["movie_record_id"].nunique()),
    "theme_features_tested": int(len(theme_cols)),
    "sentiment_features_tested": int(len(sentiment_cols)),
    "theme_sentiment_tests": int(len(theme_sentiment_results)),
    "year_categories": int(df["year"].nunique()),
    "year_controls": int(df["year"].nunique() - 1),
    "nominal_residual_degrees_of_freedom": int(
        len(df) - df["year"].nunique() - 1
    ),
    "theme_gross_fdr_significant": int(
        theme_results["significant_fdr_05"].sum()
    ),
    "sentiment_gross_fdr_significant": int(
        sentiment_results["significant_fdr_05"].sum()
    ),
    "theme_sentiment_fdr_significant": int(
        theme_sentiment_results["significant_fdr_05"].sum()
    ),
    "method": (
        "Rank-transformed partial correlation controlling for year "
        "fixed effects. P-values use n-k-2 degrees of freedom, "
        "followed by Benjamini-Hochberg correction."
    ),
    "caution": (
        "Exploratory inference. Small within-year groups, observational "
        "data, uncertain sentiment classifications and undocumented "
        "box-office provenance limit interpretation."
    )
}

with open(
    SUMMARY_DIR / "ICMB_130_NOTEBOOK11_CELL17_SUMMARY.json",
    "w"
) as f:
    json.dump(summary, f, indent=4)

print("\n" + "=" * 95)
print("CELL 17 SUMMARY")
print("=" * 95)
print(json.dumps(summary, indent=4))

print("\n" + "=" * 95)
print("CELL 17 COMPLETED")
print("=" * 95)


In [ ]:

# ================================================================
# NOTEBOOK 11 — CELL 18
# FINAL RESEARCH FINDINGS AND REPORT-READY TABLES
# Project: ICMB-130 Indian Cinema Multimodal Benchmark Dataset
# ================================================================

import os
import json
import warnings
import pandas as pd
import numpy as np
from datetime import datetime

warnings.filterwarnings("ignore")

# ------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

TABLES_DIR = os.path.join(PROJECT_ROOT, "05_outputs", "tables")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "05_outputs", "figures")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "05_outputs", "reports")
SUMMARY_DIR = os.path.join(PROJECT_ROOT, "05_outputs", "summary")

for folder in [TABLES_DIR, FIGURES_DIR, REPORTS_DIR, SUMMARY_DIR]:
    os.makedirs(folder, exist_ok=True)

print("=" * 75)
print("ICMB-130 | CELL 18: FINAL RESEARCH FINDINGS")
print("=" * 75)


# ------------------------------------------------
# 2. INPUT FILES
# ------------------------------------------------

INPUT_FILES = {
    "integrated": os.path.join(
        TABLES_DIR,
        "ICMB_130_FINAL_INTEGRATED_FILM_ANALYTICS.csv"
    ),
    "period": os.path.join(
        TABLES_DIR,
        "ICMB_130_PERIOD_WISE_INTEGRATED_SUMMARY.csv"
    ),
    "longitudinal": os.path.join(
        PROJECT_ROOT,
        "06_outputs",
        "topic_modelling",
        "ICMB_130_FINAL_LONGITUDINAL_THEME_SUMMARY.csv"
    ),
    "annual_theme": os.path.join(
        PROJECT_ROOT,
        "06_outputs",
        "topic_modelling",
        "ICMB_130_FINAL_ANNUAL_THEME_PREVALENCE.csv"
    ),
    "theme_gross_raw": os.path.join(
        TABLES_DIR,
        "ICMB_130_THEME_VS_BOXOFFICE_SPEARMAN.csv"
    ),
    "sentiment_gross_raw": os.path.join(
        TABLES_DIR,
        "ICMB_130_SENTIMENT_VS_BOXOFFICE_SPEARMAN.csv"
    ),
    "theme_sentiment_raw": os.path.join(
        TABLES_DIR,
        "ICMB_130_THEME_VS_SENTIMENT_SPEARMAN.csv"
    ),
    "theme_gross_adjusted": os.path.join(
        TABLES_DIR,
        "ICMB_130_THEME_VS_BOXOFFICE_YEAR_ADJUSTED_FINAL.csv"
    ),
    "sentiment_gross_adjusted": os.path.join(
        TABLES_DIR,
        "ICMB_130_SENTIMENT_VS_BOXOFFICE_YEAR_ADJUSTED_FINAL.csv"
    ),
    "theme_sentiment_adjusted": os.path.join(
        TABLES_DIR,
        "ICMB_130_THEME_VS_SENTIMENT_YEAR_ADJUSTED_FINAL.csv"
    ),
    "film_sentiment": os.path.join(
        TABLES_DIR,
        "ICMB_130_FINAL_FILM_LEVEL_SENTIMENT_SUMMARY.csv"
    ),
    "film_theme": os.path.join(
        TABLES_DIR,
        "ICMB_130_FILM_LEVEL_THEME_FEATURE_MATRIX.csv"
    ),
    "master": os.path.join(
        PROJECT_ROOT,
        "01_data",
        "processed",
        "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
    ),
    "sentiment_comments": os.path.join(
        PROJECT_ROOT,
        "01_data",
        "processed",
        "audience_sentiment",
        "ICMB_130_YOUTUBE_COMMENTS_SENTIMENT_BASELINE.csv"
    ),
    "subtitle_manifest": os.path.join(
        PROJECT_ROOT,
        "01_data",
        "processed",
        "subtitle_corpus_v2",
        "ICMB_130_SUBTITLE_ACQUISITION_MANIFEST.csv"
    )
}

missing_files = [
    name for name, path in INPUT_FILES.items()
    if not os.path.exists(path)
]

if missing_files:
    raise FileNotFoundError(
        "Required input files are missing:\n" +
        "\n".join(
            f"{name}: {INPUT_FILES[name]}"
            for name in missing_files
        )
    )

print("\nAll required input files found.")


# ------------------------------------------------
# 3. LOAD EXISTING RESULTS
# ------------------------------------------------

data = {
    name: pd.read_csv(path)
    for name, path in INPUT_FILES.items()
}

integrated = data["integrated"]
period = data["period"]
longitudinal = data["longitudinal"]
annual_theme = data["annual_theme"]

theme_gross_raw = data["theme_gross_raw"]
sentiment_gross_raw = data["sentiment_gross_raw"]
theme_sentiment_raw = data["theme_sentiment_raw"]

theme_gross_adj = data["theme_gross_adjusted"]
sentiment_gross_adj = data["sentiment_gross_adjusted"]
theme_sentiment_adj = data["theme_sentiment_adjusted"]

film_sentiment = data["film_sentiment"]
film_theme = data["film_theme"]
master = data["master"]
comments = data["sentiment_comments"]
subtitle_manifest = data["subtitle_manifest"]

print("\nINPUT DATASET DIMENSIONS")
print("-" * 75)

for name, df in data.items():
    print(f"{name:30s} {df.shape}")


# ------------------------------------------------
# 4. HELPER FUNCTIONS
# ------------------------------------------------

def find_column(df, candidates):
    """Find the first available column from candidate names."""
    for col in candidates:
        if col in df.columns:
            return col
    return None


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def make_overview_table():
    return pd.DataFrame({
        "Metric": [
            "Benchmark films",
            "Study period",
            "Annual selection",
            "Ranking metric",
            "Integrated film records",
            "Films with positive subtitle dialogue blocks",
            "Films with all three analytical dimensions",
            "YouTube comments",
            "Classifiable YouTube comments",
            "Curated dialogue themes",
            "Annual theme records"
        ],
        "Value": [
            130,
            "2000–2025",
            "Top 5 films per year",
            "India Gross / India Nett",
            len(integrated),
            122,
            122,
            len(comments),
            6093,
            12,
            len(annual_theme)
        ]
    })


# ------------------------------------------------
# 5. BOX-OFFICE PERIOD TABLE
# ------------------------------------------------

print("\nBOX-OFFICE PERIOD SUMMARY")
print("-" * 75)

print(period.to_string(index=False))


# ------------------------------------------------
# 6. LONGITUDINAL THEME TABLE
# ------------------------------------------------

print("\nLONGITUDINAL THEME SUMMARY")
print("-" * 75)

print(longitudinal.head(15).to_string(index=False))


# ------------------------------------------------
# 7. SENTIMENT DISTRIBUTION TABLE
# ------------------------------------------------

sentiment_col = find_column(
    comments,
    ["final_sentiment", "sentiment", "predicted_sentiment"]
)

if sentiment_col is None:
    raise KeyError(
        "Could not identify the sentiment classification column."
    )

sentiment_distribution = (
    comments[sentiment_col]
    .fillna("Missing")
    .value_counts()
    .rename_axis("Sentiment Category")
    .reset_index(name="Comment Count")
)

sentiment_distribution["Percentage of All Comments"] = (
    sentiment_distribution["Comment Count"] /
    len(comments) * 100
).round(2)

sentiment_distribution["Percentage of All Comments"] = (
    sentiment_distribution["Percentage of All Comments"].astype(str) + "%"
)

print("\nYOUTUBE SENTIMENT DISTRIBUTION")
print("-" * 75)

print(sentiment_distribution.to_string(index=False))


# ------------------------------------------------
# 8. ASSOCIATION TABLES
# ------------------------------------------------

print("\nUNADJUSTED ASSOCIATION RESULTS")
print("-" * 75)

print("Theme vs Box Office:")
print(theme_gross_raw.to_string(index=False))

print("\nSentiment vs Box Office:")
print(sentiment_gross_raw.to_string(index=False))

print("\nTheme vs Sentiment:")
print(theme_sentiment_raw.to_string(index=False))


print("\nYEAR-ADJUSTED ASSOCIATION RESULTS")
print("-" * 75)

print("Theme vs Box Office:")
print(theme_gross_adj.to_string(index=False))

print("\nSentiment vs Box Office:")
print(sentiment_gross_adj.to_string(index=False))

print("\nTheme vs Sentiment:")
print(theme_sentiment_adj.to_string(index=False))


# ------------------------------------------------
# 9. COVERAGE AND DATA QUALITY TABLE
# ------------------------------------------------

subtitle_dialogue_col = find_column(
    subtitle_manifest,
    ["dialogue_blocks"]
)

subtitle_word_col = find_column(
    subtitle_manifest,
    ["word_count"]
)

if subtitle_dialogue_col and subtitle_word_col:
    subtitle_dialogue = safe_numeric(
        subtitle_manifest[subtitle_dialogue_col]
    ).fillna(0)

    subtitle_words = safe_numeric(
        subtitle_manifest[subtitle_word_col]
    ).fillna(0)

    films_with_dialogue = int(
        ((subtitle_dialogue > 0) & (subtitle_words > 0)).sum()
    )
else:
    films_with_dialogue = 122

source_col = find_column(
    master,
    ["india_gross_source"]
)

verified_col = find_column(
    master,
    ["india_gross_verified"]
)

if source_col:
    gross_source_documented = int(
        master[source_col].notna().sum()
    )
else:
    gross_source_documented = 0

if verified_col:
    gross_verified_count = int(
        master[verified_col]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["yes", "true", "1", "verified"])
        .sum()
    )
else:
    gross_verified_count = 0

coverage_table = pd.DataFrame({
    "Quality / Coverage Metric": [
        "Total benchmark films",
        "Films with positive subtitle dialogue blocks and word counts",
        "Films without positive subtitle dialogue blocks and word counts",
        "Films with all three analytical dimensions available",
        "Total YouTube comments",
        "Films with at least 100 YouTube comments",
        "Films with fewer than 100 YouTube comments",
        "Documented India Gross source entries",
        "India Gross entries marked verified",
        "Subtitle paths resolved in Cell 13",
        "YouTube video manifest records manually verified",
        "Known film-level sentiment count mismatches"
    ],
    "Value": [
        130,
        films_with_dialogue,
        130 - films_with_dialogue,
        122,
        len(comments),
        122,
        8,
        gross_source_documented,
        gross_verified_count,
        "0 / 130",
        "0 / 12 in inspected Batch 04 manifest",
        1
    ],
    "Interpretation": [
        "Top five films per year, 2000–2025",
        "Positive dialogue block and word-count values",
        "Require subtitle coverage review",
        "Availability-based integration readiness",
        "Collected audience comments",
        "At least 100 comments per film",
        "Lower comment coverage",
        "Source provenance is undocumented in the master file",
        "No entries currently marked verified",
        "Stored subtitle paths did not resolve during the path audit",
        "Manual verification was pending in the inspected manifest",
        "Vikram (2022_3): one comment is unaccounted for in category totals"
    ]
})

print("\nCOVERAGE AND DATA QUALITY")
print("-" * 75)

print(coverage_table.to_string(index=False))


# ------------------------------------------------
# 10. REPORT-READY FINDINGS
# ------------------------------------------------

findings = [
    {
        "Finding ID": "F01",
        "Research Dimension": "Commercial Performance",
        "Finding": (
            "India Gross shows a strong positive temporal association "
            "within the selected 130-film benchmark sample."
        ),
        "Evidence": (
            "Spearman rho = 0.8700; p = 3.90e-41."
        ),
        "Interpretation": (
            "The association describes nominal gross values in the "
            "selected top-five-per-year sample. It is not evidence of "
            "inflation-adjusted industry growth."
        )
    },
    {
        "Finding ID": "F02",
        "Research Dimension": "Commercial Performance",
        "Finding": (
            "Mean and median India Gross differ substantially across "
            "the three study periods."
        ),
        "Evidence": (
            "Early mean ₹68.509 crore; Middle mean ₹254.359 crore; "
            "Recent mean ₹458.803 crore."
        ),
        "Interpretation": (
            "Descriptive period differences should be interpreted "
            "in light of nominal values, inflation and sample selection."
        )
    },
    {
        "Finding ID": "F03",
        "Research Dimension": "Dialogue Themes",
        "Finding": (
            "Love and romantic relationship dialogue share declined "
            "across the three periods."
        ),
        "Evidence": (
            "Early 25.18%; Middle 21.43%; Recent 14.58%."
        ),
        "Interpretation": (
            "This is a descriptive pattern in the available subtitle "
            "corpus, not a claim about all Indian films."
        )
    },
    {
        "Finding ID": "F04",
        "Research Dimension": "Dialogue Themes",
        "Finding": (
            "Marriage and wedding dialogue share also declined."
        ),
        "Evidence": (
            "Early 8.36%; Middle 6.86%; Recent 5.16%."
        ),
        "Interpretation": (
            "The pattern is limited to the curated themes and "
            "available dialogue material."
        )
    },
    {
        "Finding ID": "F05",
        "Research Dimension": "Dialogue Themes",
        "Finding": (
            "Family and parent-child dialogue share declined initially "
            "and then remained broadly stable."
        ),
        "Evidence": (
            "Early 5.85%; Middle 4.71%; Recent 5.07%."
        ),
        "Interpretation": (
            "The pattern is descriptive and should not be generalized "
            "beyond the benchmark corpus."
        )
    },
    {
        "Finding ID": "F06",
        "Research Dimension": "Audience Sentiment",
        "Finding": (
            "A minority of collected YouTube comments received "
            "positive, negative or neutral classifications."
        ),
        "Evidence": (
            "6,093 of 15,960 comments (38.18%) were classifiable."
        ),
        "Interpretation": (
            "A substantial uncertain or otherwise unclassified share "
            "limits sentiment conclusions."
        )
    },
    {
        "Finding ID": "F07",
        "Research Dimension": "Audience Sentiment",
        "Finding": (
            "Positive comments form the largest class among "
            "classifiable comments."
        ),
        "Evidence": (
            "Positive 48.99%; Negative 35.58%; Neutral 15.43%."
        ),
        "Interpretation": (
            "These proportions describe collected comments only and "
            "are not representative audience approval estimates."
        )
    },
    {
        "Finding ID": "F08",
        "Research Dimension": "Integrated Analysis",
        "Finding": (
            "No theme–gross, sentiment–gross or theme–sentiment "
            "association remained statistically significant after "
            "year adjustment and multiple-testing correction."
        ),
        "Evidence": (
            "Cell 17: 0 significant theme–gross, 0 significant "
            "sentiment–gross and 0 significant theme–sentiment tests."
        ),
        "Interpretation": (
            "The analysis does not establish robust independent "
            "associations after accounting for year."
        )
    },
    {
        "Finding ID": "F09",
        "Research Dimension": "Integrated Analysis",
        "Finding": (
            "All 130 film identifiers were matched across the "
            "master, theme and film-level sentiment datasets."
        ),
        "Evidence": (
            "130 common unique movie_record_id values."
        ),
        "Interpretation": (
            "Successful record-level integration does not imply "
            "equal source quality or complete modality coverage."
        )
    }
]

findings_table = pd.DataFrame(findings)


# ------------------------------------------------
# 11. CREATE EXCEL WORKBOOK
# ------------------------------------------------

excel_path = os.path.join(
    REPORTS_DIR,
    "ICMB_130_FINAL_RESEARCH_FINDINGS_TABLES.xlsx"
)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:

    make_overview_table().to_excel(
        writer, sheet_name="Project Overview", index=False
    )

    period.to_excel(
        writer, sheet_name="Box Office Periods", index=False
    )

    longitudinal.to_excel(
        writer, sheet_name="Longitudinal Themes", index=False
    )

    annual_theme.to_excel(
        writer, sheet_name="Annual Theme Prevalence", index=False
    )

    sentiment_distribution.to_excel(
        writer, sheet_name="Sentiment Distribution", index=False
    )

    findings_table.to_excel(
        writer, sheet_name="Final Findings", index=False
    )

    theme_gross_raw.to_excel(
        writer, sheet_name="Theme Gross Raw", index=False
    )

    sentiment_gross_raw.to_excel(
        writer, sheet_name="Sentiment Gross Raw", index=False
    )

    theme_sentiment_raw.to_excel(
        writer, sheet_name="Theme Sentiment Raw", index=False
    )

    theme_gross_adj.to_excel(
        writer, sheet_name="Theme Gross Adjusted", index=False
    )

    sentiment_gross_adj.to_excel(
        writer, sheet_name="Sentiment Gross Adjusted", index=False
    )

    theme_sentiment_adj.to_excel(
        writer, sheet_name="Theme Sentiment Adjusted", index=False
    )

    coverage_table.to_excel(
        writer, sheet_name="Coverage Limitations", index=False
    )

print("\nExcel workbook created:")
print(excel_path)


# ------------------------------------------------
# 12. CREATE MARKDOWN RESEARCH SUMMARY
# ------------------------------------------------

markdown_path = os.path.join(
    REPORTS_DIR,
    "ICMB_130_FINAL_RESEARCH_FINDINGS_SUMMARY.md"
)

markdown_content = """
# ICMB-130: Final Research Findings

## Project Title

A Multimodal Analytics Framework for Long-Term Entertainment Trend
Discovery Across Indian Cinema (2000–2025)

## 1. Dataset Overview

The ICMB-130 benchmark contains 130 films selected as the top five
highest-grossing films per year from 2000 to 2025, using India Gross /
India Nett as the ranking metric.

The project integrates commercial metadata, subtitle-derived dialogue
themes and collected YouTube audience comments.

All 130 movie identifiers were matched across the master, theme and
film-level sentiment datasets. This indicates successful record-level
integration, not equal completeness or reliability across modalities.

## 2. Commercial Performance

The selected sample exhibits a strong positive temporal association
between release year and nominal India Gross (Spearman rho = 0.8700,
p = 3.90e-41).

Mean India Gross was ₹68.509 crore in the Early period (2000–2008),
₹254.359 crore in the Middle period (2009–2017), and ₹458.803 crore
in the Recent period (2018–2025).

These figures are descriptive. Gross values are nominal and have not
been adjusted for inflation. The top-five-per-year selection also
means that these findings should not be interpreted as a complete
representation of the Indian film industry.

## 3. Dialogue Theme Trends

The curated dialogue analysis identified 12 themes.

Love and romantic relationship dialogue share declined from 25.18%
in the Early period to 21.43% in the Middle period and 14.58% in
the Recent period.

Marriage and wedding dialogue share declined from 8.36% to 6.86%
and 5.16%, respectively.

Family and parent-child dialogue share was 5.85%, 4.71% and 5.07%
across the three periods.

These are descriptive patterns within the available subtitle corpus.
They do not establish industry-wide changes in narrative content.

## 4. Audience Sentiment

The collected YouTube dataset contained 15,960 comments. Of these,
6,093 (38.18%) were classified as positive, negative or neutral.

Among classifiable comments:
- Positive: 48.99%
- Negative: 35.58%
- Neutral: 15.43%

A substantial proportion of comments remained uncertain or otherwise
unclassified. Language coverage and model validation limitations mean
that these findings are exploratory and should not be interpreted as
representative audience opinion.

## 5. Integrated Statistical Analysis

Unadjusted exploratory associations were observed between some
dialogue or sentiment features and India Gross.

However, after year adjustment and multiple-testing correction,
no theme–gross, sentiment–gross or theme–sentiment association
remained statistically significant.

Therefore, the analysis does not establish robust independent
associations between dialogue themes, collected comment sentiment
and commercial performance in this benchmark sample.

The results should not be interpreted causally.

## 6. Data Quality and Limitations

1. The benchmark contains only the top five selected films per year.
2. India Gross values are nominal and not inflation-adjusted.
3. India Gross source attribution is not documented in the current
   master dataset, and no entries are marked verified.
4. Eight films do not have positive subtitle dialogue-block and
   word-count values in the manifest.
5. None of the 130 stored subtitle file paths resolved during the
   Notebook 11 path audit. Subtitle source-file revalidation remains
   unresolved.
6. YouTube sentiment classification is exploratory. A substantial
   proportion of comments are uncertain or unclassified.
7. The inspected Batch 04 video manifest had 12 records whose manual
   verification status was pending.
8. One film-level sentiment count mismatch was identified for
   Vikram (2022_3): 100 total comments versus 99 comments accounted
   for in the listed sentiment categories.
9. IMDb review analysis has not been established as a completed
   component of this analysis.

## 7. Overall Conclusion

ICMB-130 provides an integrated benchmark for exploring commercial
performance, dialogue themes and collected audience responses across
selected Indian films released between 2000 and 2025.

The analysis identifies descriptive changes in nominal box-office
values and curated dialogue-theme prevalence. Audience sentiment
coverage and language uncertainty constrain the strength of sentiment
interpretation.

After year adjustment and multiple-testing correction, no robust
independent statistical associations were established among the
analysed dialogue themes, audience sentiment features and India Gross.

The benchmark is therefore best understood as an exploratory,
multimodal research resource, with documented data-quality limitations
that should guide interpretation and future validation.
"""

with open(markdown_path, "w", encoding="utf-8") as f:
    f.write(markdown_content.strip())

print("\nMarkdown research summary created:")
print(markdown_path)


# ------------------------------------------------
# 13. CREATE MACHINE-READABLE SUMMARY JSON
# ------------------------------------------------

summary_json = {
    "project": "ICMB-130 Indian Cinema Multimodal Benchmark Dataset",
    "project_period": "2000-2025",
    "benchmark_films": 130,
    "selection_method": "Top 5 highest-grossing films per year",
    "ranking_metric": "India Gross / India Nett",
    "integrated_film_records": int(len(integrated)),
    "subtitle_films_with_positive_dialogue": int(films_with_dialogue),
    "films_with_all_three_dimensions": 122,
    "youtube_comments": int(len(comments)),
    "classifiable_comments": 6093,
    "classifiable_comment_percentage": round(
        6093 / len(comments) * 100, 2
    ),
    "sentiment_distribution_among_classifiable": {
        "positive_pct": 48.99,
        "negative_pct": 35.58,
        "neutral_pct": 15.43
    },
    "annual_gross_year_spearman": {
        "rho": 0.8700,
        "p_value": 3.90e-41
    },
    "year_adjusted_significant_associations": {
        "theme_vs_box_office": 0,
        "sentiment_vs_box_office": 0,
        "theme_vs_sentiment": 0
    },
    "important_limitations": [
        "Nominal India Gross values are not inflation-adjusted.",
        "India Gross source provenance is undocumented in the master dataset.",
        "Eight films lack positive subtitle dialogue-block and word-count values.",
        "Subtitle file paths did not resolve during the Notebook 11 path audit.",
        "YouTube sentiment classification remains exploratory.",
        "Manual verification was pending for the 12 inspected Batch 04 video records.",
        "One sentiment category count mismatch exists for Vikram (2022_3).",
        "IMDb review analysis has not been established as completed."
    ],
    "generated_at": datetime.now().isoformat(timespec="seconds")
}

json_path = os.path.join(
    SUMMARY_DIR,
    "ICMB_130_FINAL_RESEARCH_FINDINGS_SUMMARY.json"
)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=4, ensure_ascii=False)

print("\nJSON summary created:")
print(json_path)


# ------------------------------------------------
# 14. FINAL OUTPUT VERIFICATION
# ------------------------------------------------

print("\n" + "=" * 75)
print("CELL 18 — OUTPUT VERIFICATION")
print("=" * 75)

OUTPUT_FILES = [
    excel_path,
    markdown_path,
    json_path
]

verification = []

for path in OUTPUT_FILES:
    exists = os.path.exists(path)
    size_kb = (
        round(os.path.getsize(path) / 1024, 2)
        if exists else 0
    )

    verification.append({
        "file": os.path.basename(path),
        "exists": exists,
        "size_kb": size_kb,
        "path": path
    })

verification_df = pd.DataFrame(verification)

print(verification_df.to_string(index=False))

assert verification_df["exists"].all(), (
    "One or more Cell 18 deliverables were not created."
)

print("\n" + "=" * 75)
print("CELL 18 COMPLETED SUCCESSFULLY")
print("=" * 75)

print("\nDeliverables:")
print("1. Excel workbook with report-ready research tables")
print("2. Markdown final research findings summary")
print("3. JSON machine-readable research summary")

print("\nNo additional statistical tests were performed.")


In [ ]:

# ================================================================
# NOTEBOOK 11 — CELL 19
# FINAL PROJECT DELIVERABLES AUDIT
# Project: ICMB-130 Indian Cinema Multimodal Benchmark Dataset
# ================================================================

import os
import pandas as pd
from datetime import datetime

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

OUTPUT_ROOT = os.path.join(PROJECT_ROOT, "05_outputs")
AUDIT_DIR = os.path.join(OUTPUT_ROOT, "reports")
os.makedirs(AUDIT_DIR, exist_ok=True)

print("=" * 80)
print("ICMB-130 | CELL 19: FINAL DELIVERABLES AUDIT")
print("=" * 80)

# ------------------------------------------------
# 1. CHECK PROJECT ROOT
# ------------------------------------------------

if not os.path.exists(PROJECT_ROOT):
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_ROOT}"
    )

print("\nPROJECT ROOT:")
print(PROJECT_ROOT)

# ------------------------------------------------
# 2. INVENTORY ALL PROJECT FILES
# ------------------------------------------------

records = []

for root, dirs, files in os.walk(PROJECT_ROOT):

    # Avoid scanning unnecessary cache folders
    dirs[:] = [
        d for d in dirs
        if d not in ["__pycache__", ".ipynb_checkpoints", ".git"]
    ]

    for filename in files:

        full_path = os.path.join(root, filename)
        relative_path = os.path.relpath(full_path, PROJECT_ROOT)

        try:
            size_kb = round(os.path.getsize(full_path) / 1024, 2)
            modified = datetime.fromtimestamp(
                os.path.getmtime(full_path)
            ).strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            size_kb = None
            modified = None

        extension = os.path.splitext(filename)[1].lower()

        records.append({
            "file_name": filename,
            "extension": extension,
            "relative_path": relative_path,
            "size_kb": size_kb,
            "last_modified": modified
        })

inventory = pd.DataFrame(records)

print("\nTOTAL PROJECT FILES:", len(inventory))

# ------------------------------------------------
# 3. CLASSIFY PROJECT DELIVERABLES
# ------------------------------------------------

def classify_file(row):

    name = row["file_name"].lower()
    ext = row["extension"]
    path = row["relative_path"].lower()

    if ext == ".ipynb":
        return "Notebook"

    if ext in [".sql"]:
        return "SQL Script"

    if ext == ".pbix":
        return "Power BI Dashboard"

    if ext in [".xlsx", ".xls"]:
        return "Excel Workbook"

    if ext == ".csv":
        return "CSV Dataset"

    if ext in [".png", ".jpg", ".jpeg", ".svg", ".pdf"]:
        if "figure" in path or "figures" in path:
            return "Research Figure"
        return "Document / Image"

    if ext in [".md", ".txt", ".docx", ".pdf"]:
        return "Research Document"

    if ext == ".json":
        return "JSON Summary / Configuration"

    if ext in [".py"]:
        return "Python Script"

    return "Other"

inventory["deliverable_category"] = inventory.apply(
    classify_file, axis=1
)

# ------------------------------------------------
# 4. NOTEBOOK AUDIT
# ------------------------------------------------

notebooks = inventory[
    inventory["extension"] == ".ipynb"
].copy()

print("\n" + "=" * 80)
print("NOTEBOOK INVENTORY")
print("=" * 80)

if len(notebooks):
    print(
        notebooks[
            ["file_name", "relative_path", "size_kb"]
        ].to_string(index=False)
    )
else:
    print("No notebook files found in the project directory.")

# ------------------------------------------------
# 5. SQL DELIVERABLE AUDIT
# ------------------------------------------------

sql_files = inventory[
    inventory["extension"] == ".sql"
].copy()

print("\n" + "=" * 80)
print("SQL DELIVERABLES")
print("=" * 80)

if len(sql_files):
    print(
        sql_files[
            ["file_name", "relative_path", "size_kb"]
        ].to_string(index=False)
    )
else:
    print("No SQL files found.")

# ------------------------------------------------
# 6. POWER BI DASHBOARD AUDIT
# ------------------------------------------------

powerbi_files = inventory[
    inventory["extension"] == ".pbix"
].copy()

print("\n" + "=" * 80)
print("POWER BI DASHBOARD")
print("=" * 80)

if len(powerbi_files):
    print(
        powerbi_files[
            ["file_name", "relative_path", "size_kb"]
        ].to_string(index=False)
    )
else:
    print("No .pbix dashboard file found.")

# ------------------------------------------------
# 7. RESEARCH FIGURES AUDIT
# ------------------------------------------------

figure_extensions = [".png", ".jpg", ".jpeg", ".svg"]

figures = inventory[
    (inventory["extension"].isin(figure_extensions)) &
    (inventory["relative_path"].str.contains(
        "figure", case=False, na=False
    ))
].copy()

print("\n" + "=" * 80)
print("RESEARCH FIGURES")
print("=" * 80)

print("Total figures found:", len(figures))

if len(figures):
    print(
        figures[
            ["file_name", "relative_path", "size_kb"]
        ].to_string(index=False)
    )

# ------------------------------------------------
# 8. FINAL REPORT AND SUMMARY AUDIT
# ------------------------------------------------

report_extensions = [".md", ".docx", ".pdf", ".txt"]

research_reports = inventory[
    inventory["extension"].isin(report_extensions)
].copy()

print("\n" + "=" * 80)
print("RESEARCH DOCUMENTS")
print("=" * 80)

if len(research_reports):
    print(
        research_reports[
            ["file_name", "relative_path", "size_kb"]
        ].to_string(index=False)
    )
else:
    print("No research documents found.")

# ------------------------------------------------
# 9. KEY PROJECT DELIVERABLE CHECKLIST
# ------------------------------------------------

required_deliverables = [
    {
        "deliverable": "Notebook 9 - Topic Modelling",
        "condition": (
            notebooks["file_name"].str.contains(
                "9", case=False, na=False
            ).any()
        )
    },
    {
        "deliverable": "Notebook 10 - YouTube / Sentiment",
        "condition": (
            notebooks["file_name"].str.contains(
                "10", case=False, na=False
            ).any()
        )
    },
    {
        "deliverable": "Notebook 11 - Final Results",
        "condition": (
            notebooks["file_name"].str.contains(
                "11", case=False, na=False
            ).any()
        )
    },
    {
        "deliverable": "Integrated Film Analytics CSV",
        "condition": inventory["file_name"].eq(
            "ICMB_130_FINAL_INTEGRATED_FILM_ANALYTICS.csv"
        ).any()
    },
    {
        "deliverable": "Final Research Findings Excel",
        "condition": inventory["file_name"].eq(
            "ICMB_130_FINAL_RESEARCH_FINDINGS_TABLES.xlsx"
        ).any()
    },
    {
        "deliverable": "Final Research Findings Markdown",
        "condition": inventory["file_name"].eq(
            "ICMB_130_FINAL_RESEARCH_FINDINGS_SUMMARY.md"
        ).any()
    },
    {
        "deliverable": "Final Research Findings JSON",
        "condition": inventory["file_name"].eq(
            "ICMB_130_FINAL_RESEARCH_FINDINGS_SUMMARY.json"
        ).any()
    },
    {
        "deliverable": "SQL Analytical Scripts",
        "condition": len(sql_files) > 0
    },
    {
        "deliverable": "Power BI Dashboard",
        "condition": len(powerbi_files) > 0
    },
    {
        "deliverable": "Research Figures",
        "condition": len(figures) > 0
    },
    {
        "deliverable": "Final MTP-II Report (PDF / DOCX)",
        "condition": inventory[
            "file_name"
        ].str.contains(
            "mtp|final_report|major_project_report",
            case=False,
            na=False
        ).any()
    },
    {
        "deliverable": "Final Research Presentation",
        "condition": inventory[
            "extension"
        ].isin([".pptx", ".ppt"]).any()
    }
]

checklist = pd.DataFrame(required_deliverables)

checklist["status"] = checklist["condition"].map({
    True: "FOUND",
    False: "NOT FOUND"
})

checklist.drop(columns=["condition"], inplace=True)

print("\n" + "=" * 80)
print("KEY DELIVERABLE CHECKLIST")
print("=" * 80)

print(checklist.to_string(index=False))

# ------------------------------------------------
# 10. FILE CATEGORY SUMMARY
# ------------------------------------------------

category_summary = (
    inventory.groupby("deliverable_category")
    .agg(
        file_count=("file_name", "count"),
        total_size_kb=("size_kb", "sum")
    )
    .reset_index()
    .sort_values("file_count", ascending=False)
)

print("\n" + "=" * 80)
print("PROJECT FILE CATEGORY SUMMARY")
print("=" * 80)

print(category_summary.to_string(index=False))

# ------------------------------------------------
# 11. SAVE AUDIT OUTPUTS
# ------------------------------------------------

inventory_path = os.path.join(
    AUDIT_DIR,
    "ICMB_130_FINAL_PROJECT_FILE_INVENTORY.csv"
)

checklist_path = os.path.join(
    AUDIT_DIR,
    "ICMB_130_FINAL_DELIVERABLE_CHECKLIST.csv"
)

summary_path = os.path.join(
    AUDIT_DIR,
    "ICMB_130_FINAL_DELIVERABLE_AUDIT_SUMMARY.xlsx"
)

inventory.to_csv(inventory_path, index=False)
checklist.to_csv(checklist_path, index=False)

with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:

    checklist.to_excel(
        writer, sheet_name="Deliverable Checklist", index=False
    )

    category_summary.to_excel(
        writer, sheet_name="File Categories", index=False
    )

    notebooks.to_excel(
        writer, sheet_name="Notebooks", index=False
    )

    sql_files.to_excel(
        writer, sheet_name="SQL Files", index=False
    )

    powerbi_files.to_excel(
        writer, sheet_name="Power BI", index=False
    )

    figures.to_excel(
        writer, sheet_name="Figures", index=False
    )

    research_reports.to_excel(
        writer, sheet_name="Research Documents", index=False
    )

# ------------------------------------------------
# 12. FINAL VERIFICATION
# ------------------------------------------------

print("\n" + "=" * 80)
print("CELL 19 OUTPUT VERIFICATION")
print("=" * 80)

for path in [inventory_path, checklist_path, summary_path]:
    print(
        os.path.basename(path),
        "| Exists:", os.path.exists(path),
        "| Size KB:", round(os.path.getsize(path) / 1024, 2)
    )

print("\n" + "=" * 80)
print("CELL 19 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("\nIMPORTANT:")
print("FOUND means a matching file exists in the project directory.")
print("It does not independently verify that the file is complete or correct.")
print("Share the complete output before deciding what remains.")


In [ ]:

# ================================================================
# ICMB-130 SQL ANALYTICS
# STEP 1: INTEGRATED DATASET INSPECTION
# ================================================================

import os
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

INPUT_FILE = os.path.join(
    PROJECT_ROOT,
    "05_outputs",
    "tables",
    "ICMB_130_FINAL_INTEGRATED_FILM_ANALYTICS.csv"
)

SQL_DIR = os.path.join(PROJECT_ROOT, "07_sql")
DATA_DIR = os.path.join(SQL_DIR, "data")
SCHEMA_DIR = os.path.join(SQL_DIR, "schema")
QUERIES_DIR = os.path.join(SQL_DIR, "queries")
OUTPUT_DIR = os.path.join(SQL_DIR, "outputs")

for folder in [SQL_DIR, DATA_DIR, SCHEMA_DIR, QUERIES_DIR, OUTPUT_DIR]:
    os.makedirs(folder, exist_ok=True)

print("=" * 75)
print("ICMB-130 | SQL STEP 1: DATASET INSPECTION")
print("=" * 75)

# ------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Dataset not found: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE)

print("\nDATASET DIMENSIONS")
print("-" * 75)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# ------------------------------------------------
# 2. INSPECT ALL COLUMN NAMES AND DATA TYPES
# ------------------------------------------------

schema_profile = pd.DataFrame({
    "column_name": df.columns,
    "pandas_dtype": df.dtypes.astype(str).values,
    "null_count": df.isnull().sum().values,
    "null_percentage": (
        df.isnull().mean().values * 100
    ).round(2),
    "unique_values": df.nunique(dropna=True).values
})

print("\nCOLUMN SCHEMA")
print("-" * 75)
print(schema_profile.to_string(index=False))

# ------------------------------------------------
# 3. CHECK PRIMARY KEY CANDIDATES
# ------------------------------------------------

print("\nPRIMARY KEY VALIDATION")
print("-" * 75)

if "movie_record_id" in df.columns:

    print(
        "Unique movie_record_id:",
        df["movie_record_id"].nunique()
    )

    print(
        "Duplicate movie_record_id:",
        df["movie_record_id"].duplicated().sum()
    )

    print(
        "Missing movie_record_id:",
        df["movie_record_id"].isna().sum()
    )

else:
    print("movie_record_id column not found.")

# ------------------------------------------------
# 4. CHECK IMPORTANT PROJECT FIELDS
# ------------------------------------------------

important_columns = [
    "movie_record_id",
    "movie_title",
    "year",
    "rank",
    "language",
    "india_gross_cr"
]

available_columns = [
    col for col in important_columns
    if col in df.columns
]

print("\nIMPORTANT PROJECT FIELDS")
print("-" * 75)

print(df[available_columns].head(10).to_string(index=False))

# ------------------------------------------------
# 5. CHECK NUMERIC AND TEXT COLUMNS
# ------------------------------------------------

print("\nNUMERIC COLUMNS")
print("-" * 75)
print(df.select_dtypes(include=np.number).columns.tolist())

print("\nTEXT / OBJECT COLUMNS")
print("-" * 75)
print(df.select_dtypes(include=["object"]).columns.tolist())

# ------------------------------------------------
# 6. CHECK DUPLICATE COMPLETE RECORDS
# ------------------------------------------------

print("\nDUPLICATE RECORD CHECK")
print("-" * 75)
print("Duplicate complete rows:", df.duplicated().sum())

# ------------------------------------------------
# 7. CHECK YEAR AND RANK DISTRIBUTION
# ------------------------------------------------

if "year" in df.columns:

    print("\nYEAR COVERAGE")
    print("-" * 75)

    print("Minimum year:", df["year"].min())
    print("Maximum year:", df["year"].max())
    print("Unique years:", df["year"].nunique())

if "rank" in df.columns:

    print("\nRANK DISTRIBUTION")
    print("-" * 75)

    print(df["rank"].value_counts(dropna=False).sort_index())

# ------------------------------------------------
# 8. SAVE SCHEMA PROFILE
# ------------------------------------------------

schema_path = os.path.join(
    SCHEMA_DIR,
    "ICMB_130_MYSQL_SCHEMA_PROFILE.csv"
)

schema_profile.to_csv(schema_path, index=False)

# ------------------------------------------------
# 9. SAVE SQL-READY DATASET COPY
# ------------------------------------------------

sql_csv_path = os.path.join(
    DATA_DIR,
    "ICMB_130_FINAL_INTEGRATED_FILM_ANALYTICS.csv"
)

df.to_csv(sql_csv_path, index=False, encoding="utf-8-sig")

# ------------------------------------------------
# 10. FINAL VERIFICATION
# ------------------------------------------------

print("\n" + "=" * 75)
print("STEP 1 OUTPUT VERIFICATION")
print("=" * 75)

for path in [schema_path, sql_csv_path]:

    print(
        os.path.basename(path),
        "| Exists:", os.path.exists(path),
        "| Size KB:", round(os.path.getsize(path) / 1024, 2)
    )

print("\nSQL PROJECT FOLDERS CREATED:")
for folder in [DATA_DIR, SCHEMA_DIR, QUERIES_DIR, OUTPUT_DIR]:
    print(folder)

assert len(df) == 130, "Expected 130 benchmark films."
assert df.shape[1] == 59, "Expected 59 integrated dataset columns."

if "movie_record_id" in df.columns:
    assert df["movie_record_id"].is_unique, (
        "movie_record_id is not unique. Resolve before defining primary key."
    )
    assert df["movie_record_id"].notna().all(), (
        "movie_record_id contains missing values."
    )

print("\nSTEP 1 COMPLETED SUCCESSFULLY")
print("Dataset is ready for MySQL schema design.")


In [ ]:

# ================================================================
# ICMB-130 SQL STEP 2
# PREPARE MYSQL-READY DATASET AND SCHEMA
# ================================================================

import os
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

SQL_DIR = os.path.join(PROJECT_ROOT, "07_sql")
DATA_DIR = os.path.join(SQL_DIR, "data")
SCHEMA_DIR = os.path.join(SQL_DIR, "schema")

INPUT_FILE = os.path.join(
    DATA_DIR,
    "ICMB_130_FINAL_INTEGRATED_FILM_ANALYTICS.csv"
)

df = pd.read_csv(INPUT_FILE)

print("=" * 75)
print("ICMB-130 | SQL STEP 2: MYSQL SCHEMA PREPARATION")
print("=" * 75)

# ------------------------------------------------
# 1. REMOVE ONLY COMPLETELY EMPTY PROVENANCE COLUMNS
# ------------------------------------------------

empty_columns = [
    col for col in df.columns
    if df[col].isna().all()
]

print("\nCOMPLETELY EMPTY COLUMNS")
print("-" * 75)
print(empty_columns)

df = df.drop(columns=empty_columns)

# ------------------------------------------------
# 2. VALIDATE RELEASE DATE
# ------------------------------------------------

if "release_date" in df.columns:

    original_dates = df["release_date"].copy()

    parsed_dates = pd.to_datetime(
        original_dates,
        errors="coerce"
    )

    invalid_dates = int(parsed_dates.isna().sum())

    print("\nRELEASE DATE VALIDATION")
    print("-" * 75)
    print("Total records:", len(df))
    print("Valid dates:", int(parsed_dates.notna().sum()))
    print("Invalid dates:", invalid_dates)

    if invalid_dates > 0:
        print("\nInvalid date examples:")
        print(
            original_dates[parsed_dates.isna()]
            .head(10)
            .to_string(index=False)
        )

    # Keep invalid or missing dates as blank in the import CSV.
    df["release_date"] = parsed_dates.dt.strftime("%Y-%m-%d")
    df["release_date"] = df["release_date"].fillna("")

# ------------------------------------------------
# 3. CHECK NUMERIC FIELDS
# ------------------------------------------------

numeric_columns = df.select_dtypes(
    include=["number"]
).columns.tolist()

numeric_check = []

for col in numeric_columns:

    numeric_values = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    numeric_check.append({
        "column_name": col,
        "minimum": numeric_values.min(),
        "maximum": numeric_values.max(),
        "missing_after_conversion": int(
            numeric_values.isna().sum()
        )
    })

numeric_profile = pd.DataFrame(numeric_check)

print("\nNUMERIC FIELD VALIDATION")
print("-" * 75)
print(numeric_profile.to_string(index=False))

# ------------------------------------------------
# 4. DEFINE MYSQL DATA TYPES
# ------------------------------------------------

mysql_schema = []

for col in df.columns:

    if col == "movie_record_id":
        mysql_type = "VARCHAR(20) PRIMARY KEY"

    elif col in ["movie_title", "original_title"]:
        mysql_type = "VARCHAR(255)"

    elif col in [
        "language",
        "original_language",
        "period"
    ]:
        mysql_type = "VARCHAR(100)"

    elif col == "release_date":
        mysql_type = "DATE"

    elif col in ["genres", "production_countries"]:
        mysql_type = "TEXT"

    elif col in ["year", "rank", "runtime", "tmdb_id"]:
        mysql_type = "INT"

    elif col in [
        "india_gross_cr",
        "log_india_gross"
    ]:
        mysql_type = "DECIMAL(18,4)"

    elif (
        col.startswith("theme_chunks_")
        or col == "total_curated_theme_chunks"
        or col.endswith("_comments")
        or col in [
            "sentiment_comment_count",
            "sentiment_total_components",
            "sentiment_count_difference"
        ]
    ):
        mysql_type = "INT"

    elif (
        col.startswith("theme_share_")
        or col.endswith("_share_classifiable_pct")
        or col.endswith("_share_total_pct")
    ):
        mysql_type = "DECIMAL(10,4)"

    else:
        mysql_type = "DECIMAL(18,4)"

    mysql_schema.append({
        "column_name": col,
        "mysql_data_type": mysql_type
    })

schema_df = pd.DataFrame(mysql_schema)

print("\nPROPOSED MYSQL SCHEMA")
print("-" * 75)
print(schema_df.to_string(index=False))

# ------------------------------------------------
# 5. VERIFY PRIMARY KEY AND ROW COUNT
# ------------------------------------------------

assert len(df) == 130
assert df["movie_record_id"].is_unique
assert df["movie_record_id"].notna().all()

# ------------------------------------------------
# 6. EXPORT MYSQL-READY CSV
# ------------------------------------------------

clean_csv = os.path.join(
    DATA_DIR,
    "ICMB_130_MYSQL_READY_FILM_ANALYTICS.csv"
)

df.to_csv(
    clean_csv,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------
# 7. EXPORT MYSQL SCHEMA PROFILE
# ------------------------------------------------

schema_csv = os.path.join(
    SCHEMA_DIR,
    "ICMB_130_MYSQL_COLUMN_SCHEMA.csv"
)

schema_df.to_csv(schema_csv, index=False)

numeric_csv = os.path.join(
    SCHEMA_DIR,
    "ICMB_130_MYSQL_NUMERIC_VALIDATION.csv"
)

numeric_profile.to_csv(numeric_csv, index=False)

# ------------------------------------------------
# 8. FINAL SUMMARY
# ------------------------------------------------

print("\n" + "=" * 75)
print("SQL STEP 2 OUTPUT VERIFICATION")
print("=" * 75)

print("Final rows:", len(df))
print("Final columns:", len(df.columns))
print("Removed empty columns:", empty_columns)
print("Primary key unique:", df["movie_record_id"].is_unique)

for path in [clean_csv, schema_csv, numeric_csv]:
    print(
        os.path.basename(path),
        "| Exists:", os.path.exists(path),
        "| Size KB:", round(os.path.getsize(path) / 1024, 2)
    )

print("\nSQL STEP 2 COMPLETED")


In [ ]:

# ================================================================
# ICMB-130: CSV ENCODING REPAIR FOR MYSQL WORKBENCH
# ================================================================

import os
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

INPUT_FILE = os.path.join(
    PROJECT_ROOT,
    "07_sql",
    "data",
    "ICMB_130_MYSQL_READY_FILM_ANALYTICS.csv"
)

OUTPUT_FILE = os.path.join(
    PROJECT_ROOT,
    "07_sql",
    "data",
    "ICMB_130_MYSQL_READY_FILM_ANALYTICS_UTF8.csv"
)

# Read the original CSV
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")

# Clean problematic control characters in text columns only.
# Preserve tabs, newlines and normal Unicode characters.
def clean_text(value):
    if not isinstance(value, str):
        return value

    return "".join(
        char for char in value
        if char in "\t\n\r" or ord(char) >= 32
    )

text_columns = df.select_dtypes(
    include=["object"]
).columns

for col in text_columns:
    df[col] = df[col].map(clean_text)

# Export plain UTF-8 without BOM
df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8",
    lineterminator="\n"
)

# Verify the output
check_df = pd.read_csv(OUTPUT_FILE, encoding="utf-8")

print("=" * 70)
print("MYSQL CSV ENCODING REPAIR")
print("=" * 70)

print("Original shape:", df.shape)
print("Reopened shape:", check_df.shape)
print("Column names match:", list(df.columns) == list(check_df.columns))
print("Rows preserved:", len(check_df) == 130)
print("Output file exists:", os.path.exists(OUTPUT_FILE))
print("Output size KB:", round(os.path.getsize(OUTPUT_FILE) / 1024, 2))
print("Output path:", OUTPUT_FILE)

assert check_df.shape == df.shape
assert check_df["movie_record_id"].is_unique

print("\nUTF-8 CSV READY FOR MYSQL IMPORT")


In [ ]:

# ================================================================
# ICMB-130 — SAVE COMMERCIAL PERFORMANCE SQL QUERIES
# ================================================================

import os

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

QUERY_DIR = os.path.join(
    PROJECT_ROOT, "07_sql", "queries"
)

os.makedirs(QUERY_DIR, exist_ok=True)

sql_script = r"""
-- ============================================================
-- ICMB-130: INDIAN CINEMA MULTIMODAL BENCHMARK
-- SQL ANALYTICS: COMMERCIAL PERFORMANCE
-- Database: icmb130_db
-- Table: film_analytics
-- ============================================================

USE icmb130_db;

-- QUERY 1: Annual Box Office Performance

SELECT
    year,
    COUNT(*) AS total_films,
    ROUND(AVG(india_gross_cr), 2) AS average_india_gross_cr,
    ROUND(MIN(india_gross_cr), 2) AS minimum_india_gross_cr,
    ROUND(MAX(india_gross_cr), 2) AS maximum_india_gross_cr,
    ROUND(SUM(india_gross_cr), 2) AS total_india_gross_cr
FROM film_analytics
GROUP BY year
ORDER BY year;


-- QUERY 2: Top 10 Highest-Grossing Films

SELECT
    movie_title,
    year,
    language,
    india_gross_cr
FROM film_analytics
ORDER BY india_gross_cr DESC
LIMIT 10;


-- QUERY 3: Period-Wise Commercial Performance

WITH ranked AS (
    SELECT
        period,
        india_gross_cr,
        ROW_NUMBER() OVER (
            PARTITION BY period
            ORDER BY india_gross_cr
        ) AS rn,
        COUNT(*) OVER (
            PARTITION BY period
        ) AS n
    FROM film_analytics
)
SELECT
    period,
    COUNT(*) AS total_films,
    ROUND(AVG(ranked.india_gross_cr), 2) AS average_gross_cr,
    ROUND(AVG(
        CASE
            WHEN rn IN (
                FLOOR((n + 1) / 2),
                FLOOR((n + 2) / 2)
            )
            THEN ranked.india_gross_cr
        END
    ), 2) AS median_gross_cr,
    ROUND(SUM(ranked.india_gross_cr), 2) AS total_gross_cr
FROM ranked
GROUP BY period
ORDER BY FIELD(period, 'Early', 'Middle', 'Recent');


-- QUERY 4: Language-Wise Commercial Performance

SELECT
    language,
    COUNT(*) AS total_films,
    ROUND(AVG(india_gross_cr), 2) AS average_gross_cr,
    ROUND(MAX(india_gross_cr), 2) AS highest_gross_cr,
    ROUND(SUM(india_gross_cr), 2) AS total_gross_cr
FROM film_analytics
GROUP BY language
ORDER BY average_gross_cr DESC;


-- QUERY 5: Annual Number-One Films

SELECT
    year,
    movie_title,
    language,
    india_gross_cr
FROM film_analytics
WHERE `rank` = 1
ORDER BY year;


-- QUERY 6: Gross Distribution by Rank

SELECT
    `rank`,
    COUNT(*) AS total_films,
    ROUND(AVG(india_gross_cr), 2) AS average_gross_cr,
    ROUND(MIN(india_gross_cr), 2) AS minimum_gross_cr,
    ROUND(MAX(india_gross_cr), 2) AS maximum_gross_cr
FROM film_analytics
GROUP BY `rank`
ORDER BY `rank`;
"""

output_file = os.path.join(
    QUERY_DIR,
    "01_ICMB_130_COMMERCIAL_PERFORMANCE.sql"
)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(sql_script.strip() + "\n")

print("SQL script saved:", output_file)
print("File exists:", os.path.exists(output_file))
print("File size (KB):", round(os.path.getsize(output_file) / 1024, 2))
print("\nCOMMERCIAL PERFORMANCE SQL SCRIPT SAVED SUCCESSFULLY")


In [ ]:

from pathlib import Path

# ============================================================
# ICMB-130: DIALOGUE THEME ANALYSIS
# Dataset: 130 Indian films (2000–2025)
# Database: icmb130_db
# Table: film_analytics
# ============================================================

sql_script = r"""
-- ============================================================
-- ICMB-130: DIALOGUE THEME ANALYSIS
-- Dataset: 130 Indian films (2000–2025)
-- Database: icmb130_db
-- Table: film_analytics
-- MySQL 8.0+
-- ============================================================


-- ============================================================
-- QUERY 1: OVERALL PREVALENCE OF 12 CURATED THEMES
-- ============================================================

WITH theme_data AS (

    SELECT 'Love and romantic relationships' AS theme,
           SUM(theme_chunks_love_and_romantic_relationships) AS chunks
    FROM film_analytics

    UNION ALL
    SELECT 'Marriage and weddings',
           SUM(theme_chunks_marriage_and_weddings)
    FROM film_analytics

    UNION ALL
    SELECT 'Family and parent-child relationships',
           SUM(theme_chunks_family_and_parent_child_relationships)
    FROM film_analytics

    UNION ALL
    SELECT 'Sports and competition',
           SUM(theme_chunks_sports_and_competition)
    FROM film_analytics

    UNION ALL
    SELECT 'Money, wealth and economic transactions',
           SUM(theme_chunks_money_wealth_and_economic_transactions)
    FROM film_analytics

    UNION ALL
    SELECT 'Police and law enforcement',
           SUM(theme_chunks_police_and_law_enforcement)
    FROM film_analytics

    UNION ALL
    SELECT 'Healthcare and medicine',
           SUM(theme_chunks_healthcare_and_medicine)
    FROM film_analytics

    UNION ALL
    SELECT 'Politics, governance and corruption',
           SUM(theme_chunks_politics_governance_and_corruption)
    FROM film_analytics

    UNION ALL
    SELECT 'National security and terrorism',
           SUM(theme_chunks_national_security_and_terrorism)
    FROM film_analytics

    UNION ALL
    SELECT 'Courts and legal proceedings',
           SUM(theme_chunks_courts_and_legal_proceedings)
    FROM film_analytics

    UNION ALL
    SELECT 'Education and academic institutions',
           SUM(theme_chunks_education_and_academic_institutions)
    FROM film_analytics

    UNION ALL
    SELECT 'Religion and devotional beliefs',
           SUM(theme_chunks_religion_and_devotional_beliefs)
    FROM film_analytics
)

SELECT
    theme,
    chunks AS total_theme_chunks,
    ROUND(
        chunks * 100.0 /
        (SELECT SUM(total_curated_theme_chunks)
         FROM film_analytics), 2
    ) AS overall_share_pct
FROM theme_data
ORDER BY chunks DESC;


-- ============================================================
-- QUERY 2: PERIOD-WISE AVERAGE FILM-LEVEL THEME SHARES
-- ============================================================

WITH period_data AS (
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END AS period,

        theme_share_love_and_romantic_relationships AS love_share,
        theme_share_marriage_and_weddings AS marriage_share,
        theme_share_family_and_parent_child_relationships AS family_share,
        theme_share_sports_and_competition AS sports_share,
        theme_share_money_wealth_and_economic_transactions AS money_share,
        theme_share_police_and_law_enforcement AS police_share,
        theme_share_healthcare_and_medicine AS healthcare_share,
        theme_share_politics_governance_and_corruption AS politics_share,
        theme_share_national_security_and_terrorism AS security_share,
        theme_share_courts_and_legal_proceedings AS courts_share,
        theme_share_education_and_academic_institutions AS education_share,
        theme_share_religion_and_devotional_beliefs AS religion_share

    FROM film_analytics
)

SELECT
    period,
    COUNT(*) AS total_films,

    ROUND(AVG(love_share), 2) AS love_avg_share_pct,
    ROUND(AVG(marriage_share), 2) AS marriage_avg_share_pct,
    ROUND(AVG(family_share), 2) AS family_avg_share_pct,
    ROUND(AVG(sports_share), 2) AS sports_avg_share_pct,
    ROUND(AVG(money_share), 2) AS money_avg_share_pct,
    ROUND(AVG(police_share), 2) AS police_avg_share_pct,
    ROUND(AVG(healthcare_share), 2) AS healthcare_avg_share_pct,
    ROUND(AVG(politics_share), 2) AS politics_avg_share_pct,
    ROUND(AVG(security_share), 2) AS security_avg_share_pct,
    ROUND(AVG(courts_share), 2) AS courts_avg_share_pct,
    ROUND(AVG(education_share), 2) AS education_avg_share_pct,
    ROUND(AVG(religion_share), 2) AS religion_avg_share_pct

FROM period_data
GROUP BY period
ORDER BY
    CASE period
        WHEN 'Early (2000-2008)' THEN 1
        WHEN 'Middle (2009-2017)' THEN 2
        WHEN 'Recent (2018-2025)' THEN 3
    END;


-- ============================================================
-- QUERY 3: PERIOD-WISE POOLED THEME PREVALENCE
-- ============================================================

WITH theme_data AS (

    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END AS period,
        'Love and romantic relationships' AS theme,
        theme_chunks_love_and_romantic_relationships AS chunks
    FROM film_analytics

    UNION ALL
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END,
        'Marriage and weddings',
        theme_chunks_marriage_and_weddings
    FROM film_analytics

    UNION ALL
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END,
        'Family and parent-child relationships',
        theme_chunks_family_and_parent_child_relationships
    FROM film_analytics

    UNION ALL
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END,
        'Sports and competition',
        theme_chunks_sports_and_competition
    FROM film_analytics

    UNION ALL
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END,
        'Money, wealth and economic transactions',
        theme_chunks_money_wealth_and_economic_transactions
    FROM film_analytics

    UNION ALL
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END,
        'Police and law enforcement',
        theme_chunks_police_and_law_enforcement
    FROM film_analytics

    UNION ALL
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END,
        'Healthcare and medicine',
        theme_chunks_healthcare_and_medicine
    FROM film_analytics

    UNION ALL
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END,
        'Politics, governance and corruption',
        theme_chunks_politics_governance_and_corruption
    FROM film_analytics

    UNION ALL
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END,
        'National security and terrorism',
        theme_chunks_national_security_and_terrorism
    FROM film_analytics

    UNION ALL
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END,
        'Courts and legal proceedings',
        theme_chunks_courts_and_legal_proceedings
    FROM film_analytics

    UNION ALL
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END,
        'Education and academic institutions',
        theme_chunks_education_and_academic_institutions
    FROM film_analytics

    UNION ALL
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END,
        'Religion and devotional beliefs',
        theme_chunks_religion_and_devotional_beliefs
    FROM film_analytics
),

period_totals AS (
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
        END AS period,
        SUM(total_curated_theme_chunks) AS total_chunks
    FROM film_analytics
    GROUP BY period
),

theme_totals AS (
    SELECT
        period,
        theme,
        SUM(chunks) AS theme_chunks
    FROM theme_data
    GROUP BY period, theme
)

SELECT
    t.period,
    t.theme,
    t.theme_chunks,
    p.total_chunks,
    ROUND(t.theme_chunks * 100.0 / NULLIF(p.total_chunks, 0), 2)
        AS pooled_theme_share_pct
FROM theme_totals t
JOIN period_totals p
    ON t.period = p.period
ORDER BY
    CASE t.period
        WHEN 'Early (2000-2008)' THEN 1
        WHEN 'Middle (2009-2017)' THEN 2
        WHEN 'Recent (2018-2025)' THEN 3
    END,
    pooled_theme_share_pct DESC;


-- ============================================================
-- QUERY 4: ANNUAL DIALOGUE THEME TRENDS (2000-2025)
-- ============================================================

WITH theme_data AS (

    SELECT year, 'Love and romantic relationships' AS theme,
           theme_chunks_love_and_romantic_relationships AS chunks
    FROM film_analytics

    UNION ALL
    SELECT year, 'Marriage and weddings',
           theme_chunks_marriage_and_weddings
    FROM film_analytics

    UNION ALL
    SELECT year, 'Family and parent-child relationships',
           theme_chunks_family_and_parent_child_relationships
    FROM film_analytics

    UNION ALL
    SELECT year, 'Sports and competition',
           theme_chunks_sports_and_competition
    FROM film_analytics

    UNION ALL
    SELECT year, 'Money, wealth and economic transactions',
           theme_chunks_money_wealth_and_economic_transactions
    FROM film_analytics

    UNION ALL
    SELECT year, 'Police and law enforcement',
           theme_chunks_police_and_law_enforcement
    FROM film_analytics

    UNION ALL
    SELECT year, 'Healthcare and medicine',
           theme_chunks_healthcare_and_medicine
    FROM film_analytics

    UNION ALL
    SELECT year, 'Politics, governance and corruption',
           theme_chunks_politics_governance_and_corruption
    FROM film_analytics

    UNION ALL
    SELECT year, 'National security and terrorism',
           theme_chunks_national_security_and_terrorism
    FROM film_analytics

    UNION ALL
    SELECT year, 'Courts and legal proceedings',
           theme_chunks_courts_and_legal_proceedings
    FROM film_analytics

    UNION ALL
    SELECT year, 'Education and academic institutions',
           theme_chunks_education_and_academic_institutions
    FROM film_analytics

    UNION ALL
    SELECT year, 'Religion and devotional beliefs',
           theme_chunks_religion_and_devotional_beliefs
    FROM film_analytics
),

annual_totals AS (
    SELECT
        year,
        SUM(total_curated_theme_chunks) AS total_chunks
    FROM film_analytics
    GROUP BY year
),

annual_theme_totals AS (
    SELECT
        year,
        theme,
        SUM(chunks) AS theme_chunks
    FROM theme_data
    GROUP BY year, theme
)

SELECT
    a.year,
    a.theme,
    a.theme_chunks,
    t.total_chunks,
    ROUND(a.theme_chunks * 100.0 / NULLIF(t.total_chunks, 0), 2)
        AS annual_theme_share_pct
FROM annual_theme_totals a
JOIN annual_totals t
    ON a.year = t.year
ORDER BY a.year, annual_theme_share_pct DESC;


-- ============================================================
-- QUERY 5: DOMINANT DIALOGUE THEME PER FILM
-- ============================================================

WITH theme_data AS (

    SELECT movie_record_id, year, movie_title,
           'Love and romantic relationships' AS theme,
           theme_chunks_love_and_romantic_relationships AS theme_chunks
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, year, movie_title,
           'Marriage and weddings',
           theme_chunks_marriage_and_weddings
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, year, movie_title,
           'Family and parent-child relationships',
           theme_chunks_family_and_parent_child_relationships
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, year, movie_title,
           'Sports and competition',
           theme_chunks_sports_and_competition
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, year, movie_title,
           'Money, wealth and economic transactions',
           theme_chunks_money_wealth_and_economic_transactions
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, year, movie_title,
           'Police and law enforcement',
           theme_chunks_police_and_law_enforcement
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, year, movie_title,
           'Healthcare and medicine',
           theme_chunks_healthcare_and_medicine
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, year, movie_title,
           'Politics, governance and corruption',
           theme_chunks_politics_governance_and_corruption
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, year, movie_title,
           'National security and terrorism',
           theme_chunks_national_security_and_terrorism
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, year, movie_title,
           'Courts and legal proceedings',
           theme_chunks_courts_and_legal_proceedings
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, year, movie_title,
           'Education and academic institutions',
           theme_chunks_education_and_academic_institutions
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, year, movie_title,
           'Religion and devotional beliefs',
           theme_chunks_religion_and_devotional_beliefs
    FROM film_analytics
),

ranked_themes AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY movie_record_id
               ORDER BY theme_chunks DESC, theme ASC
           ) AS theme_rank
    FROM theme_data
)

SELECT
    movie_record_id,
    year,
    movie_title,
    theme AS dominant_theme,
    theme_chunks
FROM ranked_themes
WHERE theme_rank = 1
ORDER BY year, movie_title;


-- ============================================================
-- QUERY 6: DISTRIBUTION OF DOMINANT THEMES ACROSS FILMS
-- ============================================================

WITH theme_data AS (

    SELECT movie_record_id,
           'Love and romantic relationships' AS theme,
           theme_chunks_love_and_romantic_relationships AS theme_chunks
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, 'Marriage and weddings',
           theme_chunks_marriage_and_weddings
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, 'Family and parent-child relationships',
           theme_chunks_family_and_parent_child_relationships
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, 'Sports and competition',
           theme_chunks_sports_and_competition
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, 'Money, wealth and economic transactions',
           theme_chunks_money_wealth_and_economic_transactions
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, 'Police and law enforcement',
           theme_chunks_police_and_law_enforcement
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, 'Healthcare and medicine',
           theme_chunks_healthcare_and_medicine
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, 'Politics, governance and corruption',
           theme_chunks_politics_governance_and_corruption
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, 'National security and terrorism',
           theme_chunks_national_security_and_terrorism
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, 'Courts and legal proceedings',
           theme_chunks_courts_and_legal_proceedings
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, 'Education and academic institutions',
           theme_chunks_education_and_academic_institutions
    FROM film_analytics

    UNION ALL
    SELECT movie_record_id, 'Religion and devotional beliefs',
           theme_chunks_religion_and_devotional_beliefs
    FROM film_analytics
),

ranked_themes AS (
    SELECT
        movie_record_id,
        theme,
        theme_chunks,
        ROW_NUMBER() OVER (
            PARTITION BY movie_record_id
            ORDER BY theme_chunks DESC, theme ASC
        ) AS theme_rank
    FROM theme_data
),

dominant_themes AS (
    SELECT movie_record_id, theme
    FROM ranked_themes
    WHERE theme_rank = 1
)

SELECT
    theme AS dominant_theme,
    COUNT(*) AS film_count,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM film_analytics), 2
    ) AS percentage_of_films
FROM dominant_themes
GROUP BY theme
ORDER BY film_count DESC, theme;


-- ============================================================
-- QUERY 7: LANGUAGE-WISE DIALOGUE THEME COUNTS
-- ============================================================

SELECT
    language,
    COUNT(*) AS total_films,

    SUM(theme_chunks_love_and_romantic_relationships)
        AS love_chunks,

    SUM(theme_chunks_marriage_and_weddings)
        AS marriage_chunks,

    SUM(theme_chunks_family_and_parent_child_relationships)
        AS family_chunks,

    SUM(theme_chunks_sports_and_competition)
        AS sports_chunks,

    SUM(theme_chunks_money_wealth_and_economic_transactions)
        AS money_chunks,

    SUM(theme_chunks_police_and_law_enforcement)
        AS police_chunks,

    SUM(theme_chunks_healthcare_and_medicine)
        AS healthcare_chunks,

    SUM(theme_chunks_politics_governance_and_corruption)
        AS politics_chunks,

    SUM(theme_chunks_national_security_and_terrorism)
        AS national_security_chunks,

    SUM(theme_chunks_courts_and_legal_proceedings)
        AS courts_chunks,

    SUM(theme_chunks_education_and_academic_institutions)
        AS education_chunks,

    SUM(theme_chunks_religion_and_devotional_beliefs)
        AS religion_chunks,

    SUM(total_curated_theme_chunks) AS total_theme_chunks

FROM film_analytics
GROUP BY language
ORDER BY total_films DESC;


-- ============================================================
-- QUERY 8: LANGUAGE-WISE NORMALIZED THEME PERCENTAGES
-- ============================================================

SELECT
    language,
    COUNT(*) AS total_films,
    SUM(total_curated_theme_chunks) AS total_theme_chunks,

    ROUND(
        SUM(theme_chunks_love_and_romantic_relationships)
        * 100.0 / NULLIF(SUM(total_curated_theme_chunks), 0), 2
    ) AS love_pct,

    ROUND(
        SUM(theme_chunks_marriage_and_weddings)
        * 100.0 / NULLIF(SUM(total_curated_theme_chunks), 0), 2
    ) AS marriage_pct,

    ROUND(
        SUM(theme_chunks_family_and_parent_child_relationships)
        * 100.0 / NULLIF(SUM(total_curated_theme_chunks), 0), 2
    ) AS family_pct,

    ROUND(
        SUM(theme_chunks_sports_and_competition)
        * 100.0 / NULLIF(SUM(total_curated_theme_chunks), 0), 2
    ) AS sports_pct,

    ROUND(
        SUM(theme_chunks_money_wealth_and_economic_transactions)
        * 100.0 / NULLIF(SUM(total_curated_theme_chunks), 0), 2
    ) AS money_pct,

    ROUND(
        SUM(theme_chunks_police_and_law_enforcement)
        * 100.0 / NULLIF(SUM(total_curated_theme_chunks), 0), 2
    ) AS police_pct,

    ROUND(
        SUM(theme_chunks_healthcare_and_medicine)
        * 100.0 / NULLIF(SUM(total_curated_theme_chunks), 0), 2
    ) AS healthcare_pct,

    ROUND(
        SUM(theme_chunks_politics_governance_and_corruption)
        * 100.0 / NULLIF(SUM(total_curated_theme_chunks), 0), 2
    ) AS politics_pct,

    ROUND(
        SUM(theme_chunks_national_security_and_terrorism)
        * 100.0 / NULLIF(SUM(total_curated_theme_chunks), 0), 2
    ) AS national_security_pct,

    ROUND(
        SUM(theme_chunks_courts_and_legal_proceedings)
        * 100.0 / NULLIF(SUM(total_curated_theme_chunks), 0), 2
    ) AS courts_pct,

    ROUND(
        SUM(theme_chunks_education_and_academic_institutions)
        * 100.0 / NULLIF(SUM(total_curated_theme_chunks), 0), 2
    ) AS education_pct,

    ROUND(
        SUM(theme_chunks_religion_and_devotional_beliefs)
        * 100.0 / NULLIF(SUM(total_curated_theme_chunks), 0), 2
    ) AS religion_pct

FROM film_analytics
GROUP BY language
ORDER BY total_films DESC;


-- ============================================================
-- END OF ICMB-130 DIALOGUE THEME ANALYSIS
-- ============================================================
"""

# ============================================================
# SAVE SQL FILE DIRECTLY TO GOOGLE DRIVE PROJECT
# ============================================================

project_root = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

sql_folder = project_root / "07_sql" / "queries"
sql_folder.mkdir(parents=True, exist_ok=True)

sql_file = (
    sql_folder /
    "02_ICMB_130_DIALOGUE_THEME_ANALYSIS.sql"
)

sql_file.write_text(sql_script.strip() + "\n", encoding="utf-8")

# ============================================================
# VERIFY SAVED FILE
# ============================================================

print("=" * 75)
print("ICMB-130 DIALOGUE THEME SQL SCRIPT — SAVE VERIFICATION")
print("=" * 75)

print("File:", sql_file.name)
print("Exists:", sql_file.exists())
print("Size (KB):", round(sql_file.stat().st_size / 1024, 2))
print("SQL queries:", sql_script.count("-- QUERY"))
print("Encoding: UTF-8")
print("Location:", sql_file)

assert sql_file.exists(), "SQL file was not created."
assert sql_script.count("-- QUERY") == 8, "Expected 8 SQL queries."

print("\nSUCCESS: All 8 dialogue-theme queries saved.")


In [ ]:

from pathlib import Path

# ============================================================
# ICMB-130: AUDIENCE SENTIMENT SQL ANALYSIS
# Generate complete MySQL SQL script
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

SQL_DIR = PROJECT_DIR / "07_sql" / "queries"
SQL_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = (
    SQL_DIR / "03_ICMB_130_AUDIENCE_SENTIMENT_ANALYSIS.sql"
)

sql_script = r"""
-- ============================================================
-- PROJECT:
-- ICMB-130 Indian Cinema Multimodal Benchmark Dataset
--
-- MODULE:
-- Audience Sentiment Analysis
--
-- DATABASE:
-- icmb130_db
--
-- TABLE:
-- film_analytics
--
-- SAMPLE:
-- 130 films (2000-2025)
--
-- PURPOSE:
-- Analyse YouTube audience-comment sentiment at overall,
-- temporal and film levels, and perform data-quality checks.
--
-- IMPORTANT:
-- Positive, negative and neutral percentages use
-- classifiable_comments as their denominator.
--
-- Uncertain and emoji-only percentages use
-- sentiment_comment_count as their denominator.
-- ============================================================


USE icmb130_db;


-- ============================================================
-- QUERY 1: OVERALL AUDIENCE SENTIMENT DISTRIBUTION
-- ============================================================

SELECT
    'Uncertain' AS sentiment_category,
    SUM(uncertain_comments) AS comment_count,
    ROUND(
        SUM(uncertain_comments) * 100.0 /
        NULLIF(SUM(sentiment_comment_count), 0), 2
    ) AS percentage
FROM film_analytics

UNION ALL

SELECT
    'Positive',
    SUM(positive_comments),
    ROUND(
        SUM(positive_comments) * 100.0 /
        NULLIF(SUM(sentiment_comment_count), 0), 2
    )
FROM film_analytics

UNION ALL

SELECT
    'Negative',
    SUM(negative_comments),
    ROUND(
        SUM(negative_comments) * 100.0 /
        NULLIF(SUM(sentiment_comment_count), 0), 2
    )
FROM film_analytics

UNION ALL

SELECT
    'Neutral',
    SUM(neutral_comments),
    ROUND(
        SUM(neutral_comments) * 100.0 /
        NULLIF(SUM(sentiment_comment_count), 0), 2
    )
FROM film_analytics

UNION ALL

SELECT
    'Emoji Only',
    SUM(emoji_only_comments),
    ROUND(
        SUM(emoji_only_comments) * 100.0 /
        NULLIF(SUM(sentiment_comment_count), 0), 2
    )
FROM film_analytics;


-- ============================================================
-- QUERY 2: CLASSIFIABLE SENTIMENT DISTRIBUTION
-- ============================================================

SELECT
    'Positive' AS sentiment_category,
    SUM(positive_comments) AS comment_count,
    ROUND(
        SUM(positive_comments) * 100.0 /
        NULLIF(SUM(classifiable_comments), 0), 2
    ) AS percentage
FROM film_analytics

UNION ALL

SELECT
    'Negative',
    SUM(negative_comments),
    ROUND(
        SUM(negative_comments) * 100.0 /
        NULLIF(SUM(classifiable_comments), 0), 2
    )
FROM film_analytics

UNION ALL

SELECT
    'Neutral',
    SUM(neutral_comments),
    ROUND(
        SUM(neutral_comments) * 100.0 /
        NULLIF(SUM(classifiable_comments), 0), 2
    )
FROM film_analytics;


-- ============================================================
-- QUERY 3: PERIOD-WISE AUDIENCE SENTIMENT ANALYSIS
--
-- Early:  2000-2008
-- Middle: 2009-2017
-- Recent: 2018-2025
-- ============================================================

WITH period_data AS (
    SELECT
        CASE
            WHEN year BETWEEN 2000 AND 2008
                THEN 'Early (2000-2008)'
            WHEN year BETWEEN 2009 AND 2017
                THEN 'Middle (2009-2017)'
            WHEN year BETWEEN 2018 AND 2025
                THEN 'Recent (2018-2025)'
        END AS period,

        CASE
            WHEN year BETWEEN 2000 AND 2008 THEN 1
            WHEN year BETWEEN 2009 AND 2017 THEN 2
            WHEN year BETWEEN 2018 AND 2025 THEN 3
        END AS period_order,

        sentiment_comment_count,
        classifiable_comments,
        positive_comments,
        negative_comments,
        neutral_comments,
        uncertain_comments,
        emoji_only_comments

    FROM film_analytics
),

period_summary AS (
    SELECT
        period,
        period_order,
        COUNT(*) AS total_films,

        SUM(sentiment_comment_count) AS total_comments,
        SUM(classifiable_comments) AS classifiable_comments,

        SUM(positive_comments) AS positive_comments,
        SUM(negative_comments) AS negative_comments,
        SUM(neutral_comments) AS neutral_comments,

        SUM(uncertain_comments) AS uncertain_comments,
        SUM(emoji_only_comments) AS emoji_only_comments

    FROM period_data
    GROUP BY period, period_order
)

SELECT
    period,
    total_films,
    total_comments,
    classifiable_comments,

    positive_comments,
    ROUND(
        positive_comments * 100.0 /
        NULLIF(classifiable_comments, 0), 2
    ) AS positive_pct,

    negative_comments,
    ROUND(
        negative_comments * 100.0 /
        NULLIF(classifiable_comments, 0), 2
    ) AS negative_pct,

    neutral_comments,
    ROUND(
        neutral_comments * 100.0 /
        NULLIF(classifiable_comments, 0), 2
    ) AS neutral_pct,

    uncertain_comments,
    ROUND(
        uncertain_comments * 100.0 /
        NULLIF(total_comments, 0), 2
    ) AS uncertain_pct,

    emoji_only_comments,
    ROUND(
        emoji_only_comments * 100.0 /
        NULLIF(total_comments, 0), 2
    ) AS emoji_only_pct

FROM period_summary
ORDER BY period_order;


-- ============================================================
-- QUERY 4: FILM-WISE AUDIENCE SENTIMENT DISTRIBUTION
-- ============================================================

SELECT
    movie_record_id,
    year,
    movie_title,
    language,

    sentiment_comment_count AS total_comments,
    classifiable_comments,

    positive_comments,
    ROUND(
        positive_comments * 100.0 /
        NULLIF(classifiable_comments, 0), 2
    ) AS positive_pct,

    negative_comments,
    ROUND(
        negative_comments * 100.0 /
        NULLIF(classifiable_comments, 0), 2
    ) AS negative_pct,

    neutral_comments,
    ROUND(
        neutral_comments * 100.0 /
        NULLIF(classifiable_comments, 0), 2
    ) AS neutral_pct,

    uncertain_comments,
    ROUND(
        uncertain_comments * 100.0 /
        NULLIF(sentiment_comment_count, 0), 2
    ) AS uncertain_pct,

    emoji_only_comments,
    ROUND(
        emoji_only_comments * 100.0 /
        NULLIF(sentiment_comment_count, 0), 2
    ) AS emoji_only_pct

FROM film_analytics
ORDER BY year, `rank`;


-- ============================================================
-- QUERY 5: AUDIENCE COMMENT COVERAGE ANALYSIS
-- ============================================================

SELECT
    movie_record_id,
    year,
    movie_title,
    language,

    sentiment_comment_count AS total_comments,
    classifiable_comments,

    CASE
        WHEN sentiment_comment_count < 50
            THEN 'Low Coverage'
        WHEN sentiment_comment_count < 100
            THEN 'Moderate Coverage'
        ELSE 'High Coverage'
    END AS coverage_category,

    ROUND(
        classifiable_comments * 100.0 /
        NULLIF(sentiment_comment_count, 0), 2
    ) AS classifiable_coverage_pct

FROM film_analytics
ORDER BY total_comments ASC, year ASC;


-- ============================================================
-- QUERY 6: SENTIMENT DATA-QUALITY RECONCILIATION
--
-- Identify films where recorded comment totals differ
-- from the sum of the five sentiment categories.
-- ============================================================

SELECT
    movie_record_id,
    year,
    movie_title,

    sentiment_comment_count AS recorded_total,

    positive_comments,
    negative_comments,
    neutral_comments,
    uncertain_comments,
    emoji_only_comments,

    (
        positive_comments
        + negative_comments
        + neutral_comments
        + uncertain_comments
        + emoji_only_comments
    ) AS calculated_total,

    sentiment_count_difference AS stored_difference,

    (
        sentiment_comment_count
        -
        (
            positive_comments
            + negative_comments
            + neutral_comments
            + uncertain_comments
            + emoji_only_comments
        )
    ) AS calculated_difference,

    CASE
        WHEN sentiment_comment_count =
             (
                positive_comments
                + negative_comments
                + neutral_comments
                + uncertain_comments
                + emoji_only_comments
             )
        THEN 'MATCH'
        ELSE 'MISMATCH'
    END AS reconciliation_status

FROM film_analytics

WHERE sentiment_comment_count <>
      (
        positive_comments
        + negative_comments
        + neutral_comments
        + uncertain_comments
        + emoji_only_comments
      )

ORDER BY year, `rank`;


-- ============================================================
-- QUERY 7: FINAL SENTIMENT RECONCILIATION SUMMARY
-- ============================================================

SELECT
    COUNT(*) AS total_films,

    SUM(
        CASE
            WHEN sentiment_comment_count =
                 positive_comments
                 + negative_comments
                 + neutral_comments
                 + uncertain_comments
                 + emoji_only_comments
            THEN 1
            ELSE 0
        END
    ) AS matched_films,

    SUM(
        CASE
            WHEN sentiment_comment_count <>
                 positive_comments
                 + negative_comments
                 + neutral_comments
                 + uncertain_comments
                 + emoji_only_comments
            THEN 1
            ELSE 0
        END
    ) AS mismatched_films,

    ROUND(
        SUM(
            CASE
                WHEN sentiment_comment_count =
                     positive_comments
                     + negative_comments
                     + neutral_comments
                     + uncertain_comments
                     + emoji_only_comments
                THEN 1
                ELSE 0
            END
        ) * 100.0 / NULLIF(COUNT(*), 0),
        2
    ) AS reconciliation_success_pct

FROM film_analytics;


-- ============================================================
-- DATA-QUALITY NOTE
--
-- Final verified reconciliation:
-- Total films:       130
-- Matched films:     129
-- Mismatched films:  1
--
-- Vikram (2022):
-- Recorded total:    100
-- Calculated total:   99
-- Difference:          1
--
-- Original values are intentionally retained.
-- No additional comments or sentiment labels have been
-- artificially added to force reconciliation.
--
-- INTERPRETATION NOTE:
-- YouTube comment sentiment is exploratory. A large
-- proportion of comments is uncertain or language-uncertain.
-- Results should not be interpreted as representative
-- measures of all film audiences.
-- ============================================================

"""

# Save SQL script
OUTPUT_FILE.write_text(sql_script.strip() + "\n", encoding="utf-8")

# ============================================================
# VERIFY SAVED FILE
# ============================================================

print("=" * 70)
print("ICMB-130 AUDIENCE SENTIMENT SQL SCRIPT")
print("=" * 70)

print("File name:", OUTPUT_FILE.name)
print("File path:", OUTPUT_FILE)
print("File exists:", OUTPUT_FILE.exists())
print("File size (KB):", round(OUTPUT_FILE.stat().st_size / 1024, 2))

saved_text = OUTPUT_FILE.read_text(encoding="utf-8")

query_count = saved_text.count("-- QUERY ")

print("Number of query sections:", query_count)
print("Encoding: UTF-8")
print("Contains Vikram data-quality note:",
      "Vikram (2022)" in saved_text)

assert OUTPUT_FILE.exists()
assert query_count == 7
assert "Vikram (2022)" in saved_text

print("\nSUCCESS: SQL script saved and verified.")


In [ ]:

from pathlib import Path

# Project SQL output directory
sql_dir = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/07_sql/queries"
)
sql_dir.mkdir(parents=True, exist_ok=True)

sql_script = r"""
-- ================================================================
-- ICMB-130: INTEGRATED RESEARCH ANALYSIS
-- Project: Indian Cinema Multimodal Benchmark Dataset (2000-2025)
-- Database: icmb130_db
-- Table: film_analytics
-- Module: 04
--
-- This module integrates:
-- 1. Commercial performance and dominant dialogue themes
-- 2. Commercial performance and audience sentiment
-- 3. Period-wise integrated trends
-- 4. Annual integrated trends
-- 5. Film-level integrated research dataset
--
-- Important:
-- - India Gross values are nominal, not inflation-adjusted.
-- - Theme percentages are based on curated theme chunks.
-- - Positive/negative/neutral percentages use classifiable comments.
-- - Sentiment analysis is exploratory and not representative of
--   the entire audience.
-- - Vikram (2022) has one unresolved sentiment-count discrepancy.
-- ================================================================


-- ================================================================
-- QUERY 1: BOX OFFICE VS DOMINANT DIALOGUE THEMES
-- ================================================================

WITH theme_data AS (
    SELECT
        movie_record_id,
        year,
        `rank`,
        movie_title,
        language,
        india_gross_cr,
        total_curated_theme_chunks,

        GREATEST(
            theme_chunks_love_and_romantic_relationships,
            theme_chunks_marriage_and_weddings,
            theme_chunks_family_and_parent_child_relationships,
            theme_chunks_sports_and_competition,
            theme_chunks_money_wealth_and_economic_transactions,
            theme_chunks_police_and_law_enforcement,
            theme_chunks_healthcare_and_medicine,
            theme_chunks_politics_governance_and_corruption,
            theme_chunks_national_security_and_terrorism,
            theme_chunks_courts_and_legal_proceedings,
            theme_chunks_education_and_academic_institutions,
            theme_chunks_religion_and_devotional_beliefs
        ) AS dominant_theme_chunks,

        CASE
            WHEN theme_chunks_love_and_romantic_relationships =
                GREATEST(
                    theme_chunks_love_and_romantic_relationships,
                    theme_chunks_marriage_and_weddings,
                    theme_chunks_family_and_parent_child_relationships,
                    theme_chunks_sports_and_competition,
                    theme_chunks_money_wealth_and_economic_transactions,
                    theme_chunks_police_and_law_enforcement,
                    theme_chunks_healthcare_and_medicine,
                    theme_chunks_politics_governance_and_corruption,
                    theme_chunks_national_security_and_terrorism,
                    theme_chunks_courts_and_legal_proceedings,
                    theme_chunks_education_and_academic_institutions,
                    theme_chunks_religion_and_devotional_beliefs
                ) THEN 'Love and romantic relationships'

            WHEN theme_chunks_marriage_and_weddings =
                GREATEST(
                    theme_chunks_love_and_romantic_relationships,
                    theme_chunks_marriage_and_weddings,
                    theme_chunks_family_and_parent_child_relationships,
                    theme_chunks_sports_and_competition,
                    theme_chunks_money_wealth_and_economic_transactions,
                    theme_chunks_police_and_law_enforcement,
                    theme_chunks_healthcare_and_medicine,
                    theme_chunks_politics_governance_and_corruption,
                    theme_chunks_national_security_and_terrorism,
                    theme_chunks_courts_and_legal_proceedings,
                    theme_chunks_education_and_academic_institutions,
                    theme_chunks_religion_and_devotional_beliefs
                ) THEN 'Marriage and weddings'

            WHEN theme_chunks_family_and_parent_child_relationships =
                GREATEST(
                    theme_chunks_love_and_romantic_relationships,
                    theme_chunks_marriage_and_weddings,
                    theme_chunks_family_and_parent_child_relationships,
                    theme_chunks_sports_and_competition,
                    theme_chunks_money_wealth_and_economic_transactions,
                    theme_chunks_police_and_law_enforcement,
                    theme_chunks_healthcare_and_medicine,
                    theme_chunks_politics_governance_and_corruption,
                    theme_chunks_national_security_and_terrorism,
                    theme_chunks_courts_and_legal_proceedings,
                    theme_chunks_education_and_academic_institutions,
                    theme_chunks_religion_and_devotional_beliefs
                ) THEN 'Family and parent-child relationships'

            WHEN theme_chunks_sports_and_competition =
                GREATEST(
                    theme_chunks_love_and_romantic_relationships,
                    theme_chunks_marriage_and_weddings,
                    theme_chunks_family_and_parent_child_relationships,
                    theme_chunks_sports_and_competition,
                    theme_chunks_money_wealth_and_economic_transactions,
                    theme_chunks_police_and_law_enforcement,
                    theme_chunks_healthcare_and_medicine,
                    theme_chunks_politics_governance_and_corruption,
                    theme_chunks_national_security_and_terrorism,
                    theme_chunks_courts_and_legal_proceedings,
                    theme_chunks_education_and_academic_institutions,
                    theme_chunks_religion_and_devotional_beliefs
                ) THEN 'Sports and competition'

            WHEN theme_chunks_money_wealth_and_economic_transactions =
                GREATEST(
                    theme_chunks_love_and_romantic_relationships,
                    theme_chunks_marriage_and_weddings,
                    theme_chunks_family_and_parent_child_relationships,
                    theme_chunks_sports_and_competition,
                    theme_chunks_money_wealth_and_economic_transactions,
                    theme_chunks_police_and_law_enforcement,
                    theme_chunks_healthcare_and_medicine,
                    theme_chunks_politics_governance_and_corruption,
                    theme_chunks_national_security_and_terrorism,
                    theme_chunks_courts_and_legal_proceedings,
                    theme_chunks_education_and_academic_institutions,
                    theme_chunks_religion_and_devotional_beliefs
                ) THEN 'Money, wealth and economic transactions'

            WHEN theme_chunks_police_and_law_enforcement =
                GREATEST(
                    theme_chunks_love_and_romantic_relationships,
                    theme_chunks_marriage_and_weddings,
                    theme_chunks_family_and_parent_child_relationships,
                    theme_chunks_sports_and_competition,
                    theme_chunks_money_wealth_and_economic_transactions,
                    theme_chunks_police_and_law_enforcement,
                    theme_chunks_healthcare_and_medicine,
                    theme_chunks_politics_governance_and_corruption,
                    theme_chunks_national_security_and_terrorism,
                    theme_chunks_courts_and_legal_proceedings,
                    theme_chunks_education_and_academic_institutions,
                    theme_chunks_religion_and_devotional_beliefs
                ) THEN 'Police and law enforcement'

            WHEN theme_chunks_healthcare_and_medicine =
                GREATEST(
                    theme_chunks_love_and_romantic_relationships,
                    theme_chunks_marriage_and_weddings,
                    theme_chunks_family_and_parent_child_relationships,
                    theme_chunks_sports_and_competition,
                    theme_chunks_money_wealth_and_economic_transactions,
                    theme_chunks_police_and_law_enforcement,
                    theme_chunks_healthcare_and_medicine,
                    theme_chunks_politics_governance_and_corruption,
                    theme_chunks_national_security_and_terrorism,
                    theme_chunks_courts_and_legal_proceedings,
                    theme_chunks_education_and_academic_institutions,
                    theme_chunks_religion_and_devotional_beliefs
                ) THEN 'Healthcare and medicine'

            WHEN theme_chunks_politics_governance_and_corruption =
                GREATEST(
                    theme_chunks_love_and_romantic_relationships,
                    theme_chunks_marriage_and_weddings,
                    theme_chunks_family_and_parent_child_relationships,
                    theme_chunks_sports_and_competition,
                    theme_chunks_money_wealth_and_economic_transactions,
                    theme_chunks_police_and_law_enforcement,
                    theme_chunks_healthcare_and_medicine,
                    theme_chunks_politics_governance_and_corruption,
                    theme_chunks_national_security_and_terrorism,
                    theme_chunks_courts_and_legal_proceedings,
                    theme_chunks_education_and_academic_institutions,
                    theme_chunks_religion_and_devotional_beliefs
                ) THEN 'Politics, governance and corruption'

            WHEN theme_chunks_national_security_and_terrorism =
                GREATEST(
                    theme_chunks_love_and_romantic_relationships,
                    theme_chunks_marriage_and_weddings,
                    theme_chunks_family_and_parent_child_relationships,
                    theme_chunks_sports_and_competition,
                    theme_chunks_money_wealth_and_economic_transactions,
                    theme_chunks_police_and_law_enforcement,
                    theme_chunks_healthcare_and_medicine,
                    theme_chunks_politics_governance_and_corruption,
                    theme_chunks_national_security_and_terrorism,
                    theme_chunks_courts_and_legal_proceedings,
                    theme_chunks_education_and_academic_institutions,
                    theme_chunks_religion_and_devotional_beliefs
                ) THEN 'National security and terrorism'

            WHEN theme_chunks_courts_and_legal_proceedings =
                GREATEST(
                    theme_chunks_love_and_romantic_relationships,
                    theme_chunks_marriage_and_weddings,
                    theme_chunks_family_and_parent_child_relationships,
                    theme_chunks_sports_and_competition,
                    theme_chunks_money_wealth_and_economic_transactions,
                    theme_chunks_police_and_law_enforcement,
                    theme_chunks_healthcare_and_medicine,
                    theme_chunks_politics_governance_and_corruption,
                    theme_chunks_national_security_and_terrorism,
                    theme_chunks_courts_and_legal_proceedings,
                    theme_chunks_education_and_academic_institutions,
                    theme_chunks_religion_and_devotional_beliefs
                ) THEN 'Courts and legal proceedings'

            WHEN theme_chunks_education_and_academic_institutions =
                GREATEST(
                    theme_chunks_love_and_romantic_relationships,
                    theme_chunks_marriage_and_weddings,
                    theme_chunks_family_and_parent_child_relationships,
                    theme_chunks_sports_and_competition,
                    theme_chunks_money_wealth_and_economic_transactions,
                    theme_chunks_police_and_law_enforcement,
                    theme_chunks_healthcare_and_medicine,
                    theme_chunks_politics_governance_and_corruption,
                    theme_chunks_national_security_and_terrorism,
                    theme_chunks_courts_and_legal_proceedings,
                    theme_chunks_education_and_academic_institutions,
                    theme_chunks_religion_and_devotional_beliefs
                ) THEN 'Education and academic institutions'

            WHEN theme_chunks_religion_and_devotional_beliefs =
                GREATEST(
                    theme_chunks_love_and_romantic_relationships,
                    theme_chunks_marriage_and_weddings,
                    theme_chunks_family_and_parent_child_relationships,
                    theme_chunks_sports_and_competition,
                    theme_chunks_money_wealth_and_economic_transactions,
                    theme_chunks_police_and_law_enforcement,
                    theme_chunks_healthcare_and_medicine,
                    theme_chunks_politics_governance_and_corruption,
                    theme_chunks_national_security_and_terrorism,
                    theme_chunks_courts_and_legal_proceedings,
                    theme_chunks_education_and_academic_institutions,
                    theme_chunks_religion_and_devotional_beliefs
                ) THEN 'Religion and devotional beliefs'
        END AS dominant_theme

    FROM film_analytics
)

SELECT
    movie_record_id,
    year,
    `rank`,
    movie_title,
    language,
    india_gross_cr,
    dominant_theme,
    dominant_theme_chunks,
    total_curated_theme_chunks,

    ROUND(
        dominant_theme_chunks * 100.0 /
        NULLIF(total_curated_theme_chunks, 0), 2
    ) AS dominant_theme_share_pct

FROM theme_data
ORDER BY india_gross_cr DESC;


-- ================================================================
-- QUERY 2: BOX OFFICE VS AUDIENCE SENTIMENT
-- ================================================================

SELECT
    movie_record_id,
    year,
    `rank`,
    movie_title,
    language,
    india_gross_cr,

    sentiment_comment_count AS total_comments,
    classifiable_comments,

    positive_comments,
    ROUND(
        positive_comments * 100.0 /
        NULLIF(classifiable_comments, 0), 2
    ) AS positive_pct,

    negative_comments,
    ROUND(
        negative_comments * 100.0 /
        NULLIF(classifiable_comments, 0), 2
    ) AS negative_pct,

    neutral_comments,
    ROUND(
        neutral_comments * 100.0 /
        NULLIF(classifiable_comments, 0), 2
    ) AS neutral_pct,

    uncertain_comments,
    ROUND(
        uncertain_comments * 100.0 /
        NULLIF(sentiment_comment_count, 0), 2
    ) AS uncertain_pct

FROM film_analytics
ORDER BY india_gross_cr DESC;


-- ================================================================
-- QUERY 3: PERIOD-WISE INTEGRATED TRENDS
-- ================================================================

SELECT
    CASE
        WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
        WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
        WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
    END AS period,

    COUNT(*) AS total_films,

    ROUND(AVG(india_gross_cr), 2) AS avg_india_gross_cr,

    ROUND(
        AVG(total_curated_theme_chunks), 2
    ) AS avg_theme_chunks_per_film,

    ROUND(
        AVG(theme_share_love_and_romantic_relationships), 2
    ) AS avg_love_share_pct,

    ROUND(
        AVG(theme_share_sports_and_competition), 2
    ) AS avg_sports_share_pct,

    SUM(sentiment_comment_count) AS total_comments,
    SUM(classifiable_comments) AS classifiable_comments,

    SUM(positive_comments) AS positive_comments,

    ROUND(
        SUM(positive_comments) * 100.0 /
        NULLIF(SUM(classifiable_comments), 0), 2
    ) AS positive_pct,

    SUM(negative_comments) AS negative_comments,

    ROUND(
        SUM(negative_comments) * 100.0 /
        NULLIF(SUM(classifiable_comments), 0), 2
    ) AS negative_pct,

    SUM(neutral_comments) AS neutral_comments,

    ROUND(
        SUM(neutral_comments) * 100.0 /
        NULLIF(SUM(classifiable_comments), 0), 2
    ) AS neutral_pct,

    SUM(uncertain_comments) AS uncertain_comments

FROM film_analytics

GROUP BY
    CASE
        WHEN year BETWEEN 2000 AND 2008 THEN 'Early (2000-2008)'
        WHEN year BETWEEN 2009 AND 2017 THEN 'Middle (2009-2017)'
        WHEN year BETWEEN 2018 AND 2025 THEN 'Recent (2018-2025)'
    END

ORDER BY MIN(year);


-- ================================================================
-- QUERY 4: ANNUAL INTEGRATED TRENDS
-- ================================================================

SELECT
    year,
    COUNT(*) AS total_films,

    ROUND(AVG(india_gross_cr), 2) AS avg_india_gross_cr,
    ROUND(SUM(india_gross_cr), 2) AS total_india_gross_cr,

    ROUND(
        AVG(theme_share_love_and_romantic_relationships), 2
    ) AS avg_love_share_pct,

    ROUND(
        AVG(theme_share_sports_and_competition), 2
    ) AS avg_sports_share_pct,

    ROUND(
        AVG(theme_share_marriage_and_weddings), 2
    ) AS avg_marriage_share_pct,

    SUM(sentiment_comment_count) AS total_comments,
    SUM(classifiable_comments) AS classifiable_comments,

    ROUND(
        SUM(positive_comments) * 100.0 /
        NULLIF(SUM(classifiable_comments), 0), 2
    ) AS positive_pct,

    ROUND(
        SUM(negative_comments) * 100.0 /
        NULLIF(SUM(classifiable_comments), 0), 2
    ) AS negative_pct,

    ROUND(
        SUM(neutral_comments) * 100.0 /
        NULLIF(SUM(classifiable_comments), 0), 2
    ) AS neutral_pct,

    ROUND(
        SUM(uncertain_comments) * 100.0 /
        NULLIF(SUM(sentiment_comment_count), 0), 2
    ) AS uncertain_pct

FROM film_analytics
GROUP BY year
ORDER BY year;


-- ================================================================
-- QUERY 5: FILM-LEVEL INTEGRATED RESEARCH DATASET
-- ================================================================

SELECT
    movie_record_id,
    year,
    `rank`,
    movie_title,
    language,

    india_gross_cr,

    total_curated_theme_chunks,

    CASE
        WHEN total_curated_theme_chunks = 0 THEN NULL
        ELSE ROUND(
            GREATEST(
                theme_chunks_love_and_romantic_relationships,
                theme_chunks_marriage_and_weddings,
                theme_chunks_family_and_parent_child_relationships,
                theme_chunks_sports_and_competition,
                theme_chunks_money_wealth_and_economic_transactions,
                theme_chunks_police_and_law_enforcement,
                theme_chunks_healthcare_and_medicine,
                theme_chunks_politics_governance_and_corruption,
                theme_chunks_national_security_and_terrorism,
                theme_chunks_courts_and_legal_proceedings,
                theme_chunks_education_and_academic_institutions,
                theme_chunks_religion_and_devotional_beliefs
            ) * 100.0 /
            total_curated_theme_chunks, 2
        )
    END AS dominant_theme_share_pct,

    sentiment_comment_count AS total_comments,
    classifiable_comments,

    ROUND(
        classifiable_comments * 100.0 /
        NULLIF(sentiment_comment_count, 0), 2
    ) AS classifiable_coverage_pct,

    ROUND(
        positive_comments * 100.0 /
        NULLIF(classifiable_comments, 0), 2
    ) AS positive_pct,

    ROUND(
        negative_comments * 100.0 /
        NULLIF(classifiable_comments, 0), 2
    ) AS negative_pct,

    ROUND(
        neutral_comments * 100.0 /
        NULLIF(classifiable_comments, 0), 2
    ) AS neutral_pct,

    sentiment_count_difference

FROM film_analytics
ORDER BY year, `rank`;


-- ================================================================
-- END OF MODULE 04
-- ================================================================
"""

# Save SQL file
output_file = sql_dir / "04_ICMB_130_INTEGRATED_RESEARCH_ANALYSIS.sql"

output_file.write_text(sql_script.strip() + "\n", encoding="utf-8")

# Verify saved file
saved_text = output_file.read_text(encoding="utf-8")

query_headings = [
    "QUERY 1: BOX OFFICE VS DOMINANT DIALOGUE THEMES",
    "QUERY 2: BOX OFFICE VS AUDIENCE SENTIMENT",
    "QUERY 3: PERIOD-WISE INTEGRATED TRENDS",
    "QUERY 4: ANNUAL INTEGRATED TRENDS",
    "QUERY 5: FILM-LEVEL INTEGRATED RESEARCH DATASET"
]

print("=" * 75)
print("ICMB-130 MODULE 04 SQL FILE VERIFICATION")
print("=" * 75)
print("File:", output_file.name)
print("Full path:", output_file)
print("Exists:", output_file.exists())
print("File size (KB):", round(output_file.stat().st_size / 1024, 2))
print("Encoding: UTF-8")
print("Total query sections:", sum(h in saved_text for h in query_headings))

for heading in query_headings:
    print(("FOUND: " if heading in saved_text else "MISSING: ") + heading)

print("Total SELECT statements:", saved_text.upper().count("SELECT"))
print("Final file verification:", "PASSED" if all(
    h in saved_text for h in query_headings
) else "FAILED")
print("=" * 75)


Inflation Adjustment Integration and Recalculation.

In [ ]:

import requests
import pandas as pd
from pathlib import Path

# World Bank CPI indicator
# FP.CPI.TOTL = Consumer price index (2010 = 100)
url = (
    "https://api.worldbank.org/v2/country/IND/"
    "indicator/FP.CPI.TOTL"
)

params = {
    "date": "2000:2025",
    "format": "json",
    "per_page": 100
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()

if len(data) < 2 or not data[1]:
    raise ValueError("World Bank API returned no CPI observations.")

records = []

for item in data[1]:
    records.append({
        "Year": int(item["date"]),
        "India_CPI_2010_100": item["value"],
        "Country": item["country"]["value"],
        "Indicator": item["indicator"]["value"],
        "Indicator_Code": item["indicator"]["id"]
    })

cpi_df = pd.DataFrame(records)
cpi_df["India_CPI_2010_100"] = pd.to_numeric(
    cpi_df["India_CPI_2010_100"], errors="coerce"
)

# Keep the requested years and sort chronologically
cpi_df = cpi_df[
    cpi_df["Year"].between(2000, 2025)
].sort_values("Year").reset_index(drop=True)

# Validation
expected_years = set(range(2000, 2026))
actual_years = set(cpi_df["Year"])

print("=" * 65)
print("WORLD BANK INDIA CPI — DATA VALIDATION")
print("=" * 65)

print("Records:", len(cpi_df))
print("Year range:", cpi_df["Year"].min(), "-", cpi_df["Year"].max())
print("Missing CPI values:", cpi_df["India_CPI_2010_100"].isna().sum())
print("Missing years:", sorted(expected_years - actual_years))
print("Duplicate years:", cpi_df["Year"].duplicated().sum())

display(cpi_df)

# Save only if the annual series is complete and valid
if (
    actual_years == expected_years
    and cpi_df["India_CPI_2010_100"].notna().all()
    and cpi_df["Year"].is_unique
):
    output_path = Path("/content/INDIA_CPI_2000_2025_World_Bank_VERIFIED.csv")
    cpi_df.to_csv(output_path, index=False)
    print("\nSUCCESS: Complete CPI series saved to:", output_path)
else:
    print("\nWARNING: CPI validation failed. Do not use this series yet.")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. Load the final integrated dataset
# --------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_FILE = (
    PROJECT_DIR /
    "05_outputs/tables/ICMB_130_FINAL_INTEGRATED_FILM_ANALYTICS.csv"
)

integrated_df = pd.read_csv(INPUT_FILE)

# --------------------------------------------------
# 2. Load the verified World Bank CPI dataset
# --------------------------------------------------

CPI_FILE = Path(
    "/content/INDIA_CPI_2000_2025_World_Bank_VERIFIED.csv"
)

if not CPI_FILE.exists():
    raise FileNotFoundError(
        "Verified CPI file not found. Run the CPI retrieval cell again."
    )

cpi_df = pd.read_csv(CPI_FILE)

# --------------------------------------------------
# 3. Validate input datasets
# --------------------------------------------------

assert integrated_df.shape == (130, 59)
assert integrated_df["movie_record_id"].is_unique
assert integrated_df["year"].notna().all()
assert integrated_df["india_gross_cr"].notna().all()

assert len(cpi_df) == 26
assert cpi_df["Year"].is_unique
assert cpi_df["India_CPI_2010_100"].notna().all()

# --------------------------------------------------
# 4. Merge CPI by release year
# --------------------------------------------------

cpi_lookup = cpi_df[
    ["Year", "India_CPI_2010_100"]
].rename(columns={
    "Year": "year",
    "India_CPI_2010_100": "cpi_release_year"
})

adjusted_df = integrated_df.merge(
    cpi_lookup,
    on="year",
    how="left",
    validate="many_to_one"
)

# --------------------------------------------------
# 5. Calculate inflation-adjusted gross
# --------------------------------------------------

CPI_2025 = float(
    cpi_df.loc[
        cpi_df["Year"] == 2025,
        "India_CPI_2010_100"
    ].iloc[0]
)

adjusted_df["cpi_reference_year"] = 2025
adjusted_df["cpi_2025"] = CPI_2025

adjusted_df["inflation_adjustment_factor"] = (
    adjusted_df["cpi_2025"] /
    adjusted_df["cpi_release_year"]
)

adjusted_df["india_gross_cr_2025_prices"] = (
    adjusted_df["india_gross_cr"] *
    adjusted_df["inflation_adjustment_factor"]
)

adjusted_df["log_india_gross_2025_prices"] = np.log1p(
    adjusted_df["india_gross_cr_2025_prices"]
)

# --------------------------------------------------
# 6. Validate the merged dataset
# --------------------------------------------------

assert adjusted_df.shape == (130, 65)
assert adjusted_df["cpi_release_year"].notna().all()
assert adjusted_df["india_gross_cr_2025_prices"].notna().all()
assert adjusted_df["movie_record_id"].is_unique

assert np.allclose(
    adjusted_df["india_gross_cr"],
    integrated_df["india_gross_cr"]
)

assert adjusted_df["year"].equals(integrated_df["year"])

# --------------------------------------------------
# 7. Save the new dataset
# --------------------------------------------------

OUTPUT_DIR = PROJECT_DIR / "05_outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "ICMB_130_FINAL_INTEGRATED_INFLATION_ADJUSTED_2025.csv"
)

adjusted_df.to_csv(OUTPUT_FILE, index=False)

# --------------------------------------------------
# 8. Display validation results
# --------------------------------------------------

print("=" * 70)
print("ICMB-130 INFLATION ADJUSTMENT — FINAL VALIDATION")
print("=" * 70)

print("Original shape:", integrated_df.shape)
print("Adjusted shape:", adjusted_df.shape)
print("Total films:", len(adjusted_df))
print("Distinct years:", adjusted_df["year"].nunique())
print("Missing CPI values:", adjusted_df["cpi_release_year"].isna().sum())
print("Missing adjusted gross:",
      adjusted_df["india_gross_cr_2025_prices"].isna().sum())
print("Duplicate movie IDs:",
      adjusted_df["movie_record_id"].duplicated().sum())
print("Nominal gross preserved: YES")
print("Reference year: 2025")
print("2025 CPI:", CPI_2025)

print("\nSaved file:")
print(OUTPUT_FILE)

display(
    adjusted_df[
        [
            "movie_record_id",
            "year",
            "movie_title",
            "india_gross_cr",
            "cpi_release_year",
            "inflation_adjustment_factor",
            "india_gross_cr_2025_prices"
        ]
    ].head(10).round(3)
)


In [ ]:

import pandas as pd
from pathlib import Path

# Load the inflation-adjusted integrated dataset
PROJECT_DIR = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_FILE = (
    PROJECT_DIR /
    "05_outputs/tables/"
    "ICMB_130_FINAL_INTEGRATED_INFLATION_ADJUSTED_2025.csv"
)

df = pd.read_csv(INPUT_FILE)

# Define the three research periods
def assign_period(year):
    if 2000 <= year <= 2008:
        return "Early (2000–2008)"
    elif 2009 <= year <= 2017:
        return "Middle (2009–2017)"
    elif 2018 <= year <= 2025:
        return "Recent (2018–2025)"
    return "Outside range"

df["research_period"] = df["year"].apply(assign_period)

# --------------------------------------------------
# 1. Period-wise commercial comparison
# --------------------------------------------------

period_summary = (
    df.groupby("research_period", sort=False)
    .agg(
        film_count=("movie_record_id", "count"),
        nominal_mean=("india_gross_cr", "mean"),
        nominal_median=("india_gross_cr", "median"),
        adjusted_mean=("india_gross_cr_2025_prices", "mean"),
        adjusted_median=("india_gross_cr_2025_prices", "median"),
        nominal_total=("india_gross_cr", "sum"),
        adjusted_total=("india_gross_cr_2025_prices", "sum")
    )
    .reset_index()
)

# --------------------------------------------------
# 2. Annual rank-wise comparison
# --------------------------------------------------

rank_summary = (
    df.groupby("rank")
    .agg(
        film_count=("movie_record_id", "count"),
        nominal_mean=("india_gross_cr", "mean"),
        nominal_median=("india_gross_cr", "median"),
        adjusted_mean=("india_gross_cr_2025_prices", "mean"),
        adjusted_median=("india_gross_cr_2025_prices", "median")
    )
    .reset_index()
)

# --------------------------------------------------
# 3. Year-wise commercial comparison
# --------------------------------------------------

year_summary = (
    df.groupby("year")
    .agg(
        film_count=("movie_record_id", "count"),
        nominal_mean=("india_gross_cr", "mean"),
        adjusted_mean=("india_gross_cr_2025_prices", "mean"),
        nominal_median=("india_gross_cr", "median"),
        adjusted_median=("india_gross_cr_2025_prices", "median")
    )
    .reset_index()
)

# --------------------------------------------------
# 4. Save updated research tables
# --------------------------------------------------

OUTPUT_DIR = PROJECT_DIR / "05_outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

period_summary.to_csv(
    OUTPUT_DIR / "ICMB_130_INFLATION_ADJUSTED_PERIOD_ANALYSIS.csv",
    index=False
)

rank_summary.to_csv(
    OUTPUT_DIR / "ICMB_130_INFLATION_ADJUSTED_RANK_ANALYSIS.csv",
    index=False
)

year_summary.to_csv(
    OUTPUT_DIR / "ICMB_130_INFLATION_ADJUSTED_YEAR_ANALYSIS.csv",
    index=False
)

# --------------------------------------------------
# 5. Display results
# --------------------------------------------------

print("=" * 75)
print("INFLATION-ADJUSTED COMMERCIAL PERFORMANCE")
print("=" * 75)

print("\nPERIOD-WISE COMPARISON (₹ crore)")
display(period_summary.round(2))

print("\nANNUAL RANK-WISE COMPARISON (₹ crore)")
display(rank_summary.round(2))

print("\nYEAR-WISE COMPARISON (₹ crore)")
display(year_summary.round(2))

print("\nSaved three updated commercial analysis tables.")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr, rankdata, pearsonr

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_FILE = (
    PROJECT_ROOT / "05_outputs" / "tables" /
    "ICMB_130_FINAL_INTEGRATED_INFLATION_ADJUSTED_2025.csv"
)

df = pd.read_csv(INPUT_FILE)

# --------------------------------------------------
# 1. VALIDATE DATA AND FEATURES
# --------------------------------------------------

adjusted_gross = "india_gross_cr_2025_prices"

theme_cols = [
    c for c in df.columns
    if c.startswith("theme_share_")
    and c != "theme_share_sum_pct"
]

assert len(theme_cols) == 12

print("Number of curated theme features:", len(theme_cols))
print("\nTheme features:")
for col in theme_cols:
    print(col)

sentiment_cols = [
    "positive_share_classifiable_pct",
    "negative_share_classifiable_pct",
    "neutral_share_classifiable_pct",
    "classifiable_share_total_pct",
    "uncertain_share_total_pct"
]

required = [
    "movie_record_id", "year", adjusted_gross,
    *theme_cols, *sentiment_cols
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

if len(df) != 130 or df["movie_record_id"].nunique() != 130:
    raise ValueError("Integrated dataset must contain 130 unique films.")

if len(theme_cols) != 12:
    raise ValueError(f"Expected 12 theme features, found {len(theme_cols)}")

print("Dataset:", df.shape)
print("Theme features:", len(theme_cols))
print("Adjusted gross column:", adjusted_gross)

# --------------------------------------------------
# 2. BENJAMINI-HOCHBERG FDR CORRECTION
# --------------------------------------------------

def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p), np.nan)

    valid = np.where(np.isfinite(p))[0]

    if len(valid) == 0:
        return adjusted

    pv = p[valid]
    order = np.argsort(pv)
    sorted_p = pv[order]

    m = len(sorted_p)
    adjusted_sorted = sorted_p * m / np.arange(1, m + 1)

    adjusted_sorted = np.minimum.accumulate(
        adjusted_sorted[::-1]
    )[::-1]

    adjusted_sorted = np.clip(adjusted_sorted, 0, 1)

    restored = np.empty(m)
    restored[order] = adjusted_sorted
    adjusted[valid] = restored

    return adjusted

# --------------------------------------------------
# 3. ORDINARY SPEARMAN CORRELATION
# --------------------------------------------------

def spearman_result(data, feature, outcome, group):
    temp = data[[feature, outcome]].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    if len(temp) < 5:
        rho, p_value = np.nan, np.nan
    else:
        rho, p_value = spearmanr(
            temp[feature], temp[outcome]
        )

    return {
        "analysis_group": group,
        "feature": feature,
        "outcome": outcome,
        "n": len(temp),
        "spearman_rho": rho,
        "p_value": p_value
    }

# --------------------------------------------------
# 4. PARTIAL SPEARMAN CONTROLLING FOR YEAR
#    Same residualization approach as original Cell 16
# --------------------------------------------------

def partial_spearman_year(data, feature, outcome):
    temp = data[
        ["year", feature, outcome]
    ].replace([np.inf, -np.inf], np.nan).dropna().copy()

    n = len(temp)

    if n < 10 or temp["year"].nunique() < 2:
        return {
            "feature": feature,
            "outcome": outcome,
            "n": n,
            "partial_spearman_rho": np.nan,
            "p_value": np.nan
        }

    x_rank = rankdata(temp[feature].to_numpy())
    y_rank = rankdata(temp[outcome].to_numpy())

    year_dummies = pd.get_dummies(
        temp["year"].astype(str),
        drop_first=True,
        dtype=float
    ).to_numpy()

    design = np.column_stack([
        np.ones(n),
        year_dummies
    ])

    x_resid = x_rank - design @ np.linalg.lstsq(
        design, x_rank, rcond=None
    )[0]

    y_resid = y_rank - design @ np.linalg.lstsq(
        design, y_rank, rcond=None
    )[0]

    if np.std(x_resid) == 0 or np.std(y_resid) == 0:
        rho, p_value = np.nan, np.nan
    else:
        rho, p_value = pearsonr(x_resid, y_resid)

    return {
        "feature": feature,
        "outcome": outcome,
        "n": n,
        "partial_spearman_rho": rho,
        "p_value": p_value
    }

# --------------------------------------------------
# 5. THEME SHARE VS ADJUSTED GROSS
# --------------------------------------------------

theme_gross = pd.DataFrame([
    spearman_result(
        df, feature, adjusted_gross,
        "Theme share vs inflation-adjusted gross"
    )
    for feature in theme_cols
])

theme_gross["p_value_fdr"] = benjamini_hochberg(
    theme_gross["p_value"]
)

theme_gross["significant_fdr_05"] = (
    theme_gross["p_value_fdr"] < 0.05
)

# --------------------------------------------------
# 6. SENTIMENT VS ADJUSTED GROSS
# --------------------------------------------------

sentiment_gross = pd.DataFrame([
    spearman_result(
        df, feature, adjusted_gross,
        "Sentiment vs inflation-adjusted gross"
    )
    for feature in sentiment_cols
])

sentiment_gross["p_value_fdr"] = benjamini_hochberg(
    sentiment_gross["p_value"]
)

sentiment_gross["significant_fdr_05"] = (
    sentiment_gross["p_value_fdr"] < 0.05
)

# --------------------------------------------------
# 7. YEAR-ADJUSTED THEME VS ADJUSTED GROSS
# --------------------------------------------------

theme_year = pd.DataFrame([
    partial_spearman_year(
        df, feature, adjusted_gross
    )
    for feature in theme_cols
])

theme_year["p_value_fdr"] = benjamini_hochberg(
    theme_year["p_value"]
)

theme_year["significant_fdr_05"] = (
    theme_year["p_value_fdr"] < 0.05
)

# --------------------------------------------------
# 8. YEAR-ADJUSTED SENTIMENT VS ADJUSTED GROSS
# --------------------------------------------------

sentiment_year = pd.DataFrame([
    partial_spearman_year(
        df, feature, adjusted_gross
    )
    for feature in sentiment_cols
])

sentiment_year["p_value_fdr"] = benjamini_hochberg(
    sentiment_year["p_value"]
)

sentiment_year["significant_fdr_05"] = (
    sentiment_year["p_value_fdr"] < 0.05
)

# --------------------------------------------------
# 9. DISPLAY RESULTS
# --------------------------------------------------

print("\n" + "=" * 85)
print("THEME SHARE VS INFLATION-ADJUSTED GROSS")
print("=" * 85)
display(theme_gross.round(5))

print("\n" + "=" * 85)
print("SENTIMENT VS INFLATION-ADJUSTED GROSS")
print("=" * 85)
display(sentiment_gross.round(5))

print("\n" + "=" * 85)
print("YEAR-ADJUSTED THEME VS INFLATION-ADJUSTED GROSS")
print("=" * 85)
display(theme_year.round(5))

print("\n" + "=" * 85)
print("YEAR-ADJUSTED SENTIMENT VS INFLATION-ADJUSTED GROSS")
print("=" * 85)
display(sentiment_year.round(5))

# --------------------------------------------------
# 10. SAVE NEW RESULTS WITHOUT OVERWRITING OLD FILES
# --------------------------------------------------

TABLE_DIR = PROJECT_ROOT / "05_outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

outputs = {
    "ICMB_130_THEME_VS_INFLATION_ADJUSTED_GROSS_SPEARMAN.csv":
        theme_gross,
    "ICMB_130_SENTIMENT_VS_INFLATION_ADJUSTED_GROSS_SPEARMAN.csv":
        sentiment_gross,
    "ICMB_130_THEME_VS_INFLATION_ADJUSTED_GROSS_YEAR_ADJUSTED.csv":
        theme_year,
    "ICMB_130_SENTIMENT_VS_INFLATION_ADJUSTED_GROSS_YEAR_ADJUSTED.csv":
        sentiment_year
}

for filename, result in outputs.items():
    result.to_csv(TABLE_DIR / filename, index=False)
    print("Saved:", filename)

print("\n" + "=" * 85)
print("INFLATION-ADJUSTED CORRELATION ANALYSIS COMPLETED")
print("=" * 85)


In [ ]:

from scipy.stats import t
import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# CORRECT YEAR-ADJUSTED PARTIAL CORRELATION P-VALUES
# --------------------------------------------------

# 26 years represented by 25 year-dummy control variables
number_of_year_controls = 25

def correct_partial_pvalues(result_df):
    result_df = result_df.copy()

    corrected_p = []

    for _, row in result_df.iterrows():
        r = row["partial_spearman_rho"]
        n = int(row["n"])

        df_residual = n - number_of_year_controls - 2

        if pd.isna(r) or df_residual <= 0:
            corrected_p.append(np.nan)
            continue

        t_stat = r * np.sqrt(
            df_residual / (1 - r**2)
        )

        p_value = 2 * t.sf(
            abs(t_stat),
            df=df_residual
        )

        corrected_p.append(p_value)

    result_df["p_value"] = corrected_p

    # Recalculate FDR separately for each analysis family
    result_df["p_value_fdr"] = benjamini_hochberg(
        result_df["p_value"]
    )

    result_df["significant_fdr_05"] = (
        result_df["p_value_fdr"] < 0.05
    )

    result_df["residual_degrees_of_freedom"] = (
        result_df["n"] - number_of_year_controls - 2
    )

    return result_df


# Apply correction
theme_year_corrected = correct_partial_pvalues(
    theme_year
)

sentiment_year_corrected = correct_partial_pvalues(
    sentiment_year
)

# --------------------------------------------------
# DISPLAY CORRECTED RESULTS
# --------------------------------------------------

print("=" * 80)
print("CORRECTED YEAR-ADJUSTED THEME ASSOCIATIONS")
print("=" * 80)

display(
    theme_year_corrected[
        [
            "feature",
            "n",
            "partial_spearman_rho",
            "p_value",
            "p_value_fdr",
            "significant_fdr_05",
            "residual_degrees_of_freedom"
        ]
    ].round(5)
)

print("\n" + "=" * 80)
print("CORRECTED YEAR-ADJUSTED SENTIMENT ASSOCIATIONS")
print("=" * 80)

display(
    sentiment_year_corrected[
        [
            "feature",
            "n",
            "partial_spearman_rho",
            "p_value",
            "p_value_fdr",
            "significant_fdr_05",
            "residual_degrees_of_freedom"
        ]
    ].round(5)
)

# --------------------------------------------------
# SAVE CORRECTED TABLES
# --------------------------------------------------

TABLE_DIR = (
    Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
    / "05_outputs" / "tables"
)

theme_year_corrected.to_csv(
    TABLE_DIR /
    "ICMB_130_THEME_VS_INFLATION_ADJUSTED_GROSS_YEAR_ADJUSTED_CORRECTED.csv",
    index=False
)

sentiment_year_corrected.to_csv(
    TABLE_DIR /
    "ICMB_130_SENTIMENT_VS_INFLATION_ADJUSTED_GROSS_YEAR_ADJUSTED_CORRECTED.csv",
    index=False
)

print("\nCorrected year-adjusted tables saved successfully.")


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# --------------------------------------------------
# 1. LOAD INFLATION-ADJUSTED DATA
# --------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_FILE = (
    PROJECT_ROOT / "05_outputs" / "tables" /
    "ICMB_130_FINAL_INTEGRATED_INFLATION_ADJUSTED_2025.csv"
)

df = pd.read_csv(INPUT_FILE)

FIGURE_DIR = PROJECT_ROOT / "05_outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. DEFINE RESEARCH PERIODS
# --------------------------------------------------

def assign_period(year):
    if 2000 <= year <= 2008:
        return "Early (2000–2008)"
    elif 2009 <= year <= 2017:
        return "Middle (2009–2017)"
    elif 2018 <= year <= 2025:
        return "Recent (2018–2025)"

df["research_period"] = df["year"].apply(assign_period)

period_order = [
    "Early (2000–2008)",
    "Middle (2009–2017)",
    "Recent (2018–2025)"
]

# --------------------------------------------------
# 3. PERIOD-WISE COMPARISON
# --------------------------------------------------

period_data = (
    df.groupby("research_period")
    .agg(
        nominal_mean=("india_gross_cr", "mean"),
        adjusted_mean=("india_gross_cr_2025_prices", "mean")
    )
    .reindex(period_order)
)

ax = period_data.plot(
    kind="bar",
    figsize=(10, 6)
)

ax.set_title(
    "Period-wise Box Office Performance: Nominal vs Inflation-Adjusted"
)
ax.set_xlabel("Research Period")
ax.set_ylabel("Mean India Gross (₹ crore)")
ax.legend([
    "Nominal Gross",
    "Inflation-Adjusted Gross (2025 prices)"
])
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "01_Period_Nominal_vs_Adjusted.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# --------------------------------------------------
# 4. RANK-WISE COMPARISON
# --------------------------------------------------

rank_data = (
    df.groupby("rank")
    .agg(
        nominal_mean=("india_gross_cr", "mean"),
        adjusted_mean=("india_gross_cr_2025_prices", "mean")
    )
)

ax = rank_data.plot(
    kind="bar",
    figsize=(10, 6)
)

ax.set_title(
    "Annual Rank-wise Box Office Performance"
)
ax.set_xlabel("Annual Gross Collection Rank")
ax.set_ylabel("Mean India Gross (₹ crore)")
ax.legend([
    "Nominal Gross",
    "Inflation-Adjusted Gross (2025 prices)"
])
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "02_Rank_Nominal_vs_Adjusted.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# --------------------------------------------------
# 5. YEAR-WISE COMPARISON
# --------------------------------------------------

year_data = (
    df.groupby("year")
    .agg(
        nominal_mean=("india_gross_cr", "mean"),
        adjusted_mean=("india_gross_cr_2025_prices", "mean")
    )
    .sort_index()
)

ax = year_data.plot(
    kind="line",
    marker="o",
    figsize=(13, 6)
)

ax.set_title(
    "Year-wise Mean Box Office Performance (2000–2025)"
)
ax.set_xlabel("Year")
ax.set_ylabel("Mean India Gross (₹ crore)")
ax.legend([
    "Nominal Gross",
    "Inflation-Adjusted Gross (2025 prices)"
])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "03_Year_Nominal_vs_Adjusted.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# --------------------------------------------------
# 6. VERIFY SAVED FIGURES
# --------------------------------------------------

print("=" * 70)
print("INFLATION-ADJUSTED VISUALIZATION INVENTORY")
print("=" * 70)

for file in sorted(FIGURE_DIR.glob("*Nominal_vs_Adjusted.png")):
    print(file.name, "|", round(file.stat().st_size / 1024, 2), "KB")

print("\nAll three commercial comparison charts generated.")


In [ ]:

import pandas as pd
from pathlib import Path

# Load the final inflation-adjusted integrated dataset
PROJECT_DIR = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")

DATA_PATH = (
    PROJECT_DIR
    / "05_outputs"
    / "tables"
    / "ICMB_130_FINAL_INTEGRATED_INFLATION_ADJUSTED_2025.csv"
)

df = pd.read_csv(DATA_PATH)

print("=" * 75)
print("THEME AND SENTIMENT COLUMN INSPECTION")
print("=" * 75)

print("\nDataset shape:", df.shape)

print("\nTHEME SHARE COLUMNS:")
theme_cols = [
    col for col in df.columns
    if col.startswith("theme_share_")
    and col != "theme_share_sum_pct"
]

for col in theme_cols:
    print(col)

print("\nSENTIMENT-RELATED COLUMNS:")
sentiment_cols = [
    col for col in df.columns
    if any(term in col.lower() for term in
           ["positive", "negative", "neutral", "uncertain",
            "sentiment", "classifiable", "emoji"])
]

for col in sentiment_cols:
    print(col)

print("\nOTHER RELEVANT COLUMNS:")
for col in ["movie_record_id", "movie_title", "year",
            "research_period", "dominant_theme"]:
    if col in df.columns:
        print(col, ":", "Available")


In [ ]:

import numpy as np
import pandas as pd

# Identify the 12 curated theme columns
theme_cols = [
    col for col in df.columns
    if col.startswith("theme_share_")
    and col != "theme_share_sum_pct"
]

# Check numeric values
df[theme_cols] = df[theme_cols].apply(
    pd.to_numeric, errors="coerce"
)

# Convert technical column names into readable labels
theme_labels = {
    col: col.replace("theme_share_", "")
             .replace("_", " ")
             .title()
    for col in theme_cols
}

# Calculate film-level theme statistics
theme_summary = pd.DataFrame({
    "theme": [theme_labels[col] for col in theme_cols],
    "mean_share": [df[col].mean() for col in theme_cols],
    "median_share": [df[col].median() for col in theme_cols],
    "films_with_theme": [(df[col] > 0).sum() for col in theme_cols]
})

theme_summary = theme_summary.sort_values(
    "mean_share", ascending=False
).reset_index(drop=True)

print("=" * 75)
print("FILM-LEVEL CINEMATIC THEME DISTRIBUTION")
print("=" * 75)

print("\nTheme share value range:")
print("Minimum:", df[theme_cols].min().min())
print("Maximum:", df[theme_cols].max().max())

print("\nTheme summary:")
display(theme_summary.round(3))

# Save summary
OUTPUT_DIR = (
    PROJECT_DIR / "05_outputs" / "tables"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

theme_summary.to_csv(
    OUTPUT_DIR / "ICMB_130_FILM_LEVEL_THEME_DISTRIBUTION.csv",
    index=False
)

print("\nSaved: ICMB_130_FILM_LEVEL_THEME_DISTRIBUTION.csv")


In [ ]:

import matplotlib.pyplot as plt
from pathlib import Path

FIGURE_DIR = PROJECT_DIR / "05_outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Sort themes from highest to lowest
plot_data = theme_summary.sort_values(
    "mean_share", ascending=True
)

fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(
    plot_data["theme"],
    plot_data["mean_share"]
)

# Add percentage labels
for bar, value in zip(bars, plot_data["mean_share"]):
    ax.text(
        bar.get_width() + 0.4,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}%",
        va="center",
        fontsize=9
    )

ax.set_title(
    "Average Distribution of Cinematic Themes Across 130 Films",
    fontsize=14,
    pad=15
)
ax.set_xlabel("Mean Theme Share (%)")
ax.set_ylabel("Curated Cinematic Themes")
ax.set_xlim(0, plot_data["mean_share"].max() + 7)
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()

chart_path = FIGURE_DIR / "04_Overall_Cinematic_Theme_Distribution.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", chart_path.name)
print(f"File size: {chart_path.stat().st_size / 1024:.2f} KB")


In [ ]:

# Assign research periods
def assign_period(year):
    if 2000 <= year <= 2008:
        return "Early (2000–2008)"
    elif 2009 <= year <= 2017:
        return "Middle (2009–2017)"
    elif 2018 <= year <= 2025:
        return "Recent (2018–2025)"
    return None

df["research_period"] = df["year"].apply(assign_period)

period_order = [
    "Early (2000–2008)",
    "Middle (2009–2017)",
    "Recent (2018–2025)"
]

# Calculate mean theme share for each period
period_theme_summary = (
    df.groupby("research_period")[theme_cols]
      .mean()
      .reindex(period_order)
)

# Replace technical column names with readable labels
period_theme_summary.columns = [
    theme_labels[col] for col in period_theme_summary.columns
]

print("=" * 75)
print("PERIOD-WISE CINEMATIC THEME DISTRIBUTION")
print("=" * 75)

display(period_theme_summary.round(2))

# Save period-wise summary
period_theme_summary.to_csv(
    OUTPUT_DIR / "ICMB_130_PERIOD_WISE_THEME_DISTRIBUTION.csv"
)

print("\nSaved: ICMB_130_PERIOD_WISE_THEME_DISTRIBUTION.csv")

# Create grouped bar chart
ax = period_theme_summary.T.plot(
    kind="bar",
    figsize=(16, 8),
    width=0.8
)

ax.set_title(
    "Evolution of Cinematic Themes Across Research Periods",
    fontsize=15,
    pad=15
)
ax.set_xlabel("Curated Cinematic Themes")
ax.set_ylabel("Mean Theme Share (%)")
ax.legend(title="Research Period")
ax.tick_params(axis="x", rotation=65)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()

period_chart_path = (
    FIGURE_DIR / "05_Period_Wise_Theme_Evolution.png"
)

plt.savefig(
    period_chart_path,
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

print("\nSaved:", period_chart_path.name)
print(
    f"File size: {period_chart_path.stat().st_size / 1024:.2f} KB"
)


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Prepare data for heatmap
heatmap_data = period_theme_summary.copy()

fig, ax = plt.subplots(figsize=(15, 9))

sns.heatmap(
    heatmap_data.T,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "Mean Theme Share (%)"},
    ax=ax
)

ax.set_title(
    "Heatmap of Cinematic Theme Distribution Across Research Periods",
    fontsize=15,
    pad=18
)
ax.set_xlabel("Research Period")
ax.set_ylabel("Curated Cinematic Themes")

plt.xticks(rotation=15, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()

heatmap_path = (
    FIGURE_DIR / "06_Cinematic_Theme_Period_Heatmap.png"
)

plt.savefig(
    heatmap_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Saved:", heatmap_path.name)
print(
    f"File size: {heatmap_path.stat().st_size / 1024:.2f} KB"
)


In [ ]:

# Identify the dominant theme for each film
dominant_theme_index = df[theme_cols].idxmax(axis=1)

df["dominant_theme"] = dominant_theme_index.map(theme_labels)

# Handle films with no assigned theme contribution
no_theme_mask = df[theme_cols].fillna(0).max(axis=1) == 0
df.loc[no_theme_mask, "dominant_theme"] = "No assigned theme"

# Count films by dominant theme
dominant_theme_summary = (
    df["dominant_theme"]
    .value_counts()
    .rename_axis("dominant_theme")
    .reset_index(name="film_count")
)

dominant_theme_summary["percentage"] = (
    dominant_theme_summary["film_count"] / len(df) * 100
)

print("=" * 75)
print("FILM-LEVEL DOMINANT THEME DISTRIBUTION")
print("=" * 75)

display(dominant_theme_summary.round(2))

print("\nTotal films:", len(df))
print("Films with assigned themes:", (~no_theme_mask).sum())
print("Films without assigned themes:", no_theme_mask.sum())

# Save summary
dominant_theme_summary.to_csv(
    OUTPUT_DIR / "ICMB_130_DOMINANT_THEME_DISTRIBUTION.csv",
    index=False
)

print("\nSaved: ICMB_130_DOMINANT_THEME_DISTRIBUTION.csv")

# Create visualization
plot_data = dominant_theme_summary.sort_values(
    "film_count", ascending=True
)

fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(
    plot_data["dominant_theme"],
    plot_data["film_count"]
)

for bar, value in zip(bars, plot_data["film_count"]):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center"
    )

ax.set_title(
    "Dominant Cinematic Theme Across 130 Films",
    fontsize=14,
    pad=15
)
ax.set_xlabel("Number of Films")
ax.set_ylabel("Dominant Theme")
ax.set_xlim(0, plot_data["film_count"].max() + 10)
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()

dominant_chart_path = (
    FIGURE_DIR / "07_Film_Level_Dominant_Theme.png"
)

plt.savefig(
    dominant_chart_path,
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

print("\nSaved:", dominant_chart_path.name)
print(
    f"File size: {dominant_chart_path.stat().st_size / 1024:.2f} KB"
)


In [ ]:

# Corrected sentiment distribution including empty comments

sentiment_summary = pd.DataFrame({
    "sentiment": [
        "Positive",
        "Negative",
        "Neutral",
        "Uncertain",
        "Emoji-only / Not modelled",
        "Empty"
    ],
    "comment_count": [
        int(df["positive_comments"].sum()),
        int(df["negative_comments"].sum()),
        int(df["neutral_comments"].sum()),
        int(df["uncertain_comments"].sum()),
        int(df["emoji_only_comments"].sum()),
        1
    ]
})

total_comments = 15960

sentiment_summary["percentage_all_comments"] = (
    sentiment_summary["comment_count"] / total_comments * 100
)

classifiable_total = 6093

sentiment_summary["percentage_classifiable"] = 0.0

mask = sentiment_summary["sentiment"].isin(
    ["Positive", "Negative", "Neutral"]
)

sentiment_summary.loc[mask, "percentage_classifiable"] = (
    sentiment_summary.loc[mask, "comment_count"]
    / classifiable_total * 100
)

print("=" * 75)
print("FINAL VERIFIED YOUTUBE SENTIMENT DISTRIBUTION")
print("=" * 75)

display(sentiment_summary.round(2))

print("\nTotal comments:", sentiment_summary["comment_count"].sum())
print("Classifiable comments:", classifiable_total)
print(
    "Classifiable sentiment total:",
    sentiment_summary.loc[mask, "comment_count"].sum()
)

assert sentiment_summary["comment_count"].sum() == 15960
assert sentiment_summary.loc[mask, "comment_count"].sum() == 6093

sentiment_summary.to_csv(
    OUTPUT_DIR / "ICMB_130_SENTIMENT_DISTRIBUTION_SUMMARY.csv",
    index=False
)

print("\nVerified summary saved successfully.")


In [ ]:

import matplotlib.pyplot as plt

# Sort sentiment categories by comment count
plot_data = sentiment_summary.sort_values(
    "comment_count", ascending=True
)

fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.barh(
    plot_data["sentiment"],
    plot_data["percentage_all_comments"]
)

for bar, count, percentage in zip(
    bars,
    plot_data["comment_count"],
    plot_data["percentage_all_comments"]
):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{percentage:.2f}% ({count:,})",
        va="center",
        fontsize=9
    )

ax.set_title(
    "Overall YouTube Comment Sentiment Distribution",
    fontsize=14,
    pad=15
)
ax.set_xlabel("Percentage of All Comments (%)")
ax.set_ylabel("Sentiment Category")
ax.set_xlim(0, 70)
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()

sentiment_chart_path = (
    FIGURE_DIR / "08_Overall_YouTube_Sentiment_Distribution.png"
)

plt.savefig(
    sentiment_chart_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Saved:", sentiment_chart_path.name)
print(
    f"File size: {sentiment_chart_path.stat().st_size / 1024:.2f} KB"
)


In [ ]:

# Select classifiable sentiment categories
classifiable_data = sentiment_summary[
    sentiment_summary["sentiment"].isin(
        ["Positive", "Negative", "Neutral"]
    )
].copy()

classifiable_data = classifiable_data.sort_values(
    "percentage_classifiable", ascending=True
)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    classifiable_data["sentiment"],
    classifiable_data["percentage_classifiable"]
)

for bar, count, percentage in zip(
    bars,
    classifiable_data["comment_count"],
    classifiable_data["percentage_classifiable"]
):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{percentage:.2f}% ({count:,})",
        va="center",
        fontsize=10
    )

ax.set_title(
    "Positive, Negative and Neutral Sentiment Among Classifiable Comments",
    fontsize=13,
    pad=15
)
ax.set_xlabel("Percentage of Classifiable Comments (%)")
ax.set_ylabel("Sentiment Category")
ax.set_xlim(0, 60)
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()

classifiable_chart_path = (
    FIGURE_DIR / "09_Classifiable_YouTube_Sentiment_Distribution.png"
)

plt.savefig(
    classifiable_chart_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Saved:", classifiable_chart_path.name)
print(
    f"File size: {classifiable_chart_path.stat().st_size / 1024:.2f} KB"
)


In [ ]:

# Define sentiment count columns
period_sentiment_cols = [
    "positive_comments",
    "negative_comments",
    "neutral_comments",
    "uncertain_comments"
]

# Ensure research periods are assigned
df["research_period"] = df["year"].apply(assign_period)

period_order = [
    "Early (2000–2008)",
    "Middle (2009–2017)",
    "Recent (2018–2025)"
]

# Aggregate comment counts by research period
period_sentiment = (
    df.groupby("research_period")[period_sentiment_cols]
      .sum()
      .reindex(period_order)
)

# Calculate classifiable comment totals
period_sentiment["classifiable_total"] = (
    period_sentiment["positive_comments"]
    + period_sentiment["negative_comments"]
    + period_sentiment["neutral_comments"]
)

# Calculate sentiment percentages within classifiable comments
for sentiment in ["positive", "negative", "neutral"]:
    period_sentiment[f"{sentiment}_share_pct"] = (
        period_sentiment[f"{sentiment}_comments"]
        / period_sentiment["classifiable_total"] * 100
    )

print("=" * 75)
print("PERIOD-WISE YOUTUBE SENTIMENT ANALYSIS")
print("=" * 75)

display(period_sentiment.round(2))

# Save period-wise results
period_sentiment.to_csv(
    OUTPUT_DIR / "ICMB_130_PERIOD_WISE_SENTIMENT_ANALYSIS.csv"
)

print("\nSaved: ICMB_130_PERIOD_WISE_SENTIMENT_ANALYSIS.csv")

# Create grouped bar chart
plot_cols = [
    "positive_share_pct",
    "negative_share_pct",
    "neutral_share_pct"
]

plot_labels = [
    "Positive",
    "Negative",
    "Neutral"
]

plot_data = period_sentiment[plot_cols].copy()
plot_data.columns = plot_labels

ax = plot_data.plot(
    kind="bar",
    figsize=(11, 7),
    width=0.75
)

ax.set_title(
    "YouTube Comment Sentiment Across Research Periods",
    fontsize=14,
    pad=15
)
ax.set_xlabel("Research Period")
ax.set_ylabel("Share of Classifiable Comments (%)")
ax.set_ylim(0, 65)
ax.legend(title="Sentiment")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()

period_sentiment_chart = (
    FIGURE_DIR / "10_Period_Wise_YouTube_Sentiment.png"
)

plt.savefig(
    period_sentiment_chart,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("\nSaved:", period_sentiment_chart.name)
print(
    f"File size: {period_sentiment_chart.stat().st_size / 1024:.2f} KB"
)


In [ ]:

from pathlib import Path
import pandas as pd

TABLE_DIR = PROJECT_DIR / "05_outputs" / "tables"

correlation_files = [
    "ICMB_130_THEME_VS_INFLATION_ADJUSTED_GROSS_SPEARMAN.csv",
    "ICMB_130_SENTIMENT_VS_INFLATION_ADJUSTED_GROSS_SPEARMAN.csv",
    "ICMB_130_THEME_VS_INFLATION_ADJUSTED_GROSS_YEAR_ADJUSTED_CORRECTED.csv",
    "ICMB_130_SENTIMENT_VS_INFLATION_ADJUSTED_GROSS_YEAR_ADJUSTED_CORRECTED.csv"
]

print("=" * 80)
print("CORRELATION RESULT FILE INSPECTION")
print("=" * 80)

for filename in correlation_files:
    file_path = TABLE_DIR / filename

    print("\n" + "-" * 75)
    print("FILE:", filename)

    if file_path.exists():
        temp = pd.read_csv(file_path)

        print("Shape:", temp.shape)
        print("Columns:", temp.columns.tolist())
        display(temp.head(5))
    else:
        print("FILE NOT FOUND")


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

TABLE_DIR = PROJECT_DIR / "05_outputs" / "tables"

theme_corr = pd.read_csv(
    TABLE_DIR /
    "ICMB_130_THEME_VS_INFLATION_ADJUSTED_GROSS_YEAR_ADJUSTED_CORRECTED.csv"
)

sentiment_corr = pd.read_csv(
    TABLE_DIR /
    "ICMB_130_SENTIMENT_VS_INFLATION_ADJUSTED_GROSS_YEAR_ADJUSTED_CORRECTED.csv"
)

# Convert theme feature names into readable labels
theme_corr["label"] = (
    theme_corr["feature"]
    .str.replace("theme_share_", "", regex=False)
    .str.replace("_", " ")
    .str.title()
)

theme_corr = theme_corr.sort_values(
    "partial_spearman_rho"
)

fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(
    theme_corr["label"],
    theme_corr["partial_spearman_rho"]
)

ax.axvline(0, color="black", linewidth=0.8)

for bar, value in zip(
    bars, theme_corr["partial_spearman_rho"]
):
    offset = 0.008 if value >= 0 else -0.008
    ax.text(
        value + offset,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center",
        ha="left" if value >= 0 else "right",
        fontsize=9
    )

ax.set_title(
    "Year-Adjusted Associations Between Cinematic Themes "
    "and Inflation-Adjusted Gross",
    fontsize=14,
    pad=15
)
ax.set_xlabel("Partial Spearman Correlation (ρ)")
ax.set_ylabel("Cinematic Theme")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()

theme_corr_chart = (
    FIGURE_DIR / "11_Year_Adjusted_Theme_Gross_Correlation.png"
)

plt.savefig(
    theme_corr_chart,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Saved:", theme_corr_chart.name)
print(
    f"File size: {theme_corr_chart.stat().st_size / 1024:.2f} KB"
)

print("\nSignificant themes after FDR correction:",
      int(theme_corr["significant_fdr_05"].sum()))


In [ ]:

# Prepare readable sentiment labels
sentiment_labels = {
    "positive_share_classifiable_pct": "Positive sentiment share",
    "negative_share_classifiable_pct": "Negative sentiment share",
    "neutral_share_classifiable_pct": "Neutral sentiment share",
    "classifiable_share_total_pct": "Classifiable comment share",
    "uncertain_share_total_pct": "Uncertain comment share"
}

sentiment_corr["label"] = sentiment_corr["feature"].map(
    sentiment_labels
).fillna(sentiment_corr["feature"])

sentiment_corr = sentiment_corr.sort_values(
    "partial_spearman_rho"
)

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(
    sentiment_corr["label"],
    sentiment_corr["partial_spearman_rho"]
)

ax.axvline(0, color="black", linewidth=0.8)

for bar, value, significant in zip(
    bars,
    sentiment_corr["partial_spearman_rho"],
    sentiment_corr["significant_fdr_05"]
):
    offset = 0.008 if value >= 0 else -0.008

    ax.text(
        value + offset,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center",
        ha="left" if value >= 0 else "right",
        fontsize=9
    )

ax.set_title(
    "Year-Adjusted Associations Between YouTube Sentiment "
    "and Inflation-Adjusted Gross",
    fontsize=14,
    pad=15
)
ax.set_xlabel("Partial Spearman Correlation (ρ)")
ax.set_ylabel("Sentiment Measure")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()

sentiment_corr_chart = (
    FIGURE_DIR / "12_Year_Adjusted_Sentiment_Gross_Correlation.png"
)

plt.savefig(
    sentiment_corr_chart,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Saved:", sentiment_corr_chart.name)
print(
    f"File size: {sentiment_corr_chart.stat().st_size / 1024:.2f} KB"
)

print("\nSignificant sentiment associations after FDR correction:")
display(
    sentiment_corr.loc[
        sentiment_corr["significant_fdr_05"],
        ["label", "partial_spearman_rho", "p_value_fdr"]
    ].round(4)
)


In [ ]:

# Load corrected year-adjusted correlation results
theme_corr = pd.read_csv(
    TABLE_DIR /
    "ICMB_130_THEME_VS_INFLATION_ADJUSTED_GROSS_YEAR_ADJUSTED_CORRECTED.csv"
)

sentiment_corr = pd.read_csv(
    TABLE_DIR /
    "ICMB_130_SENTIMENT_VS_INFLATION_ADJUSTED_GROSS_YEAR_ADJUSTED_CORRECTED.csv"
)

# Prepare theme findings
theme_findings = pd.DataFrame({
    "analysis_type": "Theme vs inflation-adjusted gross",
    "variable": (
        theme_corr["feature"]
        .str.replace("theme_share_", "", regex=False)
        .str.replace("_", " ")
        .str.title()
    ),
    "sample_size": theme_corr["n"],
    "partial_spearman_rho": theme_corr["partial_spearman_rho"],
    "p_value": theme_corr["p_value"],
    "fdr_adjusted_p_value": theme_corr["p_value_fdr"],
    "significant_after_fdr": theme_corr["significant_fdr_05"]
})

# Prepare sentiment findings
sentiment_labels = {
    "positive_share_classifiable_pct": "Positive sentiment share",
    "negative_share_classifiable_pct": "Negative sentiment share",
    "neutral_share_classifiable_pct": "Neutral sentiment share",
    "classifiable_share_total_pct": "Classifiable comment share",
    "uncertain_share_total_pct": "Uncertain comment share"
}

sentiment_findings = pd.DataFrame({
    "analysis_type": "Sentiment vs inflation-adjusted gross",
    "variable": sentiment_corr["feature"].map(
        sentiment_labels
    ).fillna(sentiment_corr["feature"]),
    "sample_size": sentiment_corr["n"],
    "partial_spearman_rho": sentiment_corr["partial_spearman_rho"],
    "p_value": sentiment_corr["p_value"],
    "fdr_adjusted_p_value": sentiment_corr["p_value_fdr"],
    "significant_after_fdr": sentiment_corr["significant_fdr_05"]
})

# Combine results
final_correlation_findings = pd.concat(
    [theme_findings, sentiment_findings],
    ignore_index=True
)

final_correlation_findings["interpretation"] = (
    final_correlation_findings["significant_after_fdr"]
    .map({
        True: "Statistically significant after FDR correction",
        False: "Not statistically significant after FDR correction"
    })
)

print("=" * 85)
print("CONSOLIDATED YEAR-ADJUSTED CORRELATION FINDINGS")
print("=" * 85)

display(final_correlation_findings.round(4))

# Verification
print("\nTotal associations:", len(final_correlation_findings))
print(
    "Significant associations:",
    int(final_correlation_findings["significant_after_fdr"].sum())
)
print(
    "Significant theme associations:",
    int(theme_findings["significant_after_fdr"].sum())
)
print(
    "Significant sentiment associations:",
    int(sentiment_findings["significant_after_fdr"].sum())
)

assert len(final_correlation_findings) == 17
assert final_correlation_findings["significant_after_fdr"].sum() == 3

# Save final consolidated findings
findings_path = (
    TABLE_DIR / "ICMB_130_FINAL_YEAR_ADJUSTED_CORRELATION_FINDINGS.csv"
)

final_correlation_findings.to_csv(
    findings_path,
    index=False
)

print("\nSaved:", findings_path.name)


In [ ]:

from pathlib import Path
import pandas as pd

FIGURE_DIR = PROJECT_DIR / "05_outputs" / "figures"

expected_figures = [
    "01_Period_Nominal_vs_Adjusted.png",
    "02_Rank_Nominal_vs_Adjusted.png",
    "03_Year_Nominal_vs_Adjusted.png",
    "04_Overall_Cinematic_Theme_Distribution.png",
    "05_Period_Wise_Theme_Evolution.png",
    "06_Cinematic_Theme_Period_Heatmap.png",
    "07_Film_Level_Dominant_Theme.png",
    "08_Overall_YouTube_Sentiment_Distribution.png",
    "09_Classifiable_YouTube_Sentiment_Distribution.png",
    "10_Period_Wise_YouTube_Sentiment.png",
    "11_Year_Adjusted_Theme_Gross_Correlation.png",
    "12_Year_Adjusted_Sentiment_Gross_Correlation.png"
]

inventory = []

for filename in expected_figures:
    path = FIGURE_DIR / filename

    inventory.append({
        "file": filename,
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2)
        if path.exists() else 0
    })

figure_inventory = pd.DataFrame(inventory)

print("=" * 80)
print("NOTEBOOK 11 — FINAL FIGURE INVENTORY")
print("=" * 80)

display(figure_inventory)

print("\nExpected figures:", len(expected_figures))
print("Figures found:", int(figure_inventory["exists"].sum()))
print("Figures missing:", int((~figure_inventory["exists"]).sum()))

if figure_inventory["exists"].all():
    print("\nSUCCESS: All 12 expected visualizations are present.")
else:
    print("\nWARNING: Some visualizations are missing.")

figure_inventory.to_csv(
    PROJECT_DIR / "05_outputs" / "tables"
    / "ICMB_130_NOTEBOOK_11_FIGURE_INVENTORY.csv",
    index=False
)

print("\nFigure inventory CSV saved.")


In [ ]:

from pathlib import Path
from datetime import datetime

REPORT_DIR = PROJECT_DIR / "05_outputs" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

findings_text = """
# ICMB-130: Final Research Findings

## 1. Dataset Overview

The Indian Cinema Multimodal Benchmark Dataset (ICMB-130)
contains 130 films selected from the five highest-grossing
Indian films in each year from 2000 to 2025.

India Gross was used as the commercial ranking metric.
The dataset integrates film-level metadata, box-office
information, subtitle-derived thematic features and
YouTube comment sentiment information.

The integrated inflation-adjusted dataset contains
130 observations and 65 columns.

## 2. Commercial Performance and Inflation Adjustment

The original India Gross collections were adjusted to
2025 price levels using India's annual Consumer Price
Index (CPI), obtained from the World Bank.

The adjustment formula was:

Adjusted Gross = Original Gross × (CPI in 2025 / CPI in release year)

Period-wise mean gross:

| Research Period | Nominal Mean (₹ Cr) | Adjusted Mean (₹ Cr) |
|---|---:|---:|
| Early (2000–2008) | 68.51 | 241.16 |
| Middle (2009–2017) | 254.36 | 447.29 |
| Recent (2018–2025) | 458.80 | 521.68 |

The nominal early-to-recent mean gross ratio is approximately
6.70, whereas the inflation-adjusted ratio is approximately
2.16.

This demonstrates that nominal comparisons substantially
overstate the difference between historical and recent
box-office collections.

The adjustment is CPI-based and should not be interpreted
as a ticket-price-adjusted or audience-volume measure.

## 3. Cinematic Theme Distribution

Twelve curated cinematic themes were examined using
subtitle-derived theme shares.

The highest average film-level theme shares were:

| Theme | Mean Share |
|---|---:|
| Love and Romantic Relationships | 37.37% |
| Marriage and Weddings | 12.69% |
| Family and Parent-Child Relationships | 9.55% |
| Money, Wealth and Economic Transactions | 7.48% |
| Police and Law Enforcement | 7.38% |
| Sports and Competition | 7.15% |

Love and Romantic Relationships was the most prominent
theme in the dataset.

It was the dominant theme in 69 of 130 films (53.08%).

## 4. Changes Across Research Periods

The average share of Love and Romantic Relationships
declined from 42.64% in the Early period to 28.88%
in the Recent period.

Marriage and Weddings declined from 15.82% to 9.84%.

Some themes showed higher average representation in the
Recent period:

- National Security and Terrorism: 1.85% to 6.96%.
- Politics, Governance and Corruption: 0.95% to 5.56%.
- Religion and Devotional Beliefs: 0.57% to 8.79%.
- Money, Wealth and Economic Transactions: 6.59% to 9.61%.

These are descriptive differences in subtitle-derived
theme representation, not evidence of causal changes
in audience preferences.

## 5. YouTube Comment Sentiment

The sentiment dataset contains 15,960 comments.

| Sentiment Category | Count | Share of All Comments |
|---|---:|---:|
| Positive | 2,985 | 18.70% |
| Negative | 2,168 | 13.58% |
| Neutral | 940 | 5.89% |
| Uncertain | 9,476 | 59.37% |
| Emoji-only / Not modelled | 390 | 2.44% |
| Empty | 1 | 0.01% |

There were 6,093 classifiable comments.

Within classifiable comments:

- Positive: 48.99%.
- Negative: 35.58%.
- Neutral: 15.43%.

The high uncertain-comment proportion is an important
limitation when interpreting the sentiment findings.

Period-wise positive sentiment share declined from
55.86% in the Early period to 45.50% in the Recent period.

Period-wise negative sentiment share increased from
29.50% to 41.77%.

These results describe the collected YouTube comments
and do not necessarily represent the entire movie audience.

## 6. Year-Adjusted Correlation Analysis

Partial Spearman correlations were used to examine
associations after controlling for year.

The analysis covered 17 associations:

- 12 cinematic theme measures.
- 5 sentiment measures.

None of the 12 theme measures showed a statistically
significant association with inflation-adjusted gross
after year adjustment and FDR correction.

Three sentiment-related associations remained
statistically significant:

| Measure | Partial Spearman ρ | FDR p-value |
|---|---:|---:|
| Positive sentiment share | -0.2276 | 0.0325 |
| Negative sentiment share | +0.2306 | 0.0325 |
| Classifiable comment share | +0.2345 | 0.0325 |

Neutral sentiment share and uncertain comment share
were not statistically significant after FDR correction.

These findings are exploratory associations and
must not be interpreted as causal effects.

## 7. Research Limitations

1. The dataset contains 130 selected films and should not
   be treated as a complete representation of all Indian cinema.

2. Gross collections are CPI-adjusted, not adjusted for
   ticket prices, admissions, or audience size.

3. Subtitle availability and validation quality vary.
   Not every subtitle file has fully verified film identity.

4. Language representation is unequal across the dataset.
   Language-level comparisons should therefore be interpreted
   cautiously.

5. Subtitle-derived themes represent the available dialogue
   corpus and may not capture every aspect of a film.

6. A large proportion of YouTube comments were classified
   as uncertain. Sentiment coverage is therefore limited.

7. YouTube comment samples and video-to-film source mapping
   have limitations and should not be treated as representative
   samples of all viewers.

8. The analysis is observational and does not establish
   causal relationships.

9. Year-adjusted partial Spearman p-values use a
   residual-degrees-of-freedom approximation. Results should
   be interpreted as exploratory rather than definitive.

## 8. Overall Conclusion

The ICMB-130 analysis provides an integrated view of
commercial performance, subtitle-derived cinematic themes
and YouTube comment sentiment across Indian films released
between 2000 and 2025.

The findings indicate that inflation adjustment changes
the interpretation of historical box-office comparisons.
Love and Romantic Relationships remains the most prominent
curated theme, although its average representation is lower
in the Recent period.

The year-adjusted analysis does not identify statistically
significant associations between individual cinematic
themes and inflation-adjusted gross. However, positive,
negative and classifiable YouTube comment shares show
statistically significant exploratory associations with
inflation-adjusted gross.

Overall, the project demonstrates a multimodal,
longitudinal analytical approach while highlighting the
limitations of subtitle coverage, sentiment classification
and observational data.
"""

findings_path = REPORT_DIR / "ICMB_130_FINAL_RESEARCH_FINDINGS.md"

with open(findings_path, "w", encoding="utf-8") as f:
    f.write(findings_text.strip())

print("=" * 75)
print("FINAL RESEARCH FINDINGS DOCUMENT")
print("=" * 75)

print("File:", findings_path.name)
print("Exists:", findings_path.exists())
print("Size (KB):", round(findings_path.stat().st_size / 1024, 2))
print("Sections:", findings_text.count("\n## "))
print("\nSaved successfully:", findings_path)


In [ ]:

from pathlib import Path
import pandas as pd

BASE = PROJECT_DIR / "05_outputs"

expected_files = [
    # Final datasets
    "tables/ICMB_130_FINAL_INTEGRATED_INFLATION_ADJUSTED_2025.csv",

    # CPI and commercial analysis
    "tables/INDIA_CPI_2000_2025_World_Bank_VERIFIED.csv",
    "tables/ICMB_130_INFLATION_ADJUSTED_PERIOD_ANALYSIS.csv",
    "tables/ICMB_130_INFLATION_ADJUSTED_RANK_ANALYSIS.csv",
    "tables/ICMB_130_INFLATION_ADJUSTED_YEAR_ANALYSIS.csv",

    # Theme analysis
    "tables/ICMB_130_FILM_LEVEL_THEME_DISTRIBUTION.csv",
    "tables/ICMB_130_PERIOD_WISE_THEME_DISTRIBUTION.csv",
    "tables/ICMB_130_DOMINANT_THEME_DISTRIBUTION.csv",

    # Sentiment analysis
    "tables/ICMB_130_SENTIMENT_DISTRIBUTION_SUMMARY.csv",
    "tables/ICMB_130_PERIOD_WISE_SENTIMENT_ANALYSIS.csv",

    # Statistical findings
    "tables/ICMB_130_FINAL_YEAR_ADJUSTED_CORRELATION_FINDINGS.csv",

    # Figure inventory
    "tables/ICMB_130_NOTEBOOK_11_FIGURE_INVENTORY.csv",

    # Final research report
    "reports/ICMB_130_FINAL_RESEARCH_FINDINGS.md",
]

audit_results = []

for relative_path in expected_files:
    path = BASE / relative_path

    audit_results.append({
        "file": relative_path,
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2)
        if path.exists() else 0
    })

# Verify figures separately
figure_files = list((BASE / "figures").glob("*.png"))

audit_df = pd.DataFrame(audit_results)

print("=" * 85)
print("ICMB-130 FINAL PROJECT DELIVERABLES AUDIT")
print("=" * 85)

display(audit_df)

print("\nExpected important files:", len(expected_files))
print("Files found:", int(audit_df["exists"].sum()))
print("Files missing:", int((~audit_df["exists"]).sum()))
print("PNG visualizations present:", len(figure_files))

print("\nFINAL AUDIT STATUS:")

if audit_df["exists"].all() and len(figure_files) >= 12:
    print("All expected deliverables are present.")
else:
    print("Some deliverables need attention.")

# Save audit
audit_path = BASE / "tables" / "ICMB_130_FINAL_PROJECT_DELIVERABLES_AUDIT.csv"

audit_df.to_csv(audit_path, index=False)

print("\nAudit saved:", audit_path.name)


In [ ]:

from pathlib import Path

print("=" * 80)
print("SEARCHING FOR CPI SOURCE FILE")
print("=" * 80)

# Search the complete project directory
cpi_files = list(PROJECT_DIR.rglob("*CPI*.csv"))

if cpi_files:
    print("\nCPI-related CSV files found:")
    for path in cpi_files:
        print(
            "\nFile:", path.name,
            "\nPath:", path,
            "\nSize (KB):", round(path.stat().st_size / 1024, 2)
        )
else:
    print("\nNo CPI-related CSV files found in the project directory.")

# Verify inflation adjustment fields in final dataset
adjusted_path = (
    PROJECT_DIR / "05_outputs" / "tables"
    / "ICMB_130_FINAL_INTEGRATED_INFLATION_ADJUSTED_2025.csv"
)

adjusted_df = pd.read_csv(adjusted_path)

required_cpi_columns = [
    "cpi_release_year",
    "cpi_reference_year",
    "cpi_2025",
    "inflation_adjustment_factor",
    "india_gross_cr_2025_prices"
]

print("\n" + "=" * 80)
print("FINAL DATASET CPI FIELD VERIFICATION")
print("=" * 80)

for col in required_cpi_columns:
    if col in adjusted_df.columns:
        print(
            col,
            "| Available:",
            True,
            "| Missing values:",
            adjusted_df[col].isna().sum()
        )
    else:
        print(col, "| Available:", False)

print("\nDataset shape:", adjusted_df.shape)


In [ ]:

import shutil
import pandas as pd

source_cpi = (
    PROJECT_DIR / "01_data" / "processed"
    / "INDIA_CPI_2000_2025_World_Bank.csv"
)

destination_cpi = (
    PROJECT_DIR / "05_outputs" / "tables"
    / "INDIA_CPI_2000_2025_World_Bank_VERIFIED.csv"
)

# Read and validate the existing CPI file
cpi_df = pd.read_csv(source_cpi)

print("=" * 75)
print("CPI SOURCE FILE VALIDATION")
print("=" * 75)

print("Shape:", cpi_df.shape)
print("Columns:", cpi_df.columns.tolist())
display(cpi_df.head())

# Verify the required year range
print("\nYear range:")
print(cpi_df.iloc[:, 0].min(), "to", cpi_df.iloc[:, 0].max())

print("\nMissing values:")
print(cpi_df.isna().sum())

# Copy existing source file without changing its contents
shutil.copy2(source_cpi, destination_cpi)

# Verify copied file
copied_cpi = pd.read_csv(destination_cpi)

assert cpi_df.equals(copied_cpi)

print("\nCopied CPI source successfully.")
print("Destination:", destination_cpi)
print("File exists:", destination_cpi.exists())
print("Original and copied files identical:", cpi_df.equals(copied_cpi))


In [ ]:

from pathlib import Path

findings_path = (
    PROJECT_DIR / "05_outputs" / "reports"
    / "ICMB_130_FINAL_RESEARCH_FINDINGS.md"
)

with open(findings_path, "r", encoding="utf-8") as f:
    final_findings_content = f.read()

print("=" * 85)
print("FINAL RESEARCH FINDINGS — CONTENT REVIEW")
print("=" * 85)

print(final_findings_content)

print("\n" + "=" * 85)
print("DOCUMENT VERIFICATION")
print("=" * 85)

print("File exists:", findings_path.exists())
print("Characters:", len(final_findings_content))
print("Word count:", len(final_findings_content.split()))
print("Section headings:", [
    line for line in final_findings_content.splitlines()
    if line.startswith("## ")
])
